# Study 1, part 2: measuring what participants wrote

One hundred and eleven undergraduates each rated the same ten artworks on creativity and on effort,
and wrote a short justification for every rating. Half were told a human artist made the works; half
were told an image generator made them. This notebook measures the justifications.

It answers three questions.

**1. What did people mean by "effort"?** In the arm that was told a human made the work, is effort
talk about visible detail in the picture, about time and industriousness, or about skill? The paper
maps the effort rating onto the "sweat of the brow" standard that *Feist v. Rural Telephone Service*
rejected, and that mapping assumes an answer to this question.

**2. What did people mean by "creativity"?** Was creativity talk about the idea behind the work,
about whether the work was new or already familiar, or about how finely it was executed? *Feist* set
effort aside and kept a minimal creative spark. Creativity is therefore the criterion that decides
the case, and what raters meant by the word is what a lay application of the copyright standard
would apply. The novelty answer carries a further point. Novelty against everything that came before
is the *patent* test. Copyright asks only for independent creation, so a work resembling a thousand
others is still protectable. Raters who penalise familiarity are importing a criterion that
copyright law explicitly rejects.

**3. How often did participants told "a human made this" write that a machine made it?** The
instrument contains no manipulation check (a question asking directly whether the participant
accepted what they were told), so what participants wrote is the only trace of the question there
is. The notebook counts mentions of a machine making the work and, among them, statements that a
machine made it. Neither count measures how many participants disbelieved the vignette. A
participant can doubt a story without writing so, and can write that a picture looks machine-made
without doubting it.

Section 11 adds a general description of the text, computed the same way in every study of the
project. Word norms published by other laboratories score how pleasant, how exciting and how much in
control each response's words make a reader feel, and how concrete the words are. A topic model (a
method that finds groups of words that tend to occur together, from word counts alone) shows which
words go together. The description answers no question of its own, and the integration step (the
later stage of the pipeline that joins the condition and the rating back on) reports it as
exploratory.

## How measurement works here

Every measure that answers the three questions is a **search pattern**: a written list of words and
phrases, fixed before any result was looked at and applied by the computer in the same way to all
2,220 responses. No language model reads or labels any response. No person reads any response and
decides what it means. Researcher judgment enters once, in the wording of the rules. Section 1
prints the rules in full, so a reader can disagree with a particular rule rather than with the
method in general.

The general description in section 11 uses no rule written for this study. Its scores come from
published word norms, and its topics come from an algorithm whose every setting is written in the
code. The one pretrained network in the notebook (a model trained beforehand on other text) also
appears there. It turns each response into a list of numbers, so that a second way of grouping the
responses can be compared with the topic model's. It gives no response a label, and no measure is
built from it.

Two sections show what the rules do. Section 3 prints matched responses with the triggering phrase
marked. Section 10 asks the corpus which words each rule may have missed and reports them as a short
list, so that a reader can judge each rule's coverage.

## What this notebook does not do

It does not test whether the text explains the rating. Every justification was written after the
rating it justifies, by the same person, about the same trial. Nothing here is evidence about what
caused a rating. The measures describe what people said, and the ratings are joined in only to
describe who said it. Section 12 lists other approaches that were run on this corpus and are not
used, with what each one showed, so that nobody spends time running them again.

## Where the numbers go next

The main data file this notebook writes, `nlp/study1_text_measures.csv`, is read by the integration
step, which joins the experimental condition and the numeric rating back on and fits the models.
Section 9 writes the measurement halves of two registered descriptive checks on the text (checks
fixed in the study's preregistration). They are a word-count table with its vocabulary, and each
response's list of distinct content words (the words left once function words such as *the* and *of*
are removed), from which the integration step computes within-person word overlap. Section 11 writes
the general description: affect and concreteness scores for every response, and every response's
weight on every topic. All of these go to the same integration step, and none of them carries the
condition or the rating. This notebook never sees the condition while it measures text, and section
2 enforces that with an assertion.

---

# 0. Setup and data check

This section finds the data, loads it, and checks that the file holds the study we expect. If any
check fails, the notebook stops with a plain-language message rather than produce numbers from the
wrong file.

The design is fully crossed (every participant answers both questions about every artwork): 111
participants, each rating 10 artworks on 2 questions, which is 111 x 10 x 2 = 2,220 responses. The
checks below confirm that count.

**Running in Google Colab:** the first cell mounts Google Drive (attaches it so that its files are
visible) and will ask for permission. **Running on a local machine:** it skips the mount and finds
the folder directly. Nothing else changes. The notebook needs pandas and NumPy throughout, and
scikit-learn and SciPy for the topic model in section 11; Colab has all four installed. The
agreement check at the end of section 11 also needs the sentence-transformers package and a copy of
the all-mpnet-base-v2 model already on the machine. The notebook never downloads a model. If either
is missing, the check records that it did not run and the rest of the notebook carries on.

**Writing files.** Every file the notebook writes goes first to a temporary name in the folder it
belongs in, and is renamed to its final name once it is complete. A run that stops partway therefore
never leaves a half-written file under a final name. Each run is given an identifier. Section 13
writes a manifest that records the identifier beside every output, together with a fingerprint of
the input file: its SHA-256 checksum, a 64-character string computed from the file's exact bytes,
which changes if a single byte changes. Each output can then be traced to the run and the data that
produced it. The previous run's manifest is deleted as soon as a run starts, so a manifest on disk
means that the run which wrote it reached the end. The paths the notebook prints are relative to
the Study 1 folder.

In [1]:
# ---------------------------------------------------------------------------
# Find the Study 1 folder, whether we are in Google Colab or on a local machine.
#
# In Colab, Google Drive has to be attached to the machine first ("mounted"),
# which makes the files visible under /content/drive. Locally the folder is
# already on disk: it is the folder the notebook runs from, or the one above
# it. The environment variable STUDY1_DIR, when set, names the folder and is
# tried first. We try the candidate folders in order and take the first one
# that actually contains the input file, which means the same notebook runs
# in both places with no edits.
# ---------------------------------------------------------------------------
import datetime
import hashlib
import os
import re
import textwrap
import uuid
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

try:
    import google.colab  # present only inside Google Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_TAIL = os.path.join(
    "DGK Lab", "Reasoning Lab", "DGK Lab - Copyrightability and AI Art",
    "Analysis", "R Analysis Pipeline", "Study 1",
)

CANDIDATE_FOLDERS = [
    os.path.join("/content/drive/MyDrive", DRIVE_TAIL),
    os.path.join("/content/drive/My Drive", DRIVE_TAIL),
    os.getcwd(),
    os.path.dirname(os.getcwd()),
]
if os.environ.get("STUDY1_DIR"):
    CANDIDATE_FOLDERS.insert(0, os.path.abspath(os.environ["STUDY1_DIR"]))

INPUT_NAME = "study1_long_with_text.csv"
BASE = None
for folder in CANDIDATE_FOLDERS:
    if os.path.exists(os.path.join(folder, INPUT_NAME)):
        BASE = folder
        break

if BASE is None:
    raise FileNotFoundError(
        "Could not find " + INPUT_NAME + ".\n"
        "Looked in:\n  " + "\n  ".join(CANDIDATE_FOLDERS) + "\n"
        "What to check: that Google Drive mounted successfully, and that the Study 1 folder "
        "is where this notebook expects it. You can also set BASE by hand in this cell."
    )

INPUT_PATH = os.path.join(BASE, INPUT_NAME)
NLP_DIR = os.path.join(BASE, "nlp")
TABLES_DIR = os.path.join(BASE, "tables")
os.makedirs(NLP_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)


def relative_path(path):
    """`path` written relative to the Study 1 folder, for printing.

    Files are opened and written through their full paths. Printed output
    shows only the part of a path below the Study 1 folder, so that it
    carries no account name and no folder particular to one machine."""
    return os.path.relpath(path, BASE)


print("Running in Colab:", IN_COLAB)
print("Study 1 folder:   found; the paths printed below are relative to it")
print("Input file:      ", relative_path(INPUT_PATH))
print("Writing data to: ", relative_path(NLP_DIR))
print("Writing tables to:", relative_path(TABLES_DIR))

# ---------------------------------------------------------------------------
# Name this run, and write every output safely.
#
# RUN_ID is the time the run started, in UTC, followed by eight random
# characters so that two runs started in the same second still differ.
# Section 13 writes it beside every output in the run manifest.
#
# atomic_write_text() and atomic_write_csv() write a file under a temporary
# name in the folder it belongs in, then rename it to its final name. A rename
# inside one folder either happens completely or not at all, so a run that
# fails partway never leaves a half-written file under a final name for the
# integration step to read. If the write itself fails, the temporary file is
# removed and the error is raised as usual.
# ---------------------------------------------------------------------------
RUN_ID = (datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
          + "-" + uuid.uuid4().hex[:8])


def _atomic_write(path, write):
    temporary = "%s.tmp-%s" % (path, RUN_ID)
    try:
        write(temporary)
        os.replace(temporary, path)
    except BaseException:
        if os.path.exists(temporary):
            os.remove(temporary)
        raise
    return path


def atomic_write_text(path, content):
    """Write the string `content` to `path` through a temporary file and a rename."""
    def write(target):
        with open(target, "w", encoding="utf-8") as handle:
            handle.write(content)
    return _atomic_write(path, write)


def atomic_write_csv(path, frame):
    """Write the data frame `frame` to `path` as CSV, through a temporary file and a rename."""
    return _atomic_write(path, lambda target: frame.to_csv(target, index=False))


# The previous run's manifest is deleted here, before this run writes
# anything, as the R Markdown files of the pipeline delete theirs. A run that
# stops partway therefore leaves no manifest behind, and a manifest on disk
# means that the run which wrote it reached the last cell.
MANIFEST_PATH = os.path.join(TABLES_DIR, "study1_text_run_manifest.csv")
if os.path.exists(MANIFEST_PATH):
    os.remove(MANIFEST_PATH)

print("Run identifier:  ", RUN_ID)

Running in Colab: False
Study 1 folder:   found; the paths printed below are relative to it
Input file:       study1_long_with_text.csv
Writing data to:  nlp
Writing tables to: tables
Run identifier:   20261015T205712Z-67bfab03


In [2]:
# ---------------------------------------------------------------------------
# Load the data and check it is the study we think it is.
#
# Each check below is written as: compute the number, print it, then assert it.
# Printing first means that when an assertion fails you can see the number that
# caused the failure instead of guessing.
# ---------------------------------------------------------------------------
raw = pd.read_csv(INPUT_PATH)

# A fingerprint of the input file. SHA-256 is a fixed recipe that turns the
# exact bytes of a file into a 64-character string, so that changing a single
# byte changes the string. Section 13 writes it beside every output in the run
# manifest, which ties each output to the exact data it was computed from.
with open(INPUT_PATH, "rb") as handle:
    INPUT_SHA256 = hashlib.sha256(handle.read()).hexdigest()

print("FILE LOADED")
print("  rows:    ", len(raw))
print("  columns: ", list(raw.columns))
print("  SHA-256: ", INPUT_SHA256)
print()

# --- how many responses, participants, and people per arm ---------------------
n_rows = len(raw)
n_participants = raw["id"].nunique()
arm_sizes = raw.groupby("Type")["id"].nunique().sort_index()

print("COUNTS")
print("  responses (rows):        ", n_rows)
print("  participants:            ", n_participants)
for arm, n in arm_sizes.items():
    print("  participants told '%-5s':" % arm, n)

# --- how many responses actually carry text -----------------------------------
# "Usable" here means: the response contains at least one letter of the alphabet.
# A response of pure whitespace or pure punctuation cannot be measured.
text_raw = raw["response_text"].fillna("").astype(str)
usable = text_raw.str.contains(r"[A-Za-z]", regex=True)
n_usable = int(usable.sum())
print("  responses with usable text:", n_usable, "of", n_rows)

# --- is the design fully crossed? ---------------------------------------------
# Every participant should have exactly 10 artworks x 2 questions = 20 responses,
# with each (artwork, question) pair appearing exactly once.
per_participant = raw.groupby("id").size()
images_per_participant = raw.groupby("id")["stimulus_id"].nunique()
questions_per_cell = raw.groupby(["id", "stimulus_id"])["dimension"].nunique()

print()
print("DESIGN")
print("  responses per participant: min %d, max %d" % (per_participant.min(), per_participant.max()))
print("  artworks per participant:  min %d, max %d" % (images_per_participant.min(), images_per_participant.max()))
print("  questions per artwork:     min %d, max %d" % (questions_per_cell.min(), questions_per_cell.max()))
print("  duplicated participant/artwork/question combinations:",
      int(raw.duplicated(["id", "stimulus_id", "dimension"]).sum()))
print()

# --- the assertions -----------------------------------------------------------
FAILED = []
def require(condition, message):
    """Record a failed check instead of stopping at the first one, so that a
    reader sees every problem at once rather than one per re-run."""
    if not condition:
        FAILED.append(message)

require(n_rows == 2220,
        "Expected 2,220 responses, found %d. Check that the ingestion step in the "
        "quantitative notebook ran to completion and wrote the full long file." % n_rows)
require(n_participants == 111,
        "Expected 111 participants, found %d. Check the exclusion rules applied upstream."
        % n_participants)
require(sorted(arm_sizes.tolist()) == [55, 56],
        "Expected arm sizes of 56 and 55, found %s. Check the condition assignment column."
        % arm_sizes.to_dict())
require(n_usable >= 2218,
        "Expected at least 2,218 responses with usable text, found %d. Check the text column "
        "in the export; empty justifications may have been dropped or mis-parsed." % n_usable)
require(int(per_participant.min()) == 20 and int(per_participant.max()) == 20,
        "Every participant should have exactly 20 responses (10 artworks x 2 questions). "
        "Found a range of %d to %d. Check for duplicated or missing rows."
        % (per_participant.min(), per_participant.max()))
require(int(images_per_participant.min()) == 10 and int(images_per_participant.max()) == 10,
        "Every participant should have seen exactly 10 artworks. Found a range of %d to %d."
        % (images_per_participant.min(), images_per_participant.max()))
require(int(questions_per_cell.min()) == 2 and int(questions_per_cell.max()) == 2,
        "Every participant/artwork pair should have both a creativity and an effort response.")
require(int(raw.duplicated(["id", "stimulus_id", "dimension"]).sum()) == 0,
        "Found duplicated participant/artwork/question rows; the file is not one row per response.")

if FAILED:
    raise AssertionError(
        "The data did not match what this notebook expects:\n  - " + "\n  - ".join(FAILED)
    )

print("ALL CHECKS PASSED.")
print("2,220 responses, 111 participants, %d and %d per arm, %d responses with usable text,"
      % (arm_sizes.iloc[0], arm_sizes.iloc[1], n_usable))
print("and every participant has exactly 10 artworks x 2 questions.")

FILE LOADED
  rows:     2220
  columns:  ['id', 'Type', 'stimulus_type', 'stimulus_num', 'stimulus_id', 'dimension', 'rating', 'response_text', 'Age', 'Gender', 'Race', 'Language', 'Experience']
  SHA-256:  5ed3dca926744ddf0ba0d838863c728d8eea0226a12ce75bda286dd1c34a2c6a

COUNTS
  responses (rows):         2220
  participants:             111
  participants told 'AI   ': 55
  participants told 'Human': 56
  responses with usable text: 2220 of 2220

DESIGN
  responses per participant: min 20, max 20
  artworks per participant:  min 10, max 10
  questions per artwork:     min 2, max 2
  duplicated participant/artwork/question combinations: 0

ALL CHECKS PASSED.
2,220 responses, 111 participants, 55 and 56 per arm, 2220 responses with usable text,
and every participant has exactly 10 artworks x 2 questions.


---

# 1. The rules

A **search pattern** (the technical name is a *regular expression*, usually shortened to *regex*) is
a compact way of writing "any of these words or phrases, in any of their ordinary forms". The
pattern `detail(s|ed|ing)?` matches *detail*, *details*, *detailed* and *detailing* but not
*retailer*, because the pattern also demands a word boundary at each end. The computer applies one
written rule to every response, so every response is treated the same way. A rule can be wrong, but
it cannot be wrong differently for different responses.

Eight things are measured. Each is a topic, not a judgment. Most rules produce a **flag**, a column
that is true for a response the rule matched and false for one it did not. `says_detail` records
that a response *mentions* visible detail; it does not record whether the writer thought there was a
lot of it. "There is not a lot of detail here" and "the detail is extraordinary" are both detail
talk. The one exception is prior-exposure talk, where the assertion and its denial are different
claims, so that measure has three values rather than two.

Four of the eight (detail, time, skill and concept talk) carry a second reading beside the flag,
called **polarity**: whether the phrase the rule found was claimed or denied. "The detail is
extraordinary" and "there is not much detail" are both detail talk, and the flag is true for both.
The polarity column beside it reads the first as *asserted* and the second as *negated*. The flag
stays a topic marker, because the topic is what the questions in sections 4 and 5 are about. The
polarity travels with it in a separate column, so the integration step can split the flag either way
without the rule deciding in advance which reading matters. Polarity is read with the same negation
window that prior-exposure talk uses (described below), extended with quantity words that deny an
amount: *little*, *not much*, *lack of*, *minimal*, *low* and so on. "Little detail" denies detail
as plainly as "no detail" does. *Little* is read that way only where it names an amount ("little
detail", "very little", "too little", "little to no"). Where it names a size ("a little", "the
little details", "many small little parts") it denies nothing. So *little* is not read as a denial
when the word directly before it is *a*, *the*, *some* or *such*, or a word for an amount or a size
(*many*, *so many*, *a lot of*, *lots of*, *all the*, *small*, *tiny*).

Two of the eight are paired answers to the two questions the notebook asks about wording.
`says_detail` records talk about the visible surface of the picture, which is the leading candidate
for what "effort" denoted. `says_concept` records talk about the idea, the meaning or the story
behind the work, which is the matching candidate for what "creativity" denoted. The two are written
to be symmetric, so that sections 4 and 5 can be read against each other.

The eighth, `says_machine_made`, is a narrow companion to `says_prompt`. The prompting rule fires on
any machine word. The machine-made rule fires only on a phrase about a machine making the work ("AI
generated", "made by a computer"), however the phrase is used. Beside it sits a reading column,
`machine_made_reading`, which separates the responses that put such a phrase forward as a statement
("it looks AI generated", "probably made by AI") from those that only deny it ("this does not look
AI generated") or suppose it ("if this were AI generated"). Section 6 reports the flag as mentions
of a machine making the work, and the asserted reading as statements that a machine made the work.

Each rule has two parts:

- an **include** pattern, listing the words and phrases that count; and
- an optional **exclude** pattern, listing places where those same words mean something else in this
  corpus. Six of the ten artworks contain an hourglass, so *hour glass* and *time passing* describe
  a picture, not how long the work took, and the exclusion list removes them. A phrase is counted
  only if it does not sit inside an excluded phrase.

Each rule also carries a **grounding** note: a short statement of the construct the rule measures,
meaning the reading of effort or creativity it stands for and the literature that reading comes
from. The note is not part of the pattern and does not enter the rules' checksum. Section 13 writes
the notes beside the rules as `tables/text_rules_grounding.txt`.

**The rules are fixed before any result is looked at.** Fixing them first is what lets the counts in
sections 4 to 7 answer the questions. Had the patterns been adjusted after seeing which way the
numbers came out, the numbers would only describe the adjusting. The rules are printed in full
below, so they travel with the exported notebook. Section 3 prints matched and unmatched responses,
so a reader can see what each rule caught.

Two more pieces of machinery are defined here:

- **The negation window.** Prior-exposure talk has to distinguish "I have seen this before" from "I
  have never seen anything like this". The rule looks at the 30 characters immediately before the
  matched phrase, and at the matched phrase itself, for a negating word (*not*, *never*, *n't*,
  *no*, *unlike*, and so on), so that "I have never seen" and "I haven't seen" are both read as
  denials. Two phrases are the exception: "nothing new" and "not new" each assert familiarity, and
  they are read as asserted whatever they contain. Thirty characters is roughly five or six words,
  long enough to reach back over "I do not think I have" and short enough not to reach far back into
  the sentence. The window is also cut at the nearest clause boundary (a comma, or a word such as
  *because*, *but* or *since*), because a negation does not reach across one. In "I didn't rate it
  as high because I have definitely seen many pictures like this", the *didn't* belongs to *rate*,
  not to *seen*, and the writer is asserting familiarity rather than denying it. The polarity
  reading of detail, time, skill and concept talk uses the same window, extended with the quantity
  words listed in the cell below, so that *little detail* and *lack of skill* are read as denials
  even though neither contains a negating word in the ordinary sense. The reading of machine-made
  talk uses the same window, the same negating words, and one more short list: words that open a
  supposition (*if*, *whether*, *unless*, *even if*, but not *as if*). A machine-making phrase is
  read as a statement unless a negating word sits inside it or in the window before it, or a
  supposition word sits in that window. Hedged impressions ("probably AI", "looks AI generated") are
  read as statements, because they put the claim forward.
- **The function-word list.** Section 8 asks which words are distinctive of high versus low ratings.
  Function words, such as *the*, *is* and *of*, carry grammar rather than content and would crowd
  the answer. They are ignored there, and in sections 9 to 11, which also work with the words
  themselves. The list below holds function words only (articles, pronouns, prepositions, auxiliary
  verbs, conjunctions and numbers) and no content words.

In [3]:
# ---------------------------------------------------------------------------
# The eight rules.
#
# Each rule is stored as a small dictionary with five fields:
#   name       the column it produces in the exported file
#   plain      a one-line description in ordinary words
#   grounding  a short note on the construct the rule measures: the reading of
#              effort or creativity it stands for, where that reading comes from
#              in the literature, and what the rule does and does not record.
#              The note is not part of the pattern. Section 13 writes it to
#              tables/text_rules_grounding.txt, and the checksum on
#              text_rules.txt does not cover it.
#   include    the search pattern for phrases that count
#   exclude    the search pattern for places those phrases mean something else
#              (an empty string means nothing is excluded)
#
# The patterns are written in "verbose" mode, switched on by the (?x) at the
# start. In verbose mode the computer ignores spaces and line breaks inside the
# pattern, so the pattern can be laid out one alternative per line and read like
# a list. A literal space therefore has to be written as "\ " or "\s".
#
# Notation used below:
#   \b        a word boundary, so "\bart\b" matches "art" but not "artist"
#   (?:a|b)   either a or b
#   ?         the preceding piece is optional
#   [^.]{0,25}  up to 25 characters that are not a full stop, i.e. "a few words
#               later, but still in the same sentence"
# ---------------------------------------------------------------------------

RULES = []

# --- 1. Detail talk -----------------------------------------------------------
# Visible properties of the picture surface: how much is in it, how finely it is
# rendered, how the paint or line behaves. The last two alternatives catch the
# quantity-of-content idiom ("a lot going on", "many different parts"), which is
# detail talk without using the word "detail". The quantity phrases are tied to
# a content noun on purpose: "a lot of effort" is effort talk, not detail talk,
# and must not be caught here.
RULES.append(dict(
    name="says_detail",
    plain="mentions visible detail, intricacy, complexity, texture, shading, brushwork, "
          "or quantity of content",
    grounding="Detail talk is the visible-work reading of effort: the marks on the picture "
              "surface from which a viewer infers how much labour went in, since the labour "
              "itself cannot be seen. Buell and Norton (2011) call the value people add to an "
              "identical result when they can see work being done on it the labour illusion, "
              "and Kruger, Wirtz, Van Boven and Altermatt (2004) show that judged quality "
              "follows believed effort most strongly when quality is hard to assess on its "
              "own, which a picture from an unfamiliar maker is. The rule records the cue and "
              "not the inference: a writer who names the detail is counted whether or not they "
              "went on to say it took work.",
    include=r"""(?x)
      \b detail (?:s|ed|ing)? \b
    | \b intricat (?:e|ely) \b | \b intricac (?:y|ies) \b
    | \b complex (?:ity)? \b | \b complicated \b
    | \b elaborate \b
    | \b textur (?:e|es|ed|al) \b
    | \b shading \b | \b shadow (?:s|ing)? \b | \b hatching \b
    | \b brush \s? (?: work | stroke s? ) \b | \b stroke s? \b
    | \b line \s? work \b | \b fine \s lines? \b
    | \b meticulous (?:ly)? \b | \b precise (?:ly)? \b | \b precision \b
    | \b layer (?:s|ed|ing)? \b
    | \b (?: a\ lot | lots | so\ much | too\ much | much | plenty | a\ great\ deal )
        \b [^.]{0,25}? \b going \s on \b
    | \b (?: many | multiple | numerous | several | so\ many
           | a\ lot\ of | lots\ of | tons\ of | ton\ of | plenty\ of )
        \s+ (?: different\s+ | small\s+ | little\s+ | tiny\s+ | minor\s+ )?
        (?: parts | elements | components | aspects | objects | things | pieces
          | layers | colou?rs | shapes | features | items | figures | details ) \b
    """,
    exclude=r"",
))

# --- 2. Time talk -------------------------------------------------------------
# Duration and industriousness: how long the work took, or how quickly it could
# be done. Speed words are included because a claim that something took seconds
# is a claim about duration, just with the sign reversed.
#
# The exclusions matter here. Several of the ten artworks show an hourglass and
# one shows a Christmas scene, so "hour glass", "time passing", "time running
# out", "carrying time" and "Christmas time" are descriptions of what is in the
# picture. "Minute details" uses "minute" as an adjective meaning small.
# "At the same time", "day to day" and "end of the day" are idioms, not clocks.
RULES.append(dict(
    name="says_time",
    plain="mentions time, hours, duration, speed, or tediousness",
    grounding="Time talk is the elapsed-labour reading of effort: how long the making took, or "
              "how quickly it could have been done. This is the cue Kruger, Wirtz, Van Boven "
              "and Altermatt (2004) manipulated to show the effort heuristic, the rule of "
              "thumb by which a poem, a painting or a suit of armour is rated better and worth "
              "more when it is said to have taken longer, and Chinander and Schweitzer (2003) "
              "show the same reliance on time as an input measure even where it has no bearing "
              "on the outcome being judged. Of the three effort readings this is the one "
              "closest to the labour that Feist declined to protect, so the rule catches "
              "duration claims in either direction, including the claim that something took no "
              "time at all.",
    include=r"""(?x)
      \b time \b
    | \b hours \b | \b an \s hour \b | \b minutes \b | \b a \s minute \b | \b seconds? \b
    | \b days \b | \b weeks \b | \b all \s day \b
    | \b a \s while \b | \b awhile \b
    | \b tedious (?:ness|ly)? \b | \b painstaking (?:ly)? \b
    | \b laborious \b | \b arduous \b | \b labor \b
    | \b spen[dt] (?:s|ing)? \b
    | \b took \b [^.]{0,25}? \b (?: hours? | days? | long ) \b
    | \b quick (?:ly|er)? \b | \b fast \b | \b instant (?:ly|aneous(?:ly)?)? \b
    | \b immediat (?:e|ely) \b | \b rapid (?:ly)? \b | \b in \s no \s time \b
    """,
    exclude=r"""(?x)
      christmas \s? time
    | \b time \s (?: thing | passing | as \s fire | running \s out )
    | carrying \s time
    | \b at \s the \s same \s time \b
    | \b day \s to \s day \b | \b end \s of \s the \s day \b | \b during \s the \s day \b
    | \b hour \s? glass \b
    | \b minute \s detail
    """,
))

# --- 3. Skill talk ------------------------------------------------------------
# Ability acquired by training: skill, craft, technique, talent, mastery, and the
# claim that something was done by hand. "Technically" is excluded because in
# this corpus it is a hedge ("it technically is novel"), not a claim about craft.
RULES.append(dict(
    name="says_skill",
    plain="mentions skill, craft, technique, talent, training, mastery, or hand execution",
    grounding="Skill talk is the execution reading of effort: ability acquired by training, "
              "and the craft with which something was done. Amabile's componential model "
              "(1983, 1996) treats domain-relevant skills, meaning technical skill and craft "
              "knowledge in a field, as one ingredient of creative work and keeps them apart "
              "from the creativity-relevant skills that make the work new, and her consensual "
              "assessment technique (1982) asks judges to rate creativity and technical "
              "goodness as separate dimensions and finds them related but not the same. The "
              "rule records that a writer located effort in ability rather than in time or in "
              "the picture surface; it does not record how much ability they thought was "
              "shown.",
    include=r"""(?x)
      \b skill (?:s|ed|ful|fully)? \b
    | \b craft (?:s|ed|ing|smanship|manship)? \b
    | \b techniqu e s? \b | \b technical \b
    | \b talent (?:s|ed)? \b
    | \b train (?:ed|ing) \b
    | \b master (?:y|ed|ful|ing)? \b
    | \b expert (?:ise|s)? \b
    | \b artistry \b
    | \b artistic \s (?: skill s? | abilit (?:y|ies) | talent | training ) \b
    | \b practic (?:e|ed|ing) \b | \b proficien (?:t|cy) \b
    | \b know [-\s] how \b
    | \b professional \b
    | \b good \s at \b
    | \b execut (?:ed|ion) \b
    | \b hand [-\s]? (?: drawn | painted | made | crafted ) \b | \b by \s hand \b
    | \b human \s abilities \b
    """,
    exclude=r"""(?x) \b technically \b | \b in \s practice \b """,
))

# --- 4. Concept talk ----------------------------------------------------------
# What the work is about rather than what it looks like: the idea, the meaning,
# the message, the story, the symbolism, the thought behind it. This is the
# candidate answer for creativity that parallels detail talk for effort. Detail
# talk locates effort in the picture surface: how much is in it, how finely it
# is rendered. Concept talk locates creativity behind the picture, in something
# the maker decided before any of it was rendered. If
# creativity justifications are concept talk while effort justifications are
# detail talk, the two ratings are about two different objects, and section 5
# is where that is checked.
#
# Two restrictions come from how this corpus uses the words.
#
# "Thought" on its own is not counted. In this corpus the bare word is almost
# always the past tense of "think" ("I thought that this image was creative"),
# which reports the writer's opinion and says nothing about the work having a
# thought behind it. Only the phrases in which "thought" is a noun about the
# making are counted: "a lot of thought", "well thought out", "the thought
# behind it", "thought-provoking".
#
# The noun "expression" is not counted either. Several of the ten artworks show
# faces, and in this corpus "expression" nearly always means a face rather than
# something being expressed, so only the verb forms are listed.
#
# "Imaginative" and "imagination" are left out. They are ways of
# saying a work IS creative, not statements about what creativity consists of,
# and counting them would make the rule a restatement of the rating rather than
# a description of the reasoning behind it.
RULES.append(dict(
    name="says_concept",
    plain="mentions the idea, meaning, message, story, symbolism, or thought behind the work",
    grounding="Concept talk locates creativity in the idea behind the work rather than in the "
              "picture surface. The construct is the lay implicit theory of creativity, "
              "meaning what ordinary people take the word to consist of when nobody hands them "
              "a definition: Sternberg (1985) found that laypeople's picture of a creative "
              "person turns on unconventional thinking and on imagination and aesthetic sense, "
              "which he found absent from their picture of intelligence, and Runco and Bahleda "
              "(1986) found that the picture differs by domain, so artistic creativity carries "
              "a profile of its own. Leder, Belke, Oeberst and Augustin (2004) place "
              "interpretation, the viewer's attempt to make sense of what a work is about, "
              "among the stages of taking in a work of art, which is why words for meaning, "
              "message and story sit beside words for the idea. The rule counts a mention of "
              "the idea; it does not judge whether the idea was any good.",
    include=r"""(?x)
      \b idea s? \b
    | \b concept (?: s | ual | ually )? \b
    | \b meaning (?: s | ful | fully )? \b
    | \b message s? \b
    | \b symbol (?: s | ic | ically | ism | ize | izes | ized | izing )? \b
    | \b metaphor (?: s | ical | ically )? \b
    | \b theme s? \b | \b thematic \b
    | \b stor (?: y | ies | yline | ytelling ) \b | \b backstory \b
    | \b narrative s? \b
    | \b represent (?: s | ed | ing | ation | ations )? \b
    | \b convey (?: s | ed | ing )? \b
    | \b express (?: es | ed | ing ) \b
    | \b interpret (?: s | ed | ing | ation | ations )? \b
    | \b intent (?: ion | ions | ional | ionality )? \b
    | \b vision (?: ary )? \b
    | \b thoughtful (?: ly | ness )? \b
    | \b thought [-\s] (?: provoking | out | through | up | behind | process | put ) \b
    | \b (?: much | little | some | any | no | more | a\ lot\ of | lots\ of
           | a\ great\ deal\ of | plenty\ of ) \s+ thought \b
    | \b well [-\s] thought \b
    """,
    exclude=r"""(?x) \b (?: have | had | has | i ) \s+ no \s+ idea \b """,
))

# --- 5. Prompt talk -----------------------------------------------------------
# Any mention of operating a machine to make the picture: prompting, typing,
# generating, or naming a machine, a program or an image generator by kind or by
# brand. In the arm told a machine made the work this is ordinary description; in
# the arm told a human made it, it records that a machine came up at all, which
# section 6 counts beside the narrower machine-made rule.
#
# "Type" is restricted on purpose. The bare noun is common in this corpus in the
# sense "a type of painting", which has nothing to do with a keyboard, so only
# the verb forms and the typing-at-a-machine phrases are counted.
RULES.append(dict(
    name="says_prompt",
    plain="mentions prompting, typing, generating, or names a machine or image generator",
    grounding="Prompt talk is a written trace of what the participant believed about how the "
              "picture was made. The instrument carries no manipulation check, meaning no "
              "question asking whether the participant took in the vignette, and there is "
              "support for leaving one out: a check placed inside a study is an intervention "
              "as well as a measure and can change what follows it (Hauser, Ellsworth and "
              "Gonzalez 2018), and a question that asks whether a cover story was believed is "
              "itself a demand characteristic, Orne's (1962) term for a cue that tells a "
              "participant what the study is about. Chamberlain, Mullin, Scheerlinck and "
              "Wagemans (2018) found that viewers could not tell computer-made pictures from "
              "hand-made ones and still rated pictures lower when told a computer made them, "
              "so a participant told a person made these could plausibly doubt it. The rule "
              "records any mention of a machine or of operating one, in either arm. It does not "
              "record what the writer believed about who made the pictures.",
    include=r"""(?x)
      \b prompt (?:s|ed|ing)? \b
    | \b typ (?: ing | ed ) \b
    | \b type \s (?: in | into | out | commands? | words? | some
                   | a\ prompt | a\ sentence | a\ description | a\ few\ words ) \b
    | \b (?: just | only | but | was ) \s type \b
    | \b generat (?: e|ed|es|ing|or|ors|ion|ive ) \b
    | \b a \.? i \.? \b | \b artificial \s intelligence \b
    | \b machine s? \b | \b computer (?:s|ized)? \b | \b algorithm (?:s|ic)? \b
    | \b software \b | \b program (?:s|med|ming)? \b | \b robot (?:s|ic)? \b
    | \b chat \s? gpt \b | \b midjourney \b | \b dall [-\s]? e \b | \b stable \s diffusion \b
    | \b input (?:s|ed|ted|ting)? \b
    """,
    exclude=r"",
))

# --- 6. Prior-exposure talk ---------------------------------------------------
# The claim that the work is already familiar: seen before, generic, typical,
# common, derivative, a cliche. This is the measure with three values rather than
# two, because "I have seen this before" and "I have never seen anything like
# this" are opposite claims about the same thing. The negation window below
# separates them. The contracted denials "haven't", "hasn't" and "hadn't" are
# listed among the leading words in their own right, because the bare "n't"
# alternative cannot find them: there is no word boundary between "haven" and
# "'t", so "haven't seen" would otherwise go unmatched.
#
# "Similar to each other" is excluded because in this corpus it describes
# repetition inside one picture (the stuffed animals are similar to each other),
# not resemblance to something the rater had seen elsewhere. "Been done" is
# required to be followed by a before-word, because "more could have been done"
# is not a familiarity claim. "Replicate" is absent from the list: every use of
# it in this corpus means "reproduce by hand" ("easy landscape to replicate"),
# which is a claim about difficulty, not about having seen the work before.
RULES.append(dict(
    name="prior_exposure",
    plain="says the work looks familiar, already seen, generic or derivative "
          "(asserted), explicitly denies it (negated), or neither",
    grounding="Prior-exposure talk records a familiarity judgment: the claim that the work has "
              "been seen before, is generic or typical, or the explicit denial of that. "
              "Familiarity is a criterion in its own right in the literature on aesthetic "
              "preference, through mere exposure, the finding that repeated encounters with a "
              "thing raise liking for it (Zajonc 1968), and through typicality, the closeness "
              "of a thing to the usual example of its kind, which Hekkert, Snelders and van "
              "Wieringen (2003) show works alongside novelty in shaping preference while "
              "pulling against it. On the creativity side the standard definition requires "
              "originality as well as effectiveness (Runco and Jaeger 2012), and Diedrich, "
              "Benedek, Jauk and Neubauer (2015) find that judged creativity leans more on "
              "novelty than on usefulness, so a rater who reaches for newness is applying a "
              "criterion the field itself uses. The rule takes three values because asserting "
              "familiarity and denying it are different claims about the same thing.",
    include=r"""(?x)
      \b (?: have | had | has | haven'?t | hasn'?t | hadn'?t | i'?ve | i'?d | never | not | n't
           | ever | already )
        \s (?:\w+\s){0,2} seen \b
    | \b seen \b [^.]{0,25}? \b (?: before | already | elsewhere | previously
                                  | times | everywhere ) \b
    | \b familiar (?:ity)? \b | \b unfamiliar \b
    | \b reminds? \s me \s of \b | \b reminiscent \b
    | \b similar \s to \b
    | \b generic (?:ally)? \b | \b clich [eé] (?:d|s)? \b
    | \b overdone \b | \b overused \b | \b cookie [-\s]? cutter \b
    | \b common (?:ly|place)? \b | \b typical (?:ly)? \b | \b ordinary \b | \b standard \b
    | \b unoriginal \b | \b derivative \b | \b copied \b | \b copying \b
    | \b recycled \b
    | \b done \s before \b
    | \b been \s done \s (?: before | already | many | several | a \s \w+ ) \b
    | \b nothing \s new \b | \b not \s new \b
    | \b i \s recognize \b
    """,
    exclude=r"""(?x)
      \b similar \s to \s (?: each \s other | one \s another | the \s others? \b )
    """,
))

# --- 7. Anchor echo -----------------------------------------------------------
# The rating scale's own labels were "No / Minimal / Moderate / Significant /
# A great deal of creativity (or effort) involved", with "A lot of effort
# involved" shown as an alternative wording for the fourth point. This rule
# records responses that contain the scale's own wording. It fires on the
# wording alone and says nothing about what follows it: "a great deal of
# effort because every edge was painted by hand" contains the scale's wording
# and is flagged, though it goes on to give a reason. It is not one of the
# three questions; it is a check on how much of the corpus contains the
# scale's wording, which is worth knowing before reading anything into the
# rest.
RULES.append(dict(
    name="anchor_echo",
    plain="contains the rating scale's own wording",
    grounding="Anchor echo records an explanation that contains the rating scale's own labels. "
              "The motivation is satisficing, Krosnick's (1991) term for giving an answer that "
              "is acceptable rather than the best one can, of which the least effortful form "
              "for an open question is to hand back the words already supplied, joined to "
              "Schwarz's (1999) observation that respondents read the wording of a scale as a "
              "guide to what an acceptable answer looks like. The rule does not measure "
              "satisficing. It fires on the scale's wording wherever the wording appears, "
              "including in an explanation that goes on to give a full reason, so it records "
              "that the wording is present and nothing about what else the explanation "
              "carries. It is a check on the corpus rather than one of the three questions.",
    include=r"""(?x)
      \b (?: no | minimal | moderate | significant | a\ lot\ of | a\ great\ deal\ of )
        \s+ (?: creativity | effort ) \b (?: \s+ involved )?
    | \b (?: creativity | effort ) \s+ (?: was \s+ | is \s+ )? involved \b
    """,
    exclude=r"",
))

# --- 8. Machine-made talk ------------------------------------------------------
# A narrow companion to the prompt rule. Prompt talk fires on any machine word;
# this rule fires only on a phrase about a machine MAKING the work, which
# section 6 counts as a mention of a machine making the work, whatever is said
# with it. Three shapes count: a machine noun followed, within the same
# sentence, by a making verb ("the AI generated
# it", "computer drawn"); a making verb followed by "by", "with", "using" or
# "from" and a machine noun ("made with AI", "generated by a computer"); and the
# fixed compounds "ai-generated", "computer generated", "machine-made", "ai art",
# "ai image".
#
# The machine nouns and the making verbs are written once each, just below, and
# dropped into the pattern in three places, so that the three shapes cannot
# drift apart. The making verbs are stems where a stem is safe ("generat"
# covers generated, generates and generator) and whole words where it is not:
# "creat" is spelled out as its verb forms and the noun "creation", because the
# bare stem would also match "creative" and "creativity", which name the rating
# rather than an act of making, and "did" carries a word boundary so that
# "didn't" is not read as a making verb.
#
# The exclusion removes sentences in which a person is the agent of tool use:
# "Alex used AI to paint it" says that a person made the work with a tool, which
# is the vignette's own account in the machine arm, not a claim that the machine
# was the maker. The excluded stretch runs from the person to the end of the
# sentence, so that a making verb later in the same sentence does not count. A
# mention of AI as a subject of discussion with no making verb near it ("I don't
# think AI is bad") never matches the include pattern in the first place. The
# flag records the phrase whatever is said with it, so "it does not look AI
# generated" is flagged. Whether a phrase was put forward as a statement,
# denied, or only supposed is read separately, by classify_machine_made() in
# section 2, into the column machine_made_reading.
MACHINE_NOUN = r"""(?: a\.?i | artificial \s intelligence | computer | algorithm | machine | bot
      | software | program | generator | neural \s network | midjourney | dall [-\s]? e
      | stable \s diffusion | chat \s? gpt ) s?"""
MAKING_VERB = r"""(?: generat | creat (?: e | es | ed | ing | ion | ions ) \b | produc
      | mak (?: e | es | ing | er ) \b | made \b | draw | drew \b | paint | render | design
      | did \b | done \b | built \b | output )"""
RULES.append(dict(
    name="says_machine_made",
    plain="mentions a machine, a program or an image generator making the work, whether "
          "stated, denied or supposed",
    grounding="Machine-made talk records a mention of a machine making the work: a phrase such "
              "as 'AI generated' or 'made by a computer', however it is used. The reading "
              "column beside it separates statements that a machine made the work, hedged "
              "impressions such as 'probably AI' among them, from denials ('this does not "
              "look AI generated') and suppositions ('if this were AI generated'). Neither "
              "the flag nor the statements measure disbelief in the vignette. The instrument "
              "has no manipulation check, a participant can doubt the story without writing "
              "so, and a participant can write that a picture looks machine-made while "
              "accepting what they were told about who made it; the counts describe what was "
              "written.",
    include=r"""(?x)
      \b %(noun)s
      \b [^.]{0,25}? \b
      %(verb)s
    | \b %(verb)s
      \w* [^.]{0,20}? \b (?: by | with | using | from ) \s+ (?: (?: a | an | the ) \s+ )?
      \b %(noun)s
      \b
    | \b (?: ai | computer ) [-\s] generated \b
    | \b (?: machine | ai ) [-\s] made \b
    | \b ai \s (?: art | image | artwork ) \b
    """ % dict(noun=MACHINE_NOUN, verb=MAKING_VERB),
    exclude=r"""(?x)
      (?: \b %(verb)s
          \w* \s+ by \s+ )?
      \b (?: alex | he | she | they | the \s artist | someone | a \s person | humans? ) \s+
      (?: used | using | use | uses | utiliz \w* | employ \w* ) \s+ (?: an? \s+ )?
      (?: ai | computer | software | program | algorithm | generator | machine ) [^.]*
    """ % dict(verb=MAKING_VERB),
))

# --- the negation window ------------------------------------------------------
# Used by the prior-exposure rule, and by the polarity reading of the detail,
# time, skill and concept rules in section 2. If any of these words appears in
# the 30 characters immediately before the matched phrase, or, for prior-exposure
# talk, inside the matched phrase itself, the claim is being denied.
NEGATION_WINDOW = 30
NEGATION_PATTERN = r"""(?x)
  \b not \b | \b never \b | n't \b | \b no \b | \b nothing \b | \b none \b
| \b hardly \b | \b rarely \b | \b unlike \b | \b without \b
| \b cannot \b | \b can't \b | \b barely \b | \b nowhere \b
"""

# --- quantity words that deny an amount ---------------------------------------
# Used only by the polarity reading of the four topic rules, alongside the list
# above. "Little detail", "lack of skill" and "minimal thought" deny their topic
# as plainly as "no detail" does, but through a word that names a small amount
# rather than through a negating word in the ordinary sense, so the list above
# on its own would read all three as claims. Prior-exposure talk does not use
# this list: "little" or "low" before "familiar" is not a denial of familiarity.
QUANTIFIER_NEGATION = r"""(?x)
  \b little \b | \b not \s much \b | \b not \s a \s lot \b | \b not \s many \b
| \b lack \b | \b lacking \b | \b lacks \b
| \b no \b | \b minimal \b | \b barely \b | \b hardly \b | \b without \b | \b low \b
"""

# A negating word only counts if it is in the same clause as the phrase it is
# supposed to negate. In "I didn't rate it as high because I have definitely seen
# many pictures like this one", the "didn't" belongs to "rate", not to "seen",
# and the writer is asserting familiarity, not denying it. So the window is cut
# at the nearest clause boundary: only the text after the last of these markers
# is searched for a negating word.
CLAUSE_BOUNDARY = r"""(?x)
  , | ; | \. | \b because \b | \b but \b | \b however \b | \b since \b
| \b although \b | \b though \b | \b whereas \b | \b so \s that \b
"""

# A phrase can also carry its own negation in its spelling: "unfamiliar" is the
# denial of "familiar" with no separate negating word to find. A negating word
# from the list above sitting inside the phrase ("have never seen", "haven't
# seen") is found by the same search that reads the window, in section 2.
SELF_NEGATING = r"""(?x) ^ (?: unfamiliar | never | un [-] )"""

# Two phrases assert familiarity through a negating word: "nothing new" and
# "not new" both say that the work is familiar. They are read as asserted before
# any negation reading is applied, so that neither the negating word inside
# them nor the self-negation reading turns them into denials.
FAMILIARITY_ASSERTED = r"""(?x) ^ (?: nothing | not ) \s new $"""

# "Little" names an amount in "little detail", "very little", "too little" and
# "little to no", and a size in "a little", "the little details" and "a lot of
# small little details". Only the first use denies anything. So "little" counts
# as a quantifier negation only when the word or words directly before it are
# none of those listed here: "a", "the", "some", "such", or a word for an
# amount or a size. The pattern is tested against the text that ends where
# "little" begins, which is what the $ at its end means.
LITTLE_AS_SIZE = r"""(?x)
  \b (?: a | the | some | such | many | so \s many | a \s lot \s of | lots \s of
       | all \s the | small | tiny ) \s+ $
"""

# --- words that open a supposition --------------------------------------------
# Used only by the reading of machine-made talk. "If this were AI generated",
# "whether AI made it" and "unless a computer did it" mention a machine making
# the work without saying that one did, and "even if" is caught by "if". "As
# if" is the exception: "it looks as if a computer made it" puts the
# impression forward, so an "if" directly after "as" does not count; the
# (?<! ...) at the start of the first alternative means "not directly after".
CONDITIONAL_MARKER = r"""(?x)
  (?<! \b as \s ) \b if \b | \b whether \b | \b unless \b
"""

print("%d rules defined:" % len(RULES), ", ".join(r["name"] for r in RULES))

8 rules defined: says_detail, says_time, says_skill, says_concept, says_prompt, prior_exposure, anchor_echo, says_machine_made


In [4]:
# ---------------------------------------------------------------------------
# The function-word list.
#
# Used only in sections 8 to 11, which work with the words of the responses
# themselves rather than with rules. Function words carry grammar, not content, and they
# are the most common words in any English text, so leaving them in would fill
# the answer with "the", "is" and "of".
#
# Warning, and the reason this list is written out by hand: scikit-learn ships a
# built-in English stop list, and it is not a function-word list. It contains
# content words including "detail", "take", "many", "much", "show", "made",
# "part", "front", "system", "call" and "find". Using it here would delete
# "detail" (the most important word in this corpus, and the answer to
# question 1) along with several other words the analysis is about.
# The list below therefore contains function words only: articles, pronouns,
# prepositions, conjunctions, auxiliary and copular verbs, and bare numbers.
# Nothing in it names a property of an artwork.
# ---------------------------------------------------------------------------
FUNCTION_WORDS = sorted(set("""
a about above after again against all almost along already also although always am among an and
another any anything are aren't around as at
be because been before being below between both but by
came can can't cannot could couldn't
did didn't do does doesn't doing don't down during
each either else enough even ever every everything except
few for from further
had hadn't has hasn't have haven't having he he's her here hers herself him himself his how however
i i'd i'll i'm i've if in into is isn't it it's its itself
just
let let's like
may maybe me might mine more most must my myself
near neither never nevertheless no nor not nothing now
of off often on once one only onto or other others ought our ours ourselves out over own
per perhaps
quite
rather really
same seem seemed seems shall she she's should shouldn't since so some someone something sometimes
still such
than that that's the their theirs them themselves then there there's therefore these they they're
this those though through thus to too toward towards
under until up upon us
very
was wasn't we we're well were weren't what whatever when where whereas whether which while who whom
whose why will with within without won't would wouldn't
yet you you're your yours yourself
two three four five six seven eight nine ten
""".split()))

print("Function words in the ignore list:", len(FUNCTION_WORDS))
print()
print(textwrap.fill(" ".join(FUNCTION_WORDS), width=100))
print()
for content_word in ["detail", "take", "many", "much", "show", "made", "part", "time", "skill"]:
    assert content_word not in FUNCTION_WORDS, \
        "%s is a content word and must not be in the ignore list" % content_word
print("Checked: no content word from the rules above appears in the ignore list.")

Function words in the ignore list: 229

a about above after again against all almost along already also although always am among an and
another any anything are aren't around as at be because been before being below between both but by
came can can't cannot could couldn't did didn't do does doesn't doing don't down during each eight
either else enough even ever every everything except few five for four from further had hadn't has
hasn't have haven't having he he's her here hers herself him himself his how however i i'd i'll i'm
i've if in into is isn't it it's its itself just let let's like may maybe me might mine more most
must my myself near neither never nevertheless nine no nor not nothing now of off often on once one
only onto or other others ought our ours ourselves out over own per perhaps quite rather really same
seem seemed seems seven shall she she's should shouldn't since six so some someone something
sometimes still such ten than that that's the their theirs them themselves

In [5]:
# ---------------------------------------------------------------------------
# Print every rule in full, so that the exported notebook contains the complete
# definition of every number in it. This output is also written to
# tables/text_rules.txt in section 13, with a checksum beside it.
# ---------------------------------------------------------------------------
def format_rules(rules):
    """Return the full text of every rule in `rules`, ready to print or to save."""
    out = []
    out.append("=" * 100)
    out.append("STUDY 1 TEXT MEASURES: THE RULES IN FULL")
    out.append("=" * 100)
    out.append("")
    out.append("Every measure below is applied by computer to all 2,220 responses, identically,")
    out.append("with the response text lower-cased first. No response is judged individually.")
    out.append("Patterns are written in verbose mode: spaces and line breaks inside a pattern are")
    out.append("ignored by the matcher, so a literal space is written as \\  or \\s.")
    out.append("")
    for i, rule in enumerate(rules, start=1):
        out.append("-" * 100)
        out.append("RULE %d OF %d: %s" % (i, len(rules), rule["name"]))
        out.append("-" * 100)
        out.append("In plain words: " + rule["plain"])
        out.append("")
        out.append("COUNTS AS A MATCH:")
        for line in rule["include"].strip().splitlines():
            out.append("    " + line.strip())
        if rule["exclude"].strip():
            out.append("")
            out.append("DOES NOT COUNT, even though the words above appear "
                       "(these mean something else in this corpus):")
            for line in rule["exclude"].strip().splitlines():
                out.append("    " + line.strip())
        out.append("")
    out.append("-" * 100)
    out.append("THE NEGATION WINDOW (used by prior_exposure AND by the four polarity readings:")
    out.append("detail_polarity, time_polarity, skill_polarity, concept_polarity)")
    out.append("-" * 100)
    out.append("The window is the %d characters immediately before a matched phrase. For a"
               % NEGATION_WINDOW)
    out.append("prior-exposure phrase, a negating word from the first list below inside that")
    out.append("window means the response is recorded as 'negated' rather than 'asserted'. For a")
    out.append("detail, time, skill or concept phrase, the same window is searched for the same")
    out.append("negating words and also for the quantifier negations in the second list, which")
    out.append("name a small amount rather than deny a fact. Thirty characters is about five or")
    out.append("six words: long enough to reach back over 'I do not think I have', short enough not to")
    out.append("reach far into the sentence.")
    out.append("")
    out.append("Negating words, used by prior_exposure and by all four polarity readings:")
    for line in NEGATION_PATTERN.strip().splitlines():
        out.append("    " + line.strip())
    out.append("")
    out.append("Quantifier negations, used by the four polarity readings only, never by")
    out.append("prior_exposure ('little' or 'low' before 'familiar' is not a denial of familiarity):")
    for line in QUANTIFIER_NEGATION.strip().splitlines():
        out.append("    " + line.strip())
    out.append("")
    out.append("'little' is a quantifier negation only where it names an amount ('little detail',")
    out.append("'very little', 'too little', 'little to no'). It is not read as one when the words")
    out.append("directly before it match this pattern, which marks it as a size ('a little', 'the")
    out.append("little details', 'many small little parts'):")
    for line in LITTLE_AS_SIZE.strip().splitlines():
        out.append("    " + line.strip())
    out.append("")
    out.append("A negating word counts only if it sits in the same clause as the phrase it")
    out.append("negates. The window is therefore cut at the nearest of these clause markers,")
    out.append("and only the text after the last one is searched for a negating word:")
    for line in CLAUSE_BOUNDARY.strip().splitlines():
        out.append("    " + line.strip())
    out.append("")
    out.append("For prior_exposure, a negating word from the first list sitting inside the matched")
    out.append("phrase itself ('have never seen', 'haven't seen') also records it as 'negated', and")
    out.append("so does a phrase that carries its own negation in its spelling:")
    out.append("    " + SELF_NEGATING.strip())
    out.append("Two phrases assert familiarity through a negating word and are recorded as")
    out.append("'asserted' before either reading is applied:")
    out.append("    " + FAMILIARITY_ASSERTED.strip())
    out.append("")
    out.append("-" * 100)
    out.append("THE POLARITY READING (says_detail, says_time, says_skill, says_concept)")
    out.append("-" * 100)
    out.append("Each of these four flags records a topic, whatever the writer said about it. A")
    out.append("separate polarity column beside each flag records whether the matched phrase was")
    out.append("claimed or denied: 'negated' if a negating word or a quantifier negation from the")
    out.append("two lists under THE NEGATION WINDOW above sits in the same clause-cut window")
    out.append("before the phrase or begins the phrase itself; 'asserted' otherwise; 'neither'")
    out.append("when the flag is false. The quantifier negations deny an amount rather than a")
    out.append("fact, which is why they are kept apart from the negating words and are not used")
    out.append("for prior-exposure talk.")
    out.append("")
    out.append("-" * 100)
    out.append("THE MACHINE-MADE READING (machine_made_reading, beside says_machine_made)")
    out.append("-" * 100)
    out.append("says_machine_made records a phrase about a machine making the work, however it is")
    out.append("used. machine_made_reading reads every such phrase in the response, outside the")
    out.append("rule's exclusions. A phrase is not asserted if a negating word from the first list")
    out.append("under THE NEGATION WINDOW sits inside it or in the same clause-cut window before it,")
    out.append("or if a word that opens a supposition, from the list below, sits in that window;")
    out.append("otherwise it is asserted, hedged impressions ('probably AI', 'looks AI generated')")
    out.append("included. The response is 'asserted' if any of its phrases is asserted,")
    out.append("'not_asserted' if it has phrases and none is asserted, and 'neither' exactly when")
    out.append("says_machine_made is false.")
    out.append("")
    out.append("Words that open a supposition ('as if' does not count):")
    for line in CONDITIONAL_MARKER.strip().splitlines():
        out.append("    " + line.strip())
    out.append("")
    out.append("-" * 100)
    out.append("THE FUNCTION-WORD IGNORE LIST (used only in sections 8 to 11)")
    out.append("-" * 100)
    out.append("Function words only. It contains no content words, and in particular it does not")
    out.append("contain 'detail', 'take', 'many', 'much', 'show', 'made' or 'part', all of which")
    out.append("appear in scikit-learn's built-in English stop list and would be deleted by it.")
    out.append(textwrap.fill(" ".join(FUNCTION_WORDS), width=96,
                             initial_indent="    ", subsequent_indent="    "))
    out.append("")
    return "\n".join(out)

RULES_TEXT = format_rules(RULES)
print(RULES_TEXT)

STUDY 1 TEXT MEASURES: THE RULES IN FULL

Every measure below is applied by computer to all 2,220 responses, identically,
with the response text lower-cased first. No response is judged individually.
Patterns are written in verbose mode: spaces and line breaks inside a pattern are
ignored by the matcher, so a literal space is written as \  or \s.

----------------------------------------------------------------------------------------------------
RULE 1 OF 8: says_detail
----------------------------------------------------------------------------------------------------
In plain words: mentions visible detail, intricacy, complexity, texture, shading, brushwork, or quantity of content

COUNTS AS A MATCH:
    (?x)
    \b detail (?:s|ed|ing)? \b
    | \b intricat (?:e|ely) \b | \b intricac (?:y|ies) \b
    | \b complex (?:ity)? \b | \b complicated \b
    | \b elaborate \b
    | \b textur (?:e|es|ed|al) \b
    | \b shading \b | \b shadow (?:s|ing)? \b | \b hatching \b
    | \b brush \s? (?

---

# 2. Applying the rules

This section runs the rules over the text. Two features of how it does so are what make the counts
in later sections trustworthy.

**The measurement never sees the experimental condition.** The flags are computed from a copy of the
data, called the blind slice, that holds only the identifiers, the artwork, the question and the
text. The condition column and the rating column are not in that slice, and the cell below asserts
that they are not. A rule applied by a person, or by a machine that can see which arm a response
came from, can drift towards the expected answer without anyone intending it. A search pattern
applied to text with the condition removed cannot: the same words produce the same flag whichever
arm they came from.

**The flags are topic markers, not endorsements.** `says_detail` is true for "there is barely any
detail here". The rule records that detail is what the writer is talking about, which is the
question section 4 asks. Where assertion and denial are different claims, as in prior-exposure talk,
the measure has three values instead of two. For detail, time, skill and concept talk, the direction
of the claim is kept as well, in a separate polarity column beside each flag: *asserted*, *negated*,
or *neither* when the flag is false. The flag answers "was this the topic?"; the polarity answers
"and was it claimed or denied?". Both are computed from the same blind slice, in the same cell.

**Machine-made talk has a reading column of its own.** `machine_made_reading` takes three values:
*asserted* when at least one machine-making phrase in the response is put forward as a statement,
*not_asserted* when every such phrase is denied or only supposed, and *neither* when the rule found
none. It is therefore *neither* exactly where `says_machine_made` is false. The four polarity
columns read only the first phrase their rule finds. This reading looks at every phrase, because one
response can deny in one sentence that a machine made the work and say in the next that one did. At
the end of the cell, a set of constructed sentences (written for this check, not taken from the
corpus) is run through the machine-made reading and through the reading of *little*. The notebook
stops if any of them is read differently from what the rules in section 1 say.

The cell below also records `n_words`, a count of whitespace-separated words, which is used to
describe the corpus and, in section 10, to choose readable example sentences.

In [6]:
# ---------------------------------------------------------------------------
# Build the blind slice: the only columns the measurement is allowed to see.
#
# raw contains Type (the experimental condition) and rating (the outcome).
# Neither belongs anywhere near the measurement, so we take a copy without them
# and assert what the copy contains. If a later edit ever adds a forbidden
# column to this slice, this assertion stops the notebook.
# ---------------------------------------------------------------------------
BLIND_COLUMNS = ["id", "stimulus_id", "stimulus_num", "stimulus_type", "dimension", "response_text"]

blind = raw[BLIND_COLUMNS].copy()

FORBIDDEN = ["Type", "rating", "condition", "arm"]
present = [c for c in blind.columns if c in FORBIDDEN]
assert not present, (
    "The measurement slice contains a column it must never see: %s. "
    "Remove it from BLIND_COLUMNS." % present
)
assert list(blind.columns) == BLIND_COLUMNS, (
    "The measurement slice has unexpected columns: %s" % list(blind.columns)
)
print("The measurement will see only these columns:", list(blind.columns))
print("Condition and rating are absent. Checked.")
print()

# ---------------------------------------------------------------------------
# The matching machinery.
#
# find_match(text, include, exclude) returns the FIRST phrase that matches the
# include pattern and is not sitting inside a phrase matched by the exclude
# pattern, or None if there is no such phrase. It returns the position as well
# as the phrase, because section 3 needs to mark the phrase inside the sentence.
# ---------------------------------------------------------------------------
def compile_rule(rule):
    """Turn a rule's text patterns into compiled matchers, once, for speed."""
    inc = re.compile(rule["include"])
    exc = re.compile(rule["exclude"]) if rule["exclude"].strip() else None
    return inc, exc

NEG_RE = re.compile(NEGATION_PATTERN)
QUANT_NEG_RE = re.compile(QUANTIFIER_NEGATION)
SELF_NEG_RE = re.compile(SELF_NEGATING)
FAMILIARITY_ASSERTED_RE = re.compile(FAMILIARITY_ASSERTED)
CLAUSE_RE = re.compile(CLAUSE_BOUNDARY)
LITTLE_AS_SIZE_RE = re.compile(LITTLE_AS_SIZE)
CONDITIONAL_RE = re.compile(CONDITIONAL_MARKER)


def find_match(text, include_re, exclude_re):
    """First matching phrase that is not inside an excluded phrase.

    Returns (phrase, start, end) or None. Positions are into the lower-cased
    text, which is what the patterns are written against.
    """
    blocked = [m.span() for m in exclude_re.finditer(text)] if exclude_re is not None else []
    for m in include_re.finditer(text):
        start, end = m.span()
        if any(start >= b0 and end <= b1 for b0, b1 in blocked):
            continue
        return (m.group(0), start, end)
    return None


def all_matches(text, include_re, exclude_re):
    """Every matching phrase that is not inside an excluded phrase, in order.
    The first of them is the phrase find_match() returns."""
    blocked = [m.span() for m in exclude_re.finditer(text)] if exclude_re is not None else []
    found = []
    for m in include_re.finditer(text):
        start, end = m.span()
        if any(start >= b0 and end <= b1 for b0, b1 in blocked):
            continue
        found.append((m.group(0), start, end))
    return found


# lower-cased text is what every pattern is written against
lower_text = blind["response_text"].fillna("").astype(str).str.lower().tolist()

# ---------------------------------------------------------------------------
# Reading a matched phrase as claimed or denied.
#
# Two of the readings below need to know whether the writer was asserting the
# matched phrase or denying it, and both use the same window: the 30 characters
# immediately before the phrase, cut back to the start of the clause the phrase
# sits in. Cutting at the clause boundary is what keeps "I didn't rate it as
# high because I have definitely seen this" on the asserted side, where it
# belongs: the "didn't" belongs to "rate", not to "seen".
# ---------------------------------------------------------------------------
def clause_window_start(text, start):
    """Where the window before a phrase that begins at position `start` starts:
    NEGATION_WINDOW characters back, moved forward to just after the last
    clause boundary in that stretch, so that only the part in the same clause
    as the phrase is kept."""
    lo = max(0, start - NEGATION_WINDOW)
    boundaries = [m.end() for m in CLAUSE_RE.finditer(text[lo:start])]
    return lo + (boundaries[-1] if boundaries else 0)


def clause_window(text, start):
    """The stretch of text searched for a negating word before a phrase that
    begins at position `start`: up to NEGATION_WINDOW characters, keeping only
    the part that sits in the same clause as the phrase."""
    return text[clause_window_start(text, start):start]


# prior_exposure: three values rather than two.
#
# If there is no matching phrase the answer is "neither". If there is one, a
# negating word in the window, a negating word inside the phrase itself ("have
# never seen", "haven't seen"), or a phrase that negates itself ("unfamiliar"),
# means the writer is denying familiarity rather than claiming it. "Nothing
# new" and "not new" are checked first and read as asserted, because each says
# the work is familiar. Only the ordinary negating words count here: "little"
# or "low" before "familiar" is not a denial of familiarity, so the quantity
# list is not consulted.
def classify_prior_exposure(text, match):
    if match is None:
        return "neither"
    phrase, start, _end = match
    if FAMILIARITY_ASSERTED_RE.match(phrase):
        return "asserted"
    window = clause_window(text, start)
    if SELF_NEG_RE.match(phrase) or NEG_RE.search(phrase) or NEG_RE.search(window):
        return "negated"
    return "asserted"


# Polarity for detail, time, skill and concept talk: asserted, negated, or
# neither when the rule found nothing.
#
# The flag itself stays a topic marker, true for "barely any detail" as much as
# for "incredible detail". This reading sits beside it and records the
# direction. A phrase is read as negated if an ordinary negating word ("not",
# "n't", "never") or a quantity word that denies an amount ("little", "lack of",
# "minimal", "low") sits in the clause-cut window before it, or begins the
# phrase itself: "little thought" and "no thought" are matched whole by the
# concept rule, so their denial is inside the phrase rather than in front of it.
# "Little" counts only where it names an amount, which quantifier_denies()
# checks against LITTLE_AS_SIZE, so "the little details are extraordinary" is
# detail asserted. The argument little_as_size lets the cell count, further
# down, how many readings the size exception decides; the measurement always
# uses its default.
def quantifier_denies(text, position, word, little_as_size=True):
    """Whether the quantity word `word`, found at `position` in `text`, denies
    an amount. Every quantity word does, except "little" used for a size."""
    if word != "little" or not little_as_size:
        return True
    return LITTLE_AS_SIZE_RE.search(text[:position]) is None


def classify_polarity(text, match, little_as_size=True):
    if match is None:
        return "neither"
    phrase, start, _end = match
    window_start = clause_window_start(text, start)
    window = text[window_start:start]
    if NEG_RE.search(window) or NEG_RE.match(phrase):
        return "negated"
    for found in QUANT_NEG_RE.finditer(window):
        if quantifier_denies(text, window_start + found.start(), found.group(0), little_as_size):
            return "negated"
    found = QUANT_NEG_RE.match(phrase)
    if found and quantifier_denies(text, start, found.group(0), little_as_size):
        return "negated"
    return "asserted"


# Machine-made talk: asserted, not_asserted, or neither when the rule found
# nothing.
#
# Every phrase the says_machine_made pattern finds outside its exclusions is
# read in turn, not only the first, because one response can deny in one
# sentence what it says in the next. A phrase is not asserted if a negating
# word sits inside it ("AI didn't make") or in the clause-cut window before it
# ("does not look AI generated"), or if a word that opens a supposition sits in
# that window ("if this were AI generated"). Every other phrase is asserted,
# hedged impressions included ("probably made by AI", "looks AI generated"),
# because they put the claim forward. The supposition words are searched for
# in the whole text and kept only where they fall inside the window, so that
# the "as if" exception can see an "as" sitting just outside the window.
def read_machine_phrase(text, phrase, start):
    window_start = clause_window_start(text, start)
    if NEG_RE.search(phrase) or NEG_RE.search(text[window_start:start]):
        return "not_asserted"
    if any(window_start <= found.start() < start for found in CONDITIONAL_RE.finditer(text)):
        return "not_asserted"
    return "asserted"


def classify_machine_made(text, matches):
    """The response's reading and the phrase that decided it: the first
    asserted phrase if there is one, otherwise the first phrase found."""
    if not matches:
        return "neither", None
    for phrase, start, end in matches:
        if read_machine_phrase(text, phrase, start) == "asserted":
            return "asserted", (phrase, start, end)
    return "not_asserted", matches[0]


# The four flags that carry a polarity column, and what that column is called.
# The order here is the order the columns take in the exported file.
POLARITY_COLUMNS = {
    "says_detail":  "detail_polarity",
    "says_time":    "time_polarity",
    "says_skill":   "skill_polarity",
    "says_concept": "concept_polarity",
}


# ---------------------------------------------------------------------------
# Apply a full set of rules to the blind slice.
#
# Written as a function so that the whole measurement is one call on the blind
# slice and nothing else. The function returns two things: a frame with
# one row per response holding the flags, the polarities and the word count,
# and the matched phrase with its position for every response and rule, so
# that section 3 can show its work and section 13 does not have to re-run
# anything. For machine-made talk the phrase kept beside the reading is the one
# that decided it, under the key machine_made_reading.
# ---------------------------------------------------------------------------
def apply_rules(rules):
    compiled = {rule["name"]: compile_rule(rule) for rule in rules}
    spans = {}
    for rule in rules:
        inc, exc = compiled[rule["name"]]
        spans[rule["name"]] = [find_match(t, inc, exc) for t in lower_text]

    frame = blind.copy()

    # the straightforward true/false flags
    for rule in rules:
        if rule["name"] == "prior_exposure":
            continue
        frame[rule["name"]] = [m is not None for m in spans[rule["name"]]]

    # the three-valued familiarity measure
    frame["prior_exposure"] = [
        classify_prior_exposure(t, m)
        for t, m in zip(lower_text, spans["prior_exposure"])
    ]

    # the four polarity columns, each checked against its flag: "neither"
    # exactly when the flag is false, never otherwise
    for flag, column in POLARITY_COLUMNS.items():
        frame[column] = [classify_polarity(t, m) for t, m in zip(lower_text, spans[flag])]
        assert ((frame[column] == "neither") == ~frame[flag]).all(), (
            "%s is 'neither' on a response where %s is true, or the reverse." % (column, flag))

    # the reading of machine-made talk, checked against its flag in the same
    # way, with the phrase that decided each reading kept for sections 3 and 6
    if "says_machine_made" in compiled:
        inc, exc = compiled["says_machine_made"]
        readings = [classify_machine_made(t, all_matches(t, inc, exc)) for t in lower_text]
        frame["machine_made_reading"] = [reading for reading, _ in readings]
        spans["machine_made_reading"] = [span for _, span in readings]
        assert ((frame["machine_made_reading"] == "neither") == ~frame["says_machine_made"]).all(), (
            "machine_made_reading is 'neither' on a response where says_machine_made is true, "
            "or the reverse.")

    # word count: the number of whitespace-separated words
    frame["n_words"] = blind["response_text"].fillna("").astype(str).str.split().str.len()
    return frame, spans


measures, MATCH_SPANS = apply_rules(RULES)

print("Flags computed for %d responses." % len(measures))
print()
print("HOW OFTEN EACH RULE FIRED, across the whole corpus (both arms together):")
for rule in RULES:
    name = rule["name"]
    if name == "prior_exposure":
        counts = measures[name].value_counts()
        print("  %-17s asserted %4d (%4.1f%%)   negated %3d (%3.1f%%)   neither %4d (%4.1f%%)" % (
            name,
            counts.get("asserted", 0), 100 * counts.get("asserted", 0) / len(measures),
            counts.get("negated", 0), 100 * counts.get("negated", 0) / len(measures),
            counts.get("neither", 0), 100 * counts.get("neither", 0) / len(measures)))
    else:
        n = int(measures[name].sum())
        print("  %-17s %4d of %d responses (%4.1f%%)" % (name, n, len(measures),
                                                         100 * n / len(measures)))
print()
print("POLARITY, among the responses each of the four topic rules flagged:")
for flag, column in POLARITY_COLUMNS.items():
    flagged = measures.loc[measures[flag], column]
    n_asserted = int((flagged == "asserted").sum())
    n_negated = int((flagged == "negated").sum())
    print("  %-17s asserted %4d (%4.1f%%)   negated %3d (%4.1f%%)   of %4d flagged" % (
        column, n_asserted, 100 * n_asserted / max(len(flagged), 1),
        n_negated, 100 * n_negated / max(len(flagged), 1), len(flagged)))
print()
n_flagged = int(measures["says_machine_made"].sum())
reading_counts = measures["machine_made_reading"].value_counts()
print("MACHINE-MADE READING, among the responses says_machine_made flagged:")
print("  %-17s asserted %4d (%4.1f%%)   not_asserted %3d (%4.1f%%)   of %4d flagged" % (
    "machine_made_reading",
    reading_counts.get("asserted", 0), 100 * reading_counts.get("asserted", 0) / max(n_flagged, 1),
    reading_counts.get("not_asserted", 0), 100 * reading_counts.get("not_asserted", 0) / max(n_flagged, 1),
    n_flagged))
print()
print("THE SIZE READING OF 'LITTLE'. Flagged responses read as negated, with 'little' read as a")
print("denial only where it names an amount, and with every 'little' read as a denial:")
for flag, column in POLARITY_COLUMNS.items():
    n_size_reading = int((measures[column] == "negated").sum())
    n_every_little = sum(classify_polarity(t, m, little_as_size=False) == "negated"
                         for t, m in zip(lower_text, MATCH_SPANS[flag]))
    print("  %-17s negated %3d with the size reading, %3d with every 'little' a denial"
          % (column, n_size_reading, n_every_little))
print()
print("WORD COUNTS")
print("  median %d, mean %.1f, shortest %d, longest %d" % (
    measures["n_words"].median(), measures["n_words"].mean(),
    measures["n_words"].min(), measures["n_words"].max()))
print("  quartiles:", measures["n_words"].quantile([.25, .5, .75]).round(1).tolist())

# ---------------------------------------------------------------------------
# Constructed sentences, written for this check and not taken from the corpus,
# each with the reading the rules in section 1 give it. They go through the
# same compiled rules and the same reading functions as the 2,220 responses,
# and the notebook stops if any is read otherwise. The first group covers the
# machine-made reading: two denials, two suppositions, two impressions and an
# "as if". The second covers the size reading of "little" in detail talk.
# ---------------------------------------------------------------------------
MACHINE_PROBES = [
    ("This does not look AI generated.", "not_asserted"),
    ("AI didn't make this.", "not_asserted"),
    ("If this were AI generated, it would be less creative.", "not_asserted"),
    ("I wonder whether AI made this.", "not_asserted"),
    ("It looks AI generated.", "asserted"),
    ("This was probably made by AI.", "asserted"),
    ("It looks as if a computer made it.", "asserted"),
]
DETAIL_PROBES = [
    ("The little details are extraordinary.", "asserted"),
    ("There are a lot of small little details.", "asserted"),
    ("It has a little detail in the corner.", "asserted"),
    ("There is very little detail.", "negated"),
    ("Too little detail for a painting.", "negated"),
    ("Little to no detail anywhere.", "negated"),
]
machine_inc, machine_exc = compile_rule(next(r for r in RULES if r["name"] == "says_machine_made"))
detail_inc, detail_exc = compile_rule(next(r for r in RULES if r["name"] == "says_detail"))
print()
print("CONSTRUCTED SENTENCES (not from the corpus) and how the rules read them:")
for sentence, expected in MACHINE_PROBES:
    probe = sentence.lower()
    reading, _span = classify_machine_made(probe, all_matches(probe, machine_inc, machine_exc))
    print("  machine_made_reading  %-13s %s" % (reading, sentence))
    assert reading == expected, "%r is read as %s; the rules say %s." % (sentence, reading, expected)
for sentence, expected in DETAIL_PROBES:
    probe = sentence.lower()
    polarity = classify_polarity(probe, find_match(probe, detail_inc, detail_exc))
    print("  detail_polarity       %-13s %s" % (polarity, sentence))
    assert polarity == expected, "%r is read as %s; the rules say %s." % (sentence, polarity, expected)
print("Every constructed sentence is read as the rules say. Checked.")

The measurement will see only these columns: ['id', 'stimulus_id', 'stimulus_num', 'stimulus_type', 'dimension', 'response_text']
Condition and rating are absent. Checked.

Flags computed for 2220 responses.

HOW OFTEN EACH RULE FIRED, across the whole corpus (both arms together):
  says_detail        703 of 2220 responses (31.7%)
  says_time          108 of 2220 responses ( 4.9%)
  says_skill          56 of 2220 responses ( 2.5%)
  says_concept       365 of 2220 responses (16.4%)
  says_prompt        540 of 2220 responses (24.3%)
  prior_exposure    asserted  200 ( 9.0%)   negated  70 (3.2%)   neither 1950 (87.8%)
  anchor_echo        322 of 2220 responses (14.5%)
  says_machine_made  264 of 2220 responses (11.9%)

POLARITY, among the responses each of the four topic rules flagged:
  detail_polarity   asserted  650 (92.5%)   negated  53 ( 7.5%)   of  703 flagged
  time_polarity     asserted  100 (92.6%)   negated   8 ( 7.4%)   of  108 flagged
  skill_polarity    asserted   54 (96.4%) 

---

# 3. Checking the rules by eye

This section prints what each rule caught, so that a reader can inspect the measurement rather than
take it on trust. No number in the notebook depends on that inspection.

For each rule, the notebook prints 40 responses the rule matched, with the triggering phrase wrapped
in `>>>` and `<<<`, and 20 responses it did not match. The same output is written to
`tables/text_audit_matches.txt` so it can be read outside the notebook.

The matched responses bear on one question: is this detail talk, or has the rule caught a word used
in some other sense? The unmatched responses bear on the opposite one: is this detail talk that the
rule missed because the writer used a word nobody put on the list? The first kind of error is
visible here. The second kind is not, because a rule cannot show what it never looked for. Section
10 approaches it from the other side, by asking the corpus which words keep company with each rule's
own words without being on its list.

For prior-exposure talk, all three categories are shown: matched-and-asserted, matched-and-denied,
and unmatched. The denied category is small, and it is where the negation window either works or
does not. Detail, time, skill and concept talk are split the same way, into matched-and-asserted and
matched-and-denied. Their denied groups show the same thing. A quantity word such as *low* can
describe something else in the picture rather than deny an amount ("low contrast and rich detail"),
and the denied group is where a reader can see how often the window takes the one for the other.
Machine-made talk is split by its reading in the same way, into matched-and-asserted and
matched-and-not-asserted, with the phrase that decided the reading marked.

In [7]:
# ---------------------------------------------------------------------------
# Print matched and unmatched examples for every rule, with the triggering
# phrase marked in the sentence.
#
# The phrase is located in the lower-cased text, but we display the ORIGINAL
# text with its capitals intact, using the same character positions. That works
# because lower-casing in Python does not change the length of these strings;
# the cell checks that before relying on it.
# ---------------------------------------------------------------------------
original_text = blind["response_text"].fillna("").astype(str).tolist()
assert all(len(a) == len(b) for a, b in zip(original_text, lower_text)), \
    "Lower-casing changed the length of a response, so marked positions would be wrong."

N_MATCHES_TO_SHOW = 40
N_NONMATCHES_TO_SHOW = 20
AUDIT_SEED = 20260903   # fixed so the same responses are shown on every run


def mark(text, start, end):
    """Return the response with the triggering phrase wrapped in >>> <<<."""
    return text[:start] + ">>>" + text[start:end] + "<<<" + text[end:]


def wrap_for_print(s, width=104, indent="        "):
    return textwrap.fill(s, width=width, initial_indent=indent, subsequent_indent=indent)


def audit_lines():
    """Build the full audit text: examples for every rule."""
    rng = np.random.default_rng(AUDIT_SEED)
    out = []
    out.append("=" * 104)
    out.append("STUDY 1 TEXT MEASURES: MATCHED AND UNMATCHED RESPONSES, RULE BY RULE")
    out.append("=" * 104)
    out.append("")
    out.append("The triggering phrase is wrapped in >>> and <<<.")
    out.append("For prior-exposure talk and for detail, time, skill and concept talk, the matched")
    out.append("responses are split by whether the phrase was claimed or denied. For the four topic")
    out.append("rules that split is the polarity column beside the flag (detail_polarity,")
    out.append("time_polarity, skill_polarity and concept_polarity), and each listed response")
    out.append("carries the value that column holds for it. Machine-made talk is split by")
    out.append("machine_made_reading into asserted and not asserted, and the phrase marked is the")
    out.append("one that decided the reading.")
    out.append("Responses are drawn with a fixed random seed (%d), so this file is identical"
               % AUDIT_SEED)
    out.append("every time the notebook is run.")
    out.append("")

    for rule in RULES:
        name = rule["name"]
        spans = MATCH_SPANS[name]

        # the column whose value is shown beside each listed response: the
        # three-valued measure itself for prior_exposure, the polarity column
        # for the four topic rules, nothing for the other two
        value_column = None
        if name == "prior_exposure":
            value_column = name
            groups = [
                ("MATCHED, read as ASSERTED (the writer says the work is familiar)",
                 [i for i in range(len(spans)) if measures[name].iloc[i] == "asserted"]),
                ("MATCHED, read as NEGATED (the writer denies the work is familiar)",
                 [i for i in range(len(spans)) if measures[name].iloc[i] == "negated"]),
            ]
            unmatched = [i for i in range(len(spans)) if spans[i] is None]
        elif name in POLARITY_COLUMNS:
            column = POLARITY_COLUMNS[name]
            value_column = column
            groups = [
                ("MATCHED, %s = asserted (no negating word or quantifier negation in the "
                 "same clause before the phrase)" % column,
                 [i for i in range(len(spans)) if measures[column].iloc[i] == "asserted"]),
                ("MATCHED, %s = negated (a negating word or quantifier negation sits in the "
                 "same clause before the phrase, or begins it)" % column,
                 [i for i in range(len(spans)) if measures[column].iloc[i] == "negated"]),
            ]
            unmatched = [i for i in range(len(spans)) if spans[i] is None]
        elif name == "says_machine_made":
            value_column = "machine_made_reading"
            groups = [
                ("MATCHED, machine_made_reading = asserted (at least one phrase put forward as "
                 "a statement)",
                 [i for i in range(len(spans)) if measures[value_column].iloc[i] == "asserted"]),
                ("MATCHED, machine_made_reading = not_asserted (every phrase denied, or only "
                 "supposed)",
                 [i for i in range(len(spans)) if measures[value_column].iloc[i] == "not_asserted"]),
            ]
            unmatched = [i for i in range(len(spans)) if spans[i] is None]
        else:
            groups = [("MATCHED", [i for i in range(len(spans)) if spans[i] is not None])]
            unmatched = [i for i in range(len(spans)) if spans[i] is None]

        out.append("=" * 104)
        out.append("RULE: %s. In plain words: %s" % (name, rule["plain"]))
        out.append("=" * 104)
        if value_column is not None:
            counts = measures[value_column].value_counts()
            categories = (["asserted", "not_asserted", "neither"]
                          if value_column == "machine_made_reading"
                          else ["asserted", "negated", "neither"])
            out.append("column %s: %s (of %d responses)" % (
                value_column, ", ".join("%s %d" % (c, counts.get(c, 0)) for c in categories),
                len(measures)))

        # the phrase marked: for machine-made talk, the one that decided the reading
        mark_spans = MATCH_SPANS["machine_made_reading"] if name == "says_machine_made" else spans
        for heading, idx in groups:
            k = min(N_MATCHES_TO_SHOW, len(idx))
            chosen = rng.choice(idx, size=k, replace=False) if k else []
            out.append("")
            out.append("%s  (%d responses in total; showing %d)" % (heading, len(idx), k))
            out.append("-" * 104)
            for j, i in enumerate(sorted(chosen), start=1):
                phrase, start, end = mark_spans[i]
                line = "  %2d. %s | %s | %s" % (
                    j, measures['id'].iloc[i], measures['stimulus_id'].iloc[i],
                    measures['dimension'].iloc[i])
                if value_column is not None:
                    line += " | %s: %s" % (value_column, measures[value_column].iloc[i])
                out.append(line)
                out.append(wrap_for_print(mark(original_text[i], start, end)))

        k = min(N_NONMATCHES_TO_SHOW, len(unmatched))
        chosen = rng.choice(unmatched, size=k, replace=False) if k else []
        out.append("")
        out.append("NOT MATCHED  (%d responses in total; showing %d)" % (len(unmatched), k))
        out.append("-" * 104)
        for j, i in enumerate(sorted(chosen), start=1):
            out.append("  %2d. %s | %s | %s" % (
                j, measures['id'].iloc[i], measures['stimulus_id'].iloc[i],
                measures['dimension'].iloc[i]))
            out.append(wrap_for_print(original_text[i]))
        out.append("")

    return "\n".join(out)


AUDIT_TEXT = audit_lines()
print(AUDIT_TEXT)

STUDY 1 TEXT MEASURES: MATCHED AND UNMATCHED RESPONSES, RULE BY RULE

The triggering phrase is wrapped in >>> and <<<.
For prior-exposure talk and for detail, time, skill and concept talk, the matched
responses are split by whether the phrase was claimed or denied. For the four topic
rules that split is the polarity column beside the flag (detail_polarity,
time_polarity, skill_polarity and concept_polarity), and each listed response
carries the value that column holds for it. Machine-made talk is split by
machine_made_reading into asserted and not asserted, and the phrase marked is the
one that decided the reading.
Responses are drawn with a fixed random seed (20260903), so this file is identical
every time the notebook is run.

RULE: says_detail. In plain words: mentions visible detail, intricacy, complexity, texture, shading, brushwork, or quantity of content
column detail_polarity: asserted 650, negated 53, neither 1517 (of 2220 responses)

MATCHED, detail_polarity = asserted (no ne

---

# 4. What people meant by "effort"

This is the first point at which the experimental condition is joined back on.

**Why the order matters.** The flags in section 2 were computed from text with the condition column
removed, so no rule could have been shaped, consciously or otherwise, by which arm a response came
from. Joining the condition on afterwards cannot change a single flag, because the values already
exist. Had the join come first, every count below would rest on a promise that the arm was ignored,
rather than on the order of the computation. The join is on the three columns that identify a
response uniquely (participant, artwork, question), and the cell checks that every row found exactly
one match.

**The question.** The paper reads the effort rating as a lay version of the "sweat of the brow"
standard that *Feist* rejected: reward for labour expended. But "effort" in ordinary speech can mean
at least three different things, and they are not legally equivalent. It can mean the labour itself
(time, tedium, industriousness), which is what *Feist* refused to protect. It can mean skill (craft,
technique, training), which sits closer to what copyright does care about. Or it can mean the
visible product of labour, how much is in the picture and how finely it is rendered, which is a
property of the artwork rather than of the making.

The table below counts which of the three people wrote about, in the arm that was told a human
artist made the work. Creativity justifications from the same participants are shown beside them,
because a rate means little without something to compare it to. If detail talk were simply what
these participants wrote about all the time, it would show up in both columns.

**How to read the table.** Each cell is the percentage of justifications of that kind that contain
at least one phrase from that rule. The columns need not add to 100, because a response can mention
all three things, or none. The right-hand column is the difference in percentage points between
effort justifications and creativity justifications from the same people. It shows what is specific
to the effort question rather than general to the writer.

Section 5 puts the same question to the creativity justifications, in a table of the same shape, so
that the two answers can be laid side by side.

In [8]:
# ---------------------------------------------------------------------------
# Join the experimental condition and the rating back on.
#
# measures has one row per response and no condition. raw has both. The join key
# is (id, stimulus_id, dimension), which identifies a response uniquely: the
# checks in section 0 confirmed there are no duplicates. We use an inner join
# and then assert the row count, so a key mismatch cannot silently drop rows.
# ---------------------------------------------------------------------------
KEY = ["id", "stimulus_id", "dimension"]

joined = measures.merge(
    raw[KEY + ["Type", "rating"]],
    on=KEY, how="inner", validate="one_to_one",
)
assert len(joined) == len(measures), (
    "The join changed the number of rows, from %d to %d. Check the key columns "
    "(id, stimulus_id, dimension) for mismatches between the two files."
    % (len(measures), len(joined))
)
print("Condition and rating joined back on. Rows before %d, rows after %d."
      % (len(measures), len(joined)))
print("Arms:", joined.groupby("Type")["id"].nunique().to_dict(), "participants")
print()

# ---------------------------------------------------------------------------
# The human-attribution arm: what did effort talk consist of?
# ---------------------------------------------------------------------------
human_arm = joined[joined["Type"] == "Human"]
THREE = ["says_detail", "says_time", "says_skill"]
LABELS = {
    "says_detail": "Detail talk (what is in the picture, how finely rendered)",
    "says_time":   "Time talk (how long it took, how quickly it could be done)",
    "says_skill":  "Skill talk (craft, technique, talent, training)",
}

effort = human_arm[human_arm["dimension"] == "effort"]
creativity = human_arm[human_arm["dimension"] == "creativity"]

rows = []
for flag in THREE:
    n_e, pct_e = int(effort[flag].sum()), 100 * effort[flag].mean()
    n_c, pct_c = int(creativity[flag].sum()), 100 * creativity[flag].mean()
    rows.append({
        "What the response mentions": LABELS[flag],
        "Effort justifications": "%5.1f%%  (%3d of %d)" % (pct_e, n_e, len(effort)),
        "Creativity justifications": "%5.1f%%  (%3d of %d)" % (pct_c, n_c, len(creativity)),
        "Difference (points)": "%+6.1f" % (pct_e - pct_c),
    })

effort_table = pd.DataFrame(rows)

print("=" * 104)
print("WHAT 'EFFORT' MEANT, IN THE ARM TOLD A HUMAN ARTIST MADE THE WORK")
print("(%d participants, %d effort justifications and %d creativity justifications)"
      % (human_arm["id"].nunique(), len(effort), len(creativity)))
print("=" * 104)
print(effort_table.to_string(index=False))
print()
print("For comparison, the same three rules in the arm told an image generator made the work:")
ai_arm = joined[joined["Type"] == "AI"]
comparison = pd.DataFrame({
    "What the response mentions": [LABELS[f] for f in THREE],
    "Effort justifications": ["%5.1f%%" % (100 * ai_arm[ai_arm.dimension == "effort"][f].mean())
                              for f in THREE],
    "Creativity justifications": ["%5.1f%%" % (100 * ai_arm[ai_arm.dimension == "creativity"][f].mean())
                                  for f in THREE],
})
print(comparison.to_string(index=False))

Condition and rating joined back on. Rows before 2220, rows after 2220.
Arms: {'AI': 55, 'Human': 56} participants

WHAT 'EFFORT' MEANT, IN THE ARM TOLD A HUMAN ARTIST MADE THE WORK
(56 participants, 560 effort justifications and 560 creativity justifications)
                                What the response mentions Effort justifications Creativity justifications Difference (points)
 Detail talk (what is in the picture, how finely rendered)   62.5%  (350 of 560)       12.1%  ( 68 of 560)               +50.4
Time talk (how long it took, how quickly it could be done)   10.0%  ( 56 of 560)        1.4%  (  8 of 560)                +8.6
           Skill talk (craft, technique, talent, training)    5.2%  ( 29 of 560)        0.9%  (  5 of 560)                +4.3

For comparison, the same three rules in the arm told an image generator made the work:
                                What the response mentions Effort justifications Creativity justifications
 Detail talk (what is in the picture

### Reading the result

The three rules are not equally forgiving. The detail rule lists more phrases than the skill rule
does, so some of the gap between them is a gap between the rules. But differences of the size seen
here are not the kind a longer word list would close. The comparison that carries the argument is
also within a single rule, across the two questions and across the two arms, where the rule is held
constant.

If detail talk dominates effort justifications while time talk and skill talk are both rare, then
the effort rating in this study is mostly a judgment about the artwork's visible content, not about
the labour that went into it and not about the artist's craft. That matters for the *Feist* mapping
in two directions. It weakens the reading of the effort rating as pure sweat of the brow, since
raters are not mainly reporting labour. It also strengthens the underlying worry. Lay raters would
be inferring labour from surface elaboration (a picture with more in it is judged to have cost
more), which is the inference the sweat-of-the-brow doctrine formalised and *Feist* rejected.

The comparison column guards against an alternative reading: that these participants wrote about
detail whatever they were asked. Section 10 addresses the remaining worry, that the detail rule
catches more of its topic than the skill rule catches of its own, by asking the corpus, rule by
rule, which words each one may be leaving out.

---

# 5. What people meant by "creativity"

Section 4 asked what "effort" denoted: which of several things the word picked out for the people
using it. The same question has to be asked of "creativity", and it is the more consequential of the
two. *Feist v. Rural Telephone Service* set effort aside: sheer labour, what the older cases called
sweat of the brow, is not a ground of copyright. What *Feist* kept is a minimal creative spark.
Creativity is therefore the criterion that decides the case, and what these participants meant by
the word is what a lay application of the copyright standard would apply.

There are three candidate answers, chosen to parallel the three in section 4.

- **Concept**: the idea, the meaning, the message, the story, the thought behind the work. This is
  the answer that mirrors detail talk for effort. Detail talk locates effort in the picture surface;
  concept talk locates creativity behind the surface, in something the maker decided before any of
  it was rendered. Measured by the `says_concept` rule from section 1.
- **Novelty**: whether the work is new or already familiar. Measured by the prior-exposure rule from
  section 1, reused here rather than written a second time. A response counts as novelty talk if it
  makes the familiarity claim *or* denies it, because the question in this table is what creativity
  denoted, not which way the denotation cut. The split between the two is where the doctrine bites,
  and it is taken up below.
- **Execution**: visible detail and how finely the work is rendered. Measured by `says_detail`, the
  same rule section 4 used, here applied to creativity justifications rather than effort ones.

**Why two of the three rules are borrowed rather than rebuilt.** Novelty talk and execution talk are
measured by rules the notebook uses elsewhere. A second rule for either would be a second definition
of the same thing, free to drift away from the first. Reusing the rules means the execution row of
this table and the detail row of section 4's table are the same measurement counted in two places,
so the two tables can be laid beside each other and read as one picture. `says_concept` is the only
rule written for this question.

**How to read the table.** It has the same shape as section 4's. Each cell is the percentage of
justifications of that kind, in the arm told a human artist made the work, that contain at least one
phrase from that rule. The columns need not add to 100, because a response can mention all three
things, or none. The right-hand column is the difference in percentage points between creativity
justifications and effort justifications from the same participants. The subtraction runs the
opposite way from section 4's, so that in both tables a positive number means "specific to the
question this section is about".

The doctrinal reading depends on which of the three rows is large. If creativity denotes concept,
lay judgment is tracking something close to what copyright asks for. If it denotes execution, the
creativity rating and the effort rating are two readings of the same picture surface, and these
raters are not making the distinction *Feist* drew between them. If it denotes novelty, lay judgment
is importing the patent test, and that possibility is the subject of the rest of this section.

In [9]:
# ---------------------------------------------------------------------------
# The human-attribution arm: what did creativity talk consist of?
#
# human_arm, effort and creativity were all defined in section 4; the same three
# frames are reused here so that the two tables are built from exactly the same
# responses and can be compared line for line.
#
# The novelty row is the one place where a decision has to be made. prior_exposure
# has three values, not two: "asserted" is the writer saying the work looks
# familiar, "negated" is the writer saying they have never seen anything like it,
# and "neither" is the writer not raising the subject at all. Both of the first
# two are novelty talk, since they are opposite answers to the same question, so
# the row counts a response if prior_exposure is anything other than "neither". The
# split between asserted and negated is printed underneath, because that split,
# and not the total, is what the doctrinal argument turns on.
# ---------------------------------------------------------------------------
def novelty_talk(frame):
    """True where the response raises familiarity at all, in either direction."""
    return (frame["prior_exposure"] != "neither")


CREATIVITY_ROWS = [
    ("Concept talk (the idea, meaning, message, story behind it)",
     lambda f: f["says_concept"],
     "says_concept"),
    ("Novelty talk (the work is, or is not, already familiar)",
     novelty_talk,
     "prior_exposure, either direction"),
    ("Execution talk (visible detail, how finely it is rendered)",
     lambda f: f["says_detail"],
     "says_detail"),
]

rows = []
for label, getter, source in CREATIVITY_ROWS:
    hits_c = getter(creativity)
    hits_e = getter(effort)
    pct_c, pct_e = 100 * hits_c.mean(), 100 * hits_e.mean()
    rows.append({
        "What the response mentions": label,
        "Creativity justifications": "%5.1f%%  (%3d of %d)" % (pct_c, int(hits_c.sum()), len(creativity)),
        "Effort justifications": "%5.1f%%  (%3d of %d)" % (pct_e, int(hits_e.sum()), len(effort)),
        "Difference (points)": "%+6.1f" % (pct_c - pct_e),
        "Measured by": source,
    })

creativity_table = pd.DataFrame(rows)

print("=" * 104)
print("WHAT 'CREATIVITY' MEANT, IN THE ARM TOLD A HUMAN ARTIST MADE THE WORK")
print("(%d participants, %d creativity justifications and %d effort justifications)"
      % (human_arm["id"].nunique(), len(creativity), len(effort)))
print("Difference is creativity minus effort, so a positive number is specific to the "
      "creativity question.")
print("=" * 104)
print(creativity_table.to_string(index=False))
print()

# ---------------------------------------------------------------------------
# The same three rows in the arm told an image generator made the work, laid out
# exactly as the corresponding table in section 4, so the four tables in sections
# 4 and 5 can be read as one block.
# ---------------------------------------------------------------------------
print("For comparison, the same three measures in the arm told an image generator made the work:")
ai_creativity = ai_arm[ai_arm["dimension"] == "creativity"]
ai_effort_sub = ai_arm[ai_arm["dimension"] == "effort"]
comparison_creativity = pd.DataFrame({
    "What the response mentions": [label for label, _, _ in CREATIVITY_ROWS],
    "Creativity justifications": ["%5.1f%%" % (100 * getter(ai_creativity).mean())
                                  for _, getter, _ in CREATIVITY_ROWS],
    "Effort justifications": ["%5.1f%%" % (100 * getter(ai_effort_sub).mean())
                              for _, getter, _ in CREATIVITY_ROWS],
})
print(comparison_creativity.to_string(index=False))
print()

# ---------------------------------------------------------------------------
# Splitting the novelty row, since the two directions are opposite claims.
#
# "Asserted" is a rater saying the work is familiar, which under a novelty test
# counts against it. "Negated" is a rater volunteering that they have never seen
# anything like it, which under a novelty test counts for it. Both are the patent
# question being asked; only the answer differs.
# ---------------------------------------------------------------------------
print("The novelty row split by direction, human-attribution arm, creativity justifications:")
for category, gloss in [("asserted", "says the work looks familiar or already seen"),
                        ("negated", "says they have never seen anything like it")]:
    share = 100 * (creativity["prior_exposure"] == category).mean()
    n = int((creativity["prior_exposure"] == category).sum())
    print("  %-9s %-46s %5.1f%%  (%3d of %d)" % (category, gloss, share, n, len(creativity)))
print()

# ---------------------------------------------------------------------------
# How much of the concept row and the execution row is the same responses? A
# justification can do both ("a great idea, and beautifully rendered"), and it
# matters whether the two rows are describing two groups of people or one group
# writing about two things.
# ---------------------------------------------------------------------------
both = int((creativity["says_concept"] & creativity["says_detail"]).sum())
only_concept = int((creativity["says_concept"] & ~creativity["says_detail"]).sum())
only_detail = int((~creativity["says_concept"] & creativity["says_detail"]).sum())
neither_one = len(creativity) - both - only_concept - only_detail
print("Overlap between concept talk and execution talk in creativity justifications "
      "(human-attribution arm):")
print("  concept only:   %3d   execution only: %3d   both: %3d   neither: %3d   total: %d"
      % (only_concept, only_detail, both, neither_one, len(creativity)))

WHAT 'CREATIVITY' MEANT, IN THE ARM TOLD A HUMAN ARTIST MADE THE WORK
(56 participants, 560 creativity justifications and 560 effort justifications)
Difference is creativity minus effort, so a positive number is specific to the creativity question.
                                What the response mentions Creativity justifications Effort justifications Difference (points)                      Measured by
Concept talk (the idea, meaning, message, story behind it)       26.8%  (150 of 560)    4.8%  ( 27 of 560)               +22.0                     says_concept
   Novelty talk (the work is, or is not, already familiar)       24.5%  (137 of 560)    3.0%  ( 17 of 560)               +21.4 prior_exposure, either direction
Execution talk (visible detail, how finely it is rendered)       12.1%  ( 68 of 560)   62.5%  (350 of 560)               -50.4                      says_detail

For comparison, the same three measures in the arm told an image generator made the work:
                    

### The novelty component: is a patent test being applied?

The middle row of the table above counts novelty talk in both directions at once. This part splits
it, because the two directions are opposite claims.

Copyright asks for independent creation. A work made without copying is original in the legal sense
even if a hundred other people have made something just like it. That is the point of the standard,
and it is what separates copyright from patent, where novelty against everything that came before is
the whole test.

So it matters whether ordinary raters apply the patent test when asked about creativity. The
prior-exposure rule finds responses that say a work looks familiar, generic, typical, common,
derivative, or already seen. Separately, it finds responses that explicitly deny it. The table below
counts all three categories in creativity justifications, then applies the same rule to effort
justifications from the same participants. If prior-exposure talk is a feature of how these people
judge creativity in particular, it should be concentrated in the creativity column.

The second table shows the mean creativity rating in each of the three categories. Read it as a
description, not as a causal claim. Every justification was written after the rating it justifies,
by the same person, about the same trial, so the two are two expressions of one judgment rather than
a cause and its effect. What the table can say is whether the familiarity claim and the low rating
travel together.

In [10]:
# ---------------------------------------------------------------------------
# Prior-exposure talk: counts by question and by arm.
# ---------------------------------------------------------------------------
ORDER = ["asserted", "negated", "neither"]

def exposure_table(frame, title):
    tab = pd.crosstab(frame["dimension"], frame["prior_exposure"])
    for col in ORDER:
        if col not in tab.columns:
            tab[col] = 0
    tab = tab[ORDER]
    pct = (100 * tab.div(tab.sum(axis=1), axis=0)).round(1)
    out = pd.DataFrame({
        "asserted (looks familiar)": ["%d  (%.1f%%)" % (tab.loc[i, "asserted"], pct.loc[i, "asserted"])
                                      for i in tab.index],
        "negated (denies familiarity)": ["%d  (%.1f%%)" % (tab.loc[i, "negated"], pct.loc[i, "negated"])
                                         for i in tab.index],
        "neither": ["%d  (%.1f%%)" % (tab.loc[i, "neither"], pct.loc[i, "neither"])
                    for i in tab.index],
        "total": tab.sum(axis=1).values,
    }, index=tab.index)
    print(title)
    print("-" * 104)
    print(out.to_string())
    print()
    return tab

print("=" * 104)
print("PRIOR-EXPOSURE TALK: DOES FAMILIARITY COME UP WHEN PEOPLE JUDGE CREATIVITY?")
print("=" * 104)
print()
tab_human = exposure_table(joined[joined.Type == "Human"],
                           "ARM TOLD A HUMAN ARTIST MADE THE WORK")
tab_ai = exposure_table(joined[joined.Type == "AI"],
                        "ARM TOLD AN IMAGE GENERATOR MADE THE WORK")
tab_all = exposure_table(joined, "BOTH ARMS TOGETHER")

# ---------------------------------------------------------------------------
# Do the familiarity claim and the low rating travel together?
# ---------------------------------------------------------------------------
print("=" * 104)
print("MEAN CREATIVITY RATING (1-5) BY WHAT THE JUSTIFICATION SAID ABOUT FAMILIARITY")
print("=" * 104)
cre = joined[joined["dimension"] == "creativity"]
summary = (cre.groupby(["Type", "prior_exposure"])["rating"]
             .agg(responses="size", mean_rating="mean", sd="std")
             .round(2)
             .reindex(pd.MultiIndex.from_product([["Human", "AI"], ORDER],
                                                 names=["Type", "prior_exposure"])))
print(summary.to_string())
print()
for arm in ["Human", "AI"]:
    sub = cre[cre.Type == arm]
    a = sub[sub.prior_exposure == "asserted"]["rating"]
    n = sub[sub.prior_exposure == "neither"]["rating"]
    print("%-6s arm: responses calling the work familiar average %.2f; "
          "responses saying nothing about familiarity average %.2f; difference %+.2f points."
          % (arm, a.mean(), n.mean(), a.mean() - n.mean()))
print()
print("The same rule applied to EFFORT justifications, for contrast:")
eff = joined[joined["dimension"] == "effort"]
print((eff.groupby(["Type", "prior_exposure"])["rating"]
         .agg(responses="size", mean_rating="mean").round(2)).to_string())

PRIOR-EXPOSURE TALK: DOES FAMILIARITY COME UP WHEN PEOPLE JUDGE CREATIVITY?

ARM TOLD A HUMAN ARTIST MADE THE WORK
--------------------------------------------------------------------------------------------------------
           asserted (looks familiar) negated (denies familiarity)       neither  total
dimension                                                                             
creativity               95  (17.0%)                   42  (7.5%)  423  (75.5%)    560
effort                    15  (2.7%)                    2  (0.4%)  543  (97.0%)    560

ARM TOLD AN IMAGE GENERATOR MADE THE WORK
--------------------------------------------------------------------------------------------------------
           asserted (looks familiar) negated (denies familiarity)       neither  total
dimension                                                                             
creativity               76  (13.8%)                   23  (4.2%)  451  (82.0%)    550
effort                 

In [11]:
# ---------------------------------------------------------------------------
# Print the responses behind the counts: 18 of each category, with the phrase
# that triggered the classification wrapped in >>> <<<.
#
# The "neither" category holds the responses in which the rule found nothing,
# so it has no triggering phrase and its responses are printed plainly. A
# missed familiarity claim would sit among them.
# ---------------------------------------------------------------------------
N_EXAMPLES = 18
rng = np.random.default_rng(AUDIT_SEED + 1)

cre_idx = np.where((joined["dimension"] == "creativity").values)[0]
spans_exposure = MATCH_SPANS["prior_exposure"]

print("=" * 104)
print("CREATIVITY JUSTIFICATIONS, %d OF EACH CATEGORY" % N_EXAMPLES)
print("=" * 104)

for category in ORDER:
    pool = [i for i in cre_idx if joined["prior_exposure"].iloc[i] == category]
    k = min(N_EXAMPLES, len(pool))
    chosen = sorted(rng.choice(pool, size=k, replace=False)) if k else []
    print()
    print("-" * 104)
    print("%s   (%d creativity justifications in total; showing %d)"
          % (category.upper(), len(pool), k))
    print("-" * 104)
    for j, i in enumerate(chosen, start=1):
        rating = joined["rating"].iloc[i]
        arm = joined["Type"].iloc[i]
        if category == "neither":
            shown = original_text[i]
        else:
            phrase, start, end = spans_exposure[i]
            shown = mark(original_text[i], start, end)
        print("  %2d. creativity rating %d | told: %s" % (j, rating, arm))
        print(wrap_for_print(shown))

CREATIVITY JUSTIFICATIONS, 18 OF EACH CATEGORY

--------------------------------------------------------------------------------------------------------
ASSERTED   (171 creativity justifications in total; showing 18)
--------------------------------------------------------------------------------------------------------
   1. creativity rating 2 | told: AI
        This might take multiple prompts to perfect but it is still AI creating the image and filling in
        any details not stated in the prompt, which is >>>typically<<< a lot. Although creative I wonder
        if you could just input a prompt such as "humanoid alien" to generate a similar render.
   2. creativity rating 4 | told: Human
        just some snowy houses, i feel like this idea would be very >>>common<<< around christmas time
   3. creativity rating 1 | told: AI
        >>>Derivative<<<, sloppy. Doesn't look natural at all, and the work itself is uninspired.
   4. creativity rating 2 | told: AI
        This >>>remi

---

# 6. How often did participants told "a human made this" write that a machine made it?

Every participant saw the same ten images. Half were told an artist named Alex made them; half were
told Alex made them with an image generator. The images were in fact machine-generated, so the
human-attribution arm was given a cover story, and the instrument contains no manipulation check
(a question asking whether the participant took in the vignette).

This section counts what participants in that arm wrote about machines, at three levels of
strictness. The broad rule, `says_prompt`, counts the responses that name a machine or prompting at
all ("digital art programs", "an old computer desktop background"). The narrow rule,
`says_machine_made`, counts **mentions of a machine making the work**: responses with a phrase such
as "AI generated" or "made by a computer", however the phrase is used. Its reading column,
`machine_made_reading`, counts among those the **statements that a machine made the work**:
responses in which at least one such phrase is put forward rather than denied ("this does not look
AI generated") or only supposed ("if this were AI generated"). Hedged impressions ("probably AI",
"it looks AI generated") count as statements.

None of the three counts measures how many participants disbelieved the vignette. A participant can
doubt a story and write nothing about it, and a participant can write that a picture looks
machine-made while accepting what they were told about who made it. The counts describe what was
written. The two rules are written separately, so the narrow one is not guaranteed to sit inside
the broad one, and the cell reports how many responses the narrow rule flags that the broad one
does not.

Each count is reported two ways: how many responses, and how many distinct participants produced at
least one. The second is the one to quote, since a single participant can write the same remark ten
times. Every human-arm response the narrow rule flags is printed with its reading beside it and the
phrase that decided the reading marked.


In [12]:
# ---------------------------------------------------------------------------
# Human-attribution arm only: how often did a machine come up, and how often
# did a response say that a machine made the work?
#
# Three counts, from the broadest to the strictest. says_prompt: the response
# names a machine or prompting at all. says_machine_made: the response
# mentions a machine making the work, whether it states, denies or supposes
# it. machine_made_reading == "asserted": at least one of those phrases is
# put forward as a statement. Each is counted in responses and in distinct
# participants. The two rules are written separately, so the narrow one is not
# guaranteed to sit inside the broad one, and the cell counts the responses
# where it does not. Every human-arm response the narrow rule flags is printed
# with its reading and with the phrase that decided the reading marked.
# ---------------------------------------------------------------------------
human_mask = (joined["Type"] == "Human").values
broad_mask = joined["says_prompt"].values
mention_mask = joined["says_machine_made"].values
statement_mask = (joined["machine_made_reading"] == "asserted").values

n_human_participants = joined.loc[human_mask, "id"].nunique()
n_human_responses = int(human_mask.sum())


def count_in_human_arm(label, mask):
    """Print, and return the rows of, the human-arm responses where `mask` is true,
    counted in responses and in distinct participants."""
    idx = np.where(human_mask & mask)[0]
    people = joined["id"].iloc[idx].nunique()
    print("  %s" % label)
    print("    responses:                            %d of %d (%.1f%%)"
          % (len(idx), n_human_responses, 100 * len(idx) / n_human_responses))
    print("    participants producing at least one:  %d of %d (%.1f%%)"
          % (people, n_human_participants, 100 * people / n_human_participants))
    return idx


print("=" * 104)
print("MACHINES IN THE WRITING OF THE ARM TOLD A HUMAN ARTIST MADE THE WORK")
print("=" * 104)
broad_idx = count_in_human_arm(
    "NAMES A MACHINE OR PROMPTING AT ALL (says_prompt)", broad_mask)
mention_idx = count_in_human_arm(
    "MENTIONS OF A MACHINE MAKING THE WORK (says_machine_made)", mention_mask)
statement_idx = count_in_human_arm(
    "STATEMENTS THAT A MACHINE MADE THE WORK (machine_made_reading = asserted)", statement_mask)
print()
print("  mentions read as not asserted (every phrase denied or only supposed): %d"
      % int((human_mask & mention_mask & ~statement_mask).sum()))
print("  mentions the broad rule does not flag: %d"
      % int((human_mask & mention_mask & ~broad_mask).sum()))
print("  statements by question:", joined.iloc[statement_idx].groupby("dimension").size().to_dict())
if len(statement_idx):
    per_person = joined.iloc[statement_idx].groupby("id").size()
    print("  statements per participant, among those making one: median %.0f, most %d, fewest %d"
          % (per_person.median(), per_person.max(), per_person.min()))
print()
print("  For comparison, in the arm told an image generator made the work, where the vignette itself")
print("  says a machine was used: %.1f%% of responses name a machine at all, %.1f%% mention a machine"
      % (100 * ai_arm["says_prompt"].mean(), 100 * ai_arm["says_machine_made"].mean()))
print("  making the work, and %.1f%% state that a machine made it."
      % (100 * (ai_arm["machine_made_reading"] == "asserted").mean()))
print()
print("=" * 104)
print("EVERY HUMAN-ARM RESPONSE THE NARROW RULE FLAGS, GROUPED BY PARTICIPANT")
print("=" * 104)

spans_reading = MATCH_SPANS["machine_made_reading"]
for participant in sorted(joined["id"].iloc[mention_idx].unique().tolist()):
    rows = [i for i in mention_idx if joined["id"].iloc[i] == participant]
    n_statements = sum(1 for i in rows if statement_mask[i])
    print()
    print("participant %s: %d mention(s), %d statement(s)" % (participant, len(rows), n_statements))
    print("-" * 104)
    for i in rows:
        phrase, start, end = spans_reading[i]
        print("    %-12s %-10s rating %d   %s" % (joined["stimulus_id"].iloc[i],
                                                 joined["dimension"].iloc[i],
                                                 joined["rating"].iloc[i],
                                                 joined["machine_made_reading"].iloc[i]))
        print(wrap_for_print(mark(original_text[i], start, end), indent="        "))


MACHINES IN THE WRITING OF THE ARM TOLD A HUMAN ARTIST MADE THE WORK
  NAMES A MACHINE OR PROMPTING AT ALL (says_prompt)
    responses:                            114 of 1120 (10.2%)
    participants producing at least one:  28 of 56 (50.0%)
  MENTIONS OF A MACHINE MAKING THE WORK (says_machine_made)
    responses:                            88 of 1120 (7.9%)
    participants producing at least one:  26 of 56 (46.4%)
  STATEMENTS THAT A MACHINE MADE THE WORK (machine_made_reading = asserted)
    responses:                            80 of 1120 (7.1%)
    participants producing at least one:  25 of 56 (44.6%)

  mentions read as not asserted (every phrase denied or only supposed): 8
  mentions the broad rule does not flag: 0
  statements by question: {'creativity': 17, 'effort': 63}
  statements per participant, among those making one: median 3, most 12, fewest 1

  For comparison, in the arm told an image generator made the work, where the vignette itself
  says a machine was used: 38.

---

# 7. Prompt talk in effort justifications

Section 6 counted prompt talk in the human arm as the broadest sign that a machine came up at all.
In the arm told an image generator made the work, the same words are plain description. Counted on
their own, they show what those participants took the effort question to be asking about.

If half of the machine-arm effort justifications are about prompting and generating, then the effort
rating in that arm is largely a judgment about operating a machine, while in the human arm it is a
judgment about making a picture. The two arms would then be answering the same question about
different objects, and any comparison of the ratings would have to say so.

In [13]:
# ---------------------------------------------------------------------------
# Prompt talk by arm and by question.
# ---------------------------------------------------------------------------
prompt_by_arm = (joined.groupby(["Type", "dimension"])["says_prompt"]
                       .agg(responses="size", mentions="sum"))
prompt_by_arm["percent"] = (100 * prompt_by_arm["mentions"] / prompt_by_arm["responses"]).round(1)

print("=" * 104)
print("RESPONSES MENTIONING PROMPTING, TYPING, GENERATING, OR A MACHINE")
print("=" * 104)
print(prompt_by_arm.reindex(
    pd.MultiIndex.from_product([["Human", "AI"], ["effort", "creativity"]],
                               names=["Type", "dimension"])).to_string())
print()

for arm in ["Human", "AI"]:
    e = joined[(joined.Type == arm) & (joined.dimension == "effort")]
    c = joined[(joined.Type == arm) & (joined.dimension == "creativity")]
    print("%-6s arm: %.1f%% of effort justifications and %.1f%% of creativity justifications "
          "mention the machine." % (arm, 100 * e["says_prompt"].mean(), 100 * c["says_prompt"].mean()))
print()

# how many machine-arm participants ever mention it, for symmetry with section 6
ai_effort = joined[(joined.Type == "AI") & (joined.dimension == "effort")]
who = ai_effort.groupby("id")["says_prompt"].any()
print("In the machine-attribution arm, %d of %d participants mention the machine in at least one "
      "effort justification." % (int(who.sum()), len(who)))

RESPONSES MENTIONING PROMPTING, TYPING, GENERATING, OR A MACHINE
                  responses  mentions  percent
Type  dimension                               
Human effort            560        86     15.4
      creativity        560        28      5.0
AI    effort            550       278     50.5
      creativity        550       148     26.9

Human  arm: 15.4% of effort justifications and 5.0% of creativity justifications mention the machine.
AI     arm: 50.5% of effort justifications and 26.9% of creativity justifications mention the machine.

In the machine-attribution arm, 42 of 55 participants mention the machine in at least one effort justification.


---

# 8. Distinctive words

Sections 4 to 7 count what the rules were told to look for. This section does the opposite: it lets
the corpus say which words it uses, without a list written in advance.

The question is: **which words show up more in justifications that gave a high rating than in
justifications that gave a low one, once we account for how common each word is overall?** That last
clause is the difficulty. Raw frequency cannot answer it, since *the* is the commonest word on both
sides. Simple ratios do worse: a word appearing twice on one side and never on the other has an
infinite ratio and means nothing.

The method used here is Monroe, Colaresi and Quinn's log-odds ratio with an informative Dirichlet
prior, published in *Political Analysis* in 2008. It is written out in full in the cell below, in
twenty lines, so nothing is hidden in a library. In ordinary words:

1. For each word, compute its odds of appearing in the high-rating writing, and its odds in the
   low-rating writing. The difference between the logarithms of those two odds is the raw signal.
2. Before computing those odds, add to each word's count a small amount **proportional to how common
   that word is in the corpus as a whole**. This is the "informative prior". It stops rare words
   from dominating: a word seen twice starts from almost nothing and needs a lot of evidence to
   move, while a common word starts from a solid base and can move on smaller differences.
3. Divide the result by its own standard error (a measure of how uncertain it is), which is larger
   for rare words, to get a **z score**. A z of +2 means the word is about two standard errors more
   characteristic of the high-rating side. Treat it as a ranking device, not as a significance test:
   thousands of words are scored at once, and no correction for that is applied or intended.

The analysis runs inside the human-attribution arm only, comparing ratings of 4 or 5 against ratings
of 1 or 2. Ratings of 3 are set aside so that the two sides are clearly separated. Effort and
creativity are analysed separately, because they are different questions.

**A warning about the tooling.** scikit-learn ships a built-in English stop list (a list of words a
program ignores), and it is not a list of function words: it contains *detail*, *take*, *many*,
*much*, *show*, *made* and *part*. Using it here would delete *detail*, the most important word in
this corpus, along with several others the analysis is about. The hand-written function-word list
from section 1 is used instead, and the cell below checks again that no content word slipped into
it.

In [14]:
# ---------------------------------------------------------------------------
# Which words are distinctive of high versus low ratings?
#
# Monroe, Colaresi and Quinn (2008), "Fightin' Words: Lexical Feature Selection
# and Evaluation for Identifying the Content of Political Conflict",
# Political Analysis 16(4), 372-403. Section 3.4, the log-odds ratio with an
# informative Dirichlet prior.
#
# Notation, for one word w:
#   y_high[w]  times w appears in the high-rating writing
#   y_low[w]   times w appears in the low-rating writing
#   n_high     total words in the high-rating writing
#   n_low      total words in the low-rating writing
#   a[w]       the prior for w: alpha0 * (w's share of the whole corpus)
#   a0         alpha0, the total prior weight spread across all words
# ---------------------------------------------------------------------------
WORD_RE = re.compile(r"[a-z][a-z']+")
IGNORE = set(FUNCTION_WORDS)

for content_word in ["detail", "take", "many", "much", "show", "made", "part"]:
    assert content_word not in IGNORE, "%s must not be ignored" % content_word


def tokenize(text):
    """Lower-cased words of two letters or more, with function words removed."""
    return [w for w in WORD_RE.findall(text.lower()) if w not in IGNORE]


def keyness(texts_high, texts_low, alpha0=500.0, min_total=5):
    """Log-odds ratio with an informative Dirichlet prior (Monroe et al. 2008)."""
    from collections import Counter
    y_high = Counter(w for t in texts_high for w in tokenize(t))
    y_low = Counter(w for t in texts_low for w in tokenize(t))
    vocab = sorted(set(y_high) | set(y_low))
    yh = np.array([y_high[w] for w in vocab], dtype=float)
    yl = np.array([y_low[w] for w in vocab], dtype=float)

    total = yh + yl                              # the corpus these two sides make up
    a = alpha0 * total / total.sum()             # the prior: proportional to overall frequency
    a0 = a.sum()                                 # equals alpha0, up to rounding
    n_high, n_low = yh.sum(), yl.sum()

    # odds of the word inside each side, after adding the prior
    odds_high = (yh + a) / (n_high + a0 - yh - a)
    odds_low = (yl + a) / (n_low + a0 - yl - a)
    delta = np.log(odds_high) - np.log(odds_low)   # the log-odds ratio

    # its variance, from the same paper: rare words get a bigger one, so they move less
    variance = 1.0 / (yh + a) + 1.0 / (yl + a)
    z = delta / np.sqrt(variance)

    out = pd.DataFrame({"word": vocab, "z": z, "n_high": yh.astype(int),
                        "n_low": yl.astype(int)})
    out = out[(out["n_high"] + out["n_low"]) >= min_total]   # drop words too rare to read
    return out.sort_values("z", ascending=False).reset_index(drop=True)


def run_keyness(dimension, n_show=25):
    sub = joined[(joined["Type"] == "Human") & (joined["dimension"] == dimension)]
    high = sub[sub["rating"].isin([4, 5])]["response_text"].astype(str).tolist()
    low = sub[sub["rating"].isin([1, 2])]["response_text"].astype(str).tolist()
    table = keyness(high, low)

    print("=" * 104)
    print("DISTINCTIVE WORDS IN %s JUSTIFICATIONS, HUMAN-ATTRIBUTION ARM" % dimension.upper())
    print("high ratings (4 or 5): %d responses, %d words after ignoring function words"
          % (len(high), sum(len(tokenize(t)) for t in high)))
    print("low ratings (1 or 2):  %d responses, %d words" % (len(low), sum(len(tokenize(t)) for t in low)))
    print("ratings of 3 are set aside; %d words scored in total" % len(table))
    print("=" * 104)

    top = table.head(n_show).reset_index(drop=True)
    bottom = table.tail(n_show).iloc[::-1].reset_index(drop=True)
    side_by_side = pd.DataFrame({
        "#": range(1, n_show + 1),
        "more in HIGH ratings": top["word"],
        "z": top["z"].round(2),
        "high": top["n_high"], "low": top["n_low"],
        "  ": "|",
        "more in LOW ratings": bottom["word"],
        "z ": bottom["z"].round(2),
        "high ": bottom["n_high"], "low ": bottom["n_low"],
    })
    print(side_by_side.to_string(index=False))
    print()
    return table


keyness_effort = run_keyness("effort")
keyness_creativity = run_keyness("creativity")

DISTINCTIVE WORDS IN EFFORT JUSTIFICATIONS, HUMAN-ATTRIBUTION ARM
high ratings (4 or 5): 363 responses, 3118 words after ignoring function words
low ratings (1 or 2):  70 responses, 496 words
ratings of 3 are set aside; 152 words scored in total
 # more in HIGH ratings    z  high  low    more in LOW ratings    z   high   low 
 1                  lot 3.12   135    2  |                  ai -8.94     15    47
 2               detail 2.48    84    1  |           generated -6.37     12    26
 3                 lots 1.99    47    0  |             minimal -4.06      1    10
 4            different 1.61    31    0  |                feel -3.17      4     7
 5                great 1.61    31    0  |               using -3.17      4     7
 6             detailed 1.53    50    2  |                seen -3.00      3     6
 7                 time 1.52    39    1  |            examples -2.93      0     6
 8                 took 1.50    27    0  |            generate -2.93      0     6
 9              

### Reading the two tables

The words at each end are a check on the rules, not a replacement for them. If detail vocabulary
sits at the high end of the effort table without anyone having asked for it, the section 4 result is
not an artefact of a word list: the corpus produces the same answer when nothing is specified in
advance. If familiarity vocabulary sits at the low end of the creativity table, the same holds for
section 5.

Two cautions. A z score here ranks words; it is not a hypothesis test, and no correction for the
thousands of simultaneous comparisons is applied. And the writing came after the rating, so a word
being characteristic of high ratings says only that the two appear together.

---

# 9. Two registered quantities: the measurement half

Two quantities about the justifications are registered as descriptive checks rather than as
predictions. They run in every study where the rating exists. Neither is added to any correction
family (a set of tests whose combined error rate is controlled together), and whatever they show is
reported as it falls. This notebook computes only their measurement halves, from the blind slice of
section 2, and writes them to three files that carry no condition and no rating. The integration
step joins the condition and the rating back on and does the inference. The same division of labour
holds here as for the eight rules. Nothing in this section can be shaped by which arm a response
came from, because these two cells read only the measurement slice, which carries no arm and no
rating.

**Check 1: how much of the rating, and of the arm, can be recovered from the words alone?** Every
justification was written after its rating, by the same person, about the same trial, so a
justification restates its rating to some degree. The check measures how far. If a model that sees
only the words recovers the rating well, the words and the rating are largely one judgment expressed
twice. This possibility is why no text measure anywhere in this project is read as a mechanism
behind the rating.

The measurement half is a word-count table. Each response is broken into its content words with the
`tokenize()` function from section 8, which lower-cases the text, keeps runs of letters and
apostrophes of two characters or more, and drops the function words listed in section 1. The table
is written in long form, one row per response and word, with the number of times that word appears.
A second file lists the vocabulary with the number of responses each word appears in.

The integration step pivots the table into a response-by-word matrix and fits ridge regression of
the rating on the word counts. Ridge regression is a regression in which the coefficients are pulled
towards zero, so that many weak word predictors can be fitted together without the model memorising
the sample. The fit uses cross-validation grouped by participant: the model is fitted on some
participants and scored on others it never saw, so that one person's habits of phrasing cannot count
as prediction. The same fit is run for the arm, and each estimate is set beside what the fit
recovers when there is nothing to recover. The registered vocabulary keeps the words that appear in
at least five responses. That floor is applied inside each round of the cross-validation, counting
responses among the participants the model is fitted on, so that the participants it is scored on
have no say in which words it uses. This file therefore keeps every word that appears in at least
two responses, more than any round will use. A word found in only one response is left out, since no
other response could share it.

**Check 2: do participants told a machine made the works reuse their own words across the ten images
more than participants told a human did?** This is the text version of the fourth hypothesis of the
ratings analysis, which says that under the machine label a rating tracks the person more and the
particular image less. If that is so, the same thing may show in the words: a participant who sees
the ten images as much alike may say much the same thing about each.

The measure is Jaccard overlap: the number of content words two explanations share, divided by the
number of content words either uses. Two explanations with no word in common score zero, and two
using exactly the same words score one. For each participant and each question, the integration step
takes the mean overlap across all pairs of that participant's ten explanations (forty-five pairs).
That mean is how far the participant reuses their own words. Beside it sits a stranger baseline: the
mean overlap between each of the participant's explanations and the explanations other participants
wrote about the same image in answer to the same question, averaged over the ten images. The
baseline is what overlap looks like when two people describe the same picture, so the difference
between the two is repetition beyond what the shared images account for. Overlap falls as texts get
longer, because two long word sets share a smaller fraction of their words than two short ones do.
The comparison of the two arms therefore holds constant the mean number of distinct content words
per explanation.

The measurement half, written here, is each response's set of distinct content words, sorted and
written out as one line per response. The overlaps themselves are computed in the integration step,
because the stranger baseline depends on which other participants are in the sample. That step
resamples participants (draws repeated samples from the participants observed) to put an interval (a
range of plausible values) on the difference between the arms, and it recomputes the baseline inside
each resample. The interval then reflects the variation in the baseline as well as in each
participant's own overlap.

All three files are written from the blind slice and nothing else. The cells below assert that none
of them carries a condition, a rating or the response text column, in the same way that section 13
checks the flags file.

**A disclosure that bears on the first check in this study.** Section 12 reports that the ratings in
this corpus have been predicted from sentence embeddings (lists of numbers that encode the meaning
of a sentence), a different method with the same aim. For Study 1, how much of the rating the text
recovers was therefore already partly known when the check was registered, and the check is not a
prediction. A version of the overlap check built on sentence embeddings was also run on this corpus,
and it did not hold up under resampling. The word-overlap version registered here had not been run
when it was written down.


In [15]:
# ---------------------------------------------------------------------------
# The word-count table for the first registered check: how much of the rating,
# and of the arm, can be recovered from the words alone.
#
# Measurement only; the fitting happens in the integration step. Each response
# is broken into content words with tokenize() from section 8, which lower-cases
# the text, keeps runs of letters and apostrophes of two characters or more, and
# drops the function words from section 1. Words appearing in fewer than
# WORDCOUNT_MIN_RESPONSES (two) responses are set aside, since a word no other
# response uses cannot be learned from. The registered floor of five responses
# is applied by the integration step inside each round of its cross-validation,
# counting only the responses of the participants the model is fitted on, so
# this file keeps every word any round could need. The table is long form, one
# row per response and word with a positive count, so that the integration step
# can pivot it into a response-by-word matrix; a response with no counted word
# has no row, and the integration step fills it with zeros from the full list
# of responses.
#
# Everything here is computed from the blind slice. The two files carry the
# response identifiers, the words and the counts, and nothing else: no
# condition, no rating, no text.
# ---------------------------------------------------------------------------
from collections import Counter

WORDCOUNT_MIN_RESPONSES = 2    # a word must appear in at least this many responses

# the content words of every response, in the order of the blind slice
response_tokens = [tokenize(t) for t in blind["response_text"].fillna("").astype(str)]

# how many responses each word appears in (not how many times it appears)
responses_per_word = Counter(w for tokens in response_tokens for w in set(tokens))
vocabulary = pd.DataFrame(
    sorted(((w, n) for w, n in responses_per_word.items() if n >= WORDCOUNT_MIN_RESPONSES),
           key=lambda pair: (-pair[1], pair[0])),
    columns=["word", "n_responses"],
)
VOCABULARY = set(vocabulary["word"])

rows = []
for (rid, sid, dim), tokens in zip(blind[["id", "stimulus_id", "dimension"]].itertuples(index=False),
                                   response_tokens):
    counts = Counter(w for w in tokens if w in VOCABULARY)
    for word, count in sorted(counts.items()):
        rows.append((rid, sid, dim, word, count))
wordcounts = pd.DataFrame(rows, columns=["id", "stimulus_id", "dimension", "word", "count"])

# the checks, before anything is written
for frame, label in [(wordcounts, "study1_text_wordcounts.csv"),
                     (vocabulary, "study1_text_vocabulary.csv")]:
    for forbidden in ["Type", "rating", "response_text", "condition", "arm", "Age", "Gender"]:
        assert forbidden not in frame.columns, \
            "%s: %s must not appear in this file." % (label, forbidden)
assert list(wordcounts.columns) == ["id", "stimulus_id", "dimension", "word", "count"], \
    "study1_text_wordcounts.csv: the columns are wrong: %s" % list(wordcounts.columns)
assert list(vocabulary.columns) == ["word", "n_responses"], \
    "study1_text_vocabulary.csv: the columns are wrong: %s" % list(vocabulary.columns)
assert (wordcounts["count"] > 0).all(), "Every row of the word-count table must have a positive count."
assert set(wordcounts["word"]) <= VOCABULARY, "A word outside the vocabulary reached the word-count table."
assert (vocabulary["n_responses"] >= WORDCOUNT_MIN_RESPONSES).all(), \
    "A word appearing in too few responses reached the vocabulary."
assert wordcounts.duplicated(["id", "stimulus_id", "dimension", "word"]).sum() == 0, \
    "The word-count table has more than one row for the same response and word."
# the number of responses each word appears in, recounted from the table itself,
# has to agree with the vocabulary file
recount = wordcounts.groupby("word").size().reindex(vocabulary["word"])
assert (recount.values == vocabulary["n_responses"].values).all(), \
    "The word-count table and the vocabulary disagree about how many responses a word appears in."

WORDCOUNTS_PATH = os.path.join(NLP_DIR, "study1_text_wordcounts.csv")
VOCABULARY_PATH = os.path.join(NLP_DIR, "study1_text_vocabulary.csv")
atomic_write_csv(WORDCOUNTS_PATH, wordcounts)
atomic_write_csv(VOCABULARY_PATH, vocabulary)

n_with_words = len(wordcounts[["id", "stimulus_id", "dimension"]].drop_duplicates())
print("Written:", relative_path(WORDCOUNTS_PATH))
print("  rows (response and word pairs with a positive count):", len(wordcounts))
print("  responses with at least one vocabulary word: %d of %d" % (n_with_words, len(blind)))
print("Written:", relative_path(VOCABULARY_PATH))
print("  vocabulary: %d words appearing in at least %d responses, out of %d distinct content words"
      % (len(vocabulary), WORDCOUNT_MIN_RESPONSES, len(responses_per_word)))
print("  condition, rating and text are absent from both files. Checked.")
print()
print("The ten commonest vocabulary words, by the number of responses they appear in:")
print(vocabulary.head(10).to_string(index=False))

Written: nlp/study1_text_wordcounts.csv
  rows (response and word pairs with a positive count): 17231
  responses with at least one vocabulary word: 2214 of 2220
Written: nlp/study1_text_vocabulary.csv
  vocabulary: 1300 words appearing in at least 2 responses, out of 2421 distinct content words
  condition, rating and text are absent from both files. Checked.

The ten commonest vocabulary words, by the number of responses they appear in:
      word  n_responses
    effort          472
        ai          402
     image          384
       lot          318
  creative          303
     think          280
creativity          253
     looks          239
   details          182
     piece          167


In [16]:
# ---------------------------------------------------------------------------
# The content-word lists for the second registered check: do participants
# reuse their own words across the ten images?
#
# One row per response: its distinct content words from tokenize(), lower
# case, sorted and joined with single spaces, or an empty string when the
# response has none. The integration step computes every overlap from this
# file: the mean Jaccard overlap (the words two texts share, divided by the
# words either uses) across a participant's own explanations, the stranger
# baseline over what other participants wrote about the same images, and the
# mean number of distinct content words per explanation, the length it holds
# constant. It computes them there rather than here because the stranger
# baseline depends on which other participants are in the sample, and the
# integration step recomputes it inside each resample of participants.
#
# No vocabulary cut is applied: the overlap is between whole word sets, and a
# rare word that two of a participant's explanations share is exactly the
# repetition the measure is meant to see. Everything here is computed from the
# blind slice.
# ---------------------------------------------------------------------------
TOKENS_COLUMNS = ["id", "stimulus_id", "dimension", "content_words"]
tokens_frame = pd.DataFrame({
    "id": blind["id"].values,
    "stimulus_id": blind["stimulus_id"].values,
    "dimension": blind["dimension"].values,
    "content_words": [" ".join(sorted(set(tokens))) for tokens in response_tokens],
}, columns=TOKENS_COLUMNS)

# the checks, before anything is written
for forbidden in ["Type", "rating", "response_text", "condition", "arm", "Age", "Gender"]:
    assert forbidden not in tokens_frame.columns, \
        "study1_text_tokens.csv: %s must not appear in this file." % forbidden
assert list(tokens_frame.columns) == TOKENS_COLUMNS, \
    "study1_text_tokens.csv: the columns are wrong: %s" % list(tokens_frame.columns)
assert len(tokens_frame) == len(blind), \
    "Expected one row per response, found %d rows." % len(tokens_frame)
assert tokens_frame.duplicated(["id", "stimulus_id", "dimension"]).sum() == 0, \
    "The content-word table has more than one row for the same response."
# every list holds lower-case content words, each once, in sorted order
for words in tokens_frame["content_words"]:
    listed = words.split()
    assert listed == sorted(set(listed)), "A word list is not sorted, or repeats a word."
    assert not set(listed) & IGNORE, "A function word reached the content-word lists."
    assert words == words.lower(), "A word list is not in lower case."

TOKENS_PATH = os.path.join(NLP_DIR, "study1_text_tokens.csv")
atomic_write_csv(TOKENS_PATH, tokens_frame)

words_per_response = tokens_frame["content_words"].str.split().str.len()
print("Written:", relative_path(TOKENS_PATH))
print("  rows (one per response):", len(tokens_frame))
print("  responses with no content word (an empty list): %d" % int((words_per_response == 0).sum()))
print("  distinct content words per response: median %d, mean %.1f, most %d"
      % (words_per_response.median(), words_per_response.mean(), words_per_response.max()))
print("  condition and rating are absent. Checked.")
print()
print("The within-person and stranger overlaps of the second check are computed from this file in")
print("the integration step, which recomputes the stranger baseline inside each resample.")


Written: nlp/study1_text_tokens.csv
  rows (one per response): 2220
  responses with no content word (an empty list): 2
  distinct content words per response: median 7, mean 8.3, most 43
  condition and rating are absent. Checked.

The within-person and stranger overlaps of the second check are computed from this file in
the integration step, which recomputes the stranger baseline inside each resample.


---

# 10. Auditing the rules by judging words rather than responses

Every rule above has a limitation that section 3 cannot expose. A search pattern finds exactly what
was written down and nothing else. Section 3 shows how often a rule fires when it should not. That
is **precision**: the share of what a rule catches that belongs to it, which is visible in the
matched responses. No rule can show how often it stays silent when it should have fired. That is
**recall**: the share of what belongs to a rule that the rule catches. Recall cannot be seen from
inside the method, because a rule cannot report the words nobody put on its list.

This section approaches that gap from the side. It does not ask a person to read hundreds of
responses and decide, one at a time, whether each was missed. It asks the corpus which words behave
like each rule's own words without being on its list, and reports them as a short list of *words*.
The list is a report and nothing more. It lets a reader judge the rules' coverage. No word is added
to a rule after the data have been seen, and nothing in the notebook reads the list back.

**How the list is built.** For one rule at a time:

1. Split the corpus in two: the responses that rule flags, and every other response.
2. Score every word by how over-represented it is in the flagged half, using the same calculation
   section 8 uses for distinctive words (Monroe, Colaresi and Quinn's log-odds ratio with an
   informative Dirichlet prior, written out in full in the section 8 cell and not implemented a
   second time here). High-scoring words are the vocabulary that turns up in the same places as the
   rule's own vocabulary.
3. Remove every word the rule already names. A word counts as already named if the rule's own
   pattern matches it standing alone, or if it appears anywhere inside a phrase that pattern matched
   in this corpus. What is left are the words that keep company with the rule's words without being
   on its list.
4. Remove the task's own vocabulary: the vignette's name, the words of the two rating questions
   ("How much effort was involved in the production of this work?", and the same for creativity),
   the words of their scale anchors (the labels on the points of the rating scale: *no*, *minimal*,
   *moderate*, *significant*, *a great deal*, *a lot*) and the words of the stimulus description. In
   full: *Alex*, *art*, *artwork*, *image*, *picture*, *piece*, *work*, *effort*, *effortful*,
   *creative*, *creativity*, *rating*, *rate*, *involved*, *production*, *much*, *minimal*,
   *moderate*, *significant*, *great*, *deal*, *lot*, and their plurals. These words are in the
   responses because the instrument put them there, and none of them can name a rule's topic, so
   they would only crowd out words that might. A word every participant read is over-represented
   wherever a writer restates the question rather than answers it, and the anchor-echo rule records
   the responses that contain that wording. Section 11 removes the same words before it scores
   affect and fits topics, for the same reason.
5. Keep only words that appear in at least one response the rule does **not** flag, since a word
   found solely inside already-flagged responses says nothing about coverage.
6. Take the highest-scoring thirty and print each with its counts and one example sentence. The
   example is drawn from a response the rule does not flag, which is a response the rule would have
   caught had the word been on its list.

The result is written to `tables/text_candidate_words.txt`: each word with its counts and its
example sentence, laid out to be read.

**What the list shows.** It shows phrasings the rules may have missed. If a rule's list holds no
word that plainly names its topic, the corpus agrees with the rule's vocabulary. The supplement can
make a coverage argument from this: for such a rule the corpus, asked in its own words what else
belonged, had little to add.

**What the list does not show.** It is not a recall figure, and it must not be reported as one.
Recall is the proportion of responses that belong to a rule which the rule catches. Computing it
would require somebody to look at individual responses the rule stayed silent on and judge whether
each one was a miss. No amount of word-level work substitutes for that. This procedure can only
surface vocabulary, and a response can be a miss without containing any unusual word at all: it can
say the thing in a plain sentence made entirely of common words. So there is no percentage here, no
confidence interval, and no denominator. A word on the list says the rule may have a hole of a
particular shape. It does not say how big the hole is.

**A further caution about step 2.** Words are scored against the whole rest of the corpus, so words
that travel with the rule's subject matter will rank high without belonging in the rule. A rule
about time will surface *make*, because people who write about how long something took also write
about making. That is not an error in the ranking; it is what a list of neighbours looks like.
Telling the two kinds apart is the judgment the list leaves to its reader, and the rules stand as
written either way.


In [17]:
# ---------------------------------------------------------------------------
# Candidate words for each rule.
#
# This cell introduces no new statistics. It uses two things defined in section 8:
#   keyness()  the log-odds ratio with an informative Dirichlet prior. There it
#              compared high-rated writing with low-rated writing; here the two
#              sides are "responses this rule flags" and "every other response".
#   tokenize() lower-cased words of two letters or more with function words
#              removed, so no candidate word is ever "the" or "was".
#
# What to look at in the output: the words near the top of each rule's list. A
# word there that is plainly the rule's business and is missing from it shows a
# hole in the rule's vocabulary. A subject-matter neighbour that does not belong,
# such as "make" sitting under the time rule because people who write about how
# long something took also write about making, shows nothing of the kind. Either
# way the rule stands as written: the list is a report on the rules' coverage,
# and no word is added to any rule after the data have been seen.
# ---------------------------------------------------------------------------
CANDIDATES_PER_RULE = 30      # how many words to put in front of the reader
MIN_RESPONSES = 5             # ignore words appearing in fewer responses than this

# The task's own vocabulary. The vignette named the artist Alex; the two
# questions read "How much effort was involved in the production of this work?"
# and the same for creativity; the scale's anchors ran from "no effort involved"
# through "minimal", "moderate" and "significant" to "a great deal of effort
# involved", with "a lot of effort involved" as an alternative wording of the
# fourth point; and the stimulus was described as a piece of art, an artwork,
# an image, a picture, a work. Every content word of that wording is listed
# here (the question's "much", "involved" and "production" as well as the
# anchors' "minimal", "moderate", "significant", "great", "deal" and "lot"),
# because a word every participant read is over-represented wherever a writer
# restates the question rather than answers it, and the anchor-echo rule
# records the responses that contain that wording. These words are in the
# responses because the instrument put them there, not because a writer
# reached for them, and none of them can name a rule's topic: a rule that
# added "effort" would be measuring the question rather than the answer.
# They are left out of every candidate list, together with their plurals,
# which is_task_vocabulary() below handles.
# There is no frequency ceiling beyond this: a common word that is not the
# task's own stays on the list, and it is for the reader to judge it.
# Section 11 removes the same words, through is_task_vocabulary(), before it
# scores affect and concreteness and fits topics, for the same reason.
TASK_VOCABULARY = ["alex", "art", "artwork", "image", "picture", "piece", "work",
                   "effort", "effortful", "creative", "creativity", "rating", "rate",
                   "involved", "production", "much",
                   "minimal", "moderate", "significant", "great", "deal", "lot"]
TASK_VOCABULARY_REASON = ("These words are the task's own (the vignette's name, the words of "
                          "the two rating questions and their scale anchors, and the words "
                          "describing the artworks) and cannot name any rule's topic, so they "
                          "are not listed.")


def is_task_vocabulary(word):
    """True for a task word, or for the plural of one."""
    return word in TASK_VOCABULARY or (word.endswith("s") and word[:-1] in TASK_VOCABULARY)


# An index from word to the responses containing it, built once. Everything below
# looks words up here instead of searching all 2,220 responses again each time,
# which is the difference between this cell taking a second and taking a minute.
# The word pattern is the same one tokenize() uses, so the index and the scoring
# agree about what counts as a word.
RESPONSE_WORDS = [set(WORD_RE.findall(t)) for t in lower_text]
WORD_TO_RESPONSES = {}
for _i, _words in enumerate(RESPONSE_WORDS):
    for _w in _words:
        WORD_TO_RESPONSES.setdefault(_w, []).append(_i)

print("Index built: %d distinct words across %d responses."
      % (len(WORD_TO_RESPONSES), len(RESPONSE_WORDS)))
print()


def rule_is_flagged(rule_name):
    """One true/false value per response: did this rule fire?

    prior_exposure has three values rather than two, so "fired" means it found a
    familiarity phrase in either direction, asserted or negated. "Neither" is
    the rule finding nothing, which is what counts as not fired.
    """
    if rule_name == "prior_exposure":
        return (measures["prior_exposure"] != "neither").values
    return measures[rule_name].values


def words_the_rule_already_names(rule):
    """Every word this rule can be said to know about already.

    Two sources, because neither alone is enough. A pattern such as
    detail(s|ed|ing)? matches the bare word "detailed", so testing a candidate
    against the pattern catches most cases. But a pattern can also name a
    multi-word phrase ("a lot going on") whose individual words it would not
    match on their own, so every word inside every phrase the pattern matched
    anywhere in this corpus is added too. Exclusions are ignored here on purpose:
    a word the rule names and then throws away in some contexts is still a word
    the rule has considered, and is not a candidate addition.
    """
    include_re = re.compile(rule["include"])
    known = set()
    for text in lower_text:
        for match in include_re.finditer(text):
            known.update(WORD_RE.findall(match.group(0)))
    return known, include_re


def example_response(word, flagged):
    """One response containing this word, chosen the same way on every run.

    Preference goes to a response the rule did NOT flag, because that is exactly
    the response adding the word would newly capture. Among those, the shortest
    of at least six words is used: short enough to read at a glance, long enough
    to be a sentence rather than a fragment. If the word appears only inside
    already-flagged responses, one of those is used and the caller labels it.

    Returns (row number, whether it came from an unflagged response).
    """
    pool = WORD_TO_RESPONSES.get(word, [])
    unflagged = [i for i in pool if not flagged[i]]
    from_unflagged = bool(unflagged)
    pool = unflagged if unflagged else pool
    if not pool:
        return None, from_unflagged
    long_enough = [i for i in pool if len(lower_text[i].split()) >= 6]
    choices = long_enough if long_enough else pool
    return min(choices, key=lambda i: (len(lower_text[i]), i)), from_unflagged


def candidate_words(rule):
    """Words that behave like this rule's words without being on its list."""
    name = rule["name"]
    flagged = rule_is_flagged(name)
    inside = [original_text[i] for i in range(len(original_text)) if flagged[i]]
    outside = [original_text[i] for i in range(len(original_text)) if not flagged[i]]

    # the same log-odds calculation as section 8, with flagged and unflagged
    # standing where high-rated and low-rated writing stood there
    scored = keyness(inside, outside)
    known, include_re = words_the_rule_already_names(rule)

    rows = []
    for word, z in zip(scored["word"], scored["z"]):
        if word in known or include_re.search(word) is not None:
            continue                                  # the rule already names it
        if is_task_vocabulary(word):
            continue                                  # the task's own word, not a topic
        hits = WORD_TO_RESPONSES.get(word, [])
        if len(hits) < MIN_RESPONSES:
            continue                                  # too rare to judge from
        n_flagged = sum(1 for i in hits if flagged[i])
        n_unflagged = len(hits) - n_flagged
        if n_unflagged == 0:
            continue                                  # adding it would catch nothing new
        rows.append(dict(word=word, z=float(z), n_responses=len(hits),
                         n_flagged=n_flagged, n_unflagged=n_unflagged))
        if len(rows) == CANDIDATES_PER_RULE:
            break
    return pd.DataFrame(rows), flagged


def candidate_report():
    """Build the whole report, rule by rule, ready to print or to save."""
    out = []
    out.append("=" * 100)
    out.append("STUDY 1 TEXT MEASURES: CANDIDATE WORDS FOR EACH RULE")
    out.append("=" * 100)
    out.append("")
    out.append("WHAT THIS FILE SHOWS")
    out.append("")
    out.append("Each rule below is followed by up to %d words. They are words over-represented in"
               % CANDIDATES_PER_RULE)
    out.append("the responses that rule flags, compared with the rest of the corpus, and not named")
    out.append("by the rule itself: the words that behave like the rule's own words without being")
    out.append("on its list.")
    out.append("")
    out.append("The list is reported so that a reader can judge the rules' coverage. A word here")
    out.append("that plainly names a rule's topic shows a hole in that rule's vocabulary; a word that")
    out.append("merely sits beside the rule's subject shows nothing of the kind, and most words are")
    out.append("expected to be of that second kind. No word is ever added to a rule after the data")
    out.append("have been seen: the rules stand as written in text_rules.txt, whatever this list")
    out.append("holds, and nothing in the notebook reads this file back.")
    out.append("")
    out.append("The example under each word is a real response containing it, taken from the")
    out.append("responses the rule does NOT flag, so it shows the response the rule would")
    out.append("have caught had the word been on its list. Where a word appears only inside")
    out.append("already-flagged responses, the example is labelled as such.")
    out.append("")
    out.append("WHAT THIS FILE IS NOT. It does not measure recall, the share of its subject a rule")
    out.append("catches. Measuring that would need a person to read responses the rule stayed silent")
    out.append("on and judge them one at a time, because a response can be a miss without containing")
    out.append("any unusual word: it can say the thing plainly, in common words. This file shows")
    out.append("phrasings a rule may have missed. It produces no percentage, no confidence interval")
    out.append("and no denominator, and none should be reported from it.")
    out.append("")
    out.append("Words are ranked by the log-odds score described in section 8 of the notebook. A")
    out.append("higher score means more concentrated in the flagged responses. Words appearing in")
    out.append("fewer than %d responses are not listed." % MIN_RESPONSES)
    out.append("")
    out.append(textwrap.fill("Also not listed, whatever their score: " + ", ".join(TASK_VOCABULARY)
                             + ", and their plurals. " + TASK_VOCABULARY_REASON, width=88))
    out.append("")

    tables = {}
    for rule in RULES:
        name = rule["name"]
        table, flagged = candidate_words(rule)
        tables[name] = table
        out.append("=" * 100)
        out.append("RULE: %s. In plain words: %s" % (name, rule["plain"]))
        out.append("=" * 100)
        out.append("responses this rule flags: %d of %d;   candidate words listed: %d"
                   % (int(flagged.sum()), len(flagged), len(table)))
        out.append("")
        if table.empty:
            out.append("  No word met the criteria for this rule.")
            out.append("")
            continue
        for rank, row in enumerate(table.itertuples(index=False), start=1):
            out.append("  %2d. %-18s score %6.2f   appears in %3d responses "
                       "(%3d already flagged, %3d not)"
                       % (rank, row.word, row.z, row.n_responses,
                          row.n_flagged, row.n_unflagged))
            idx, from_unflagged = example_response(row.word, flagged)
            out.append("           " + (
                "example, from a response this rule does not flag:" if from_unflagged
                else "example, from an already-flagged response (the word appears nowhere else):"))
            if idx is not None:
                out.append(textwrap.fill(original_text[idx], width=92,
                                         initial_indent="             ",
                                         subsequent_indent="             "))
            out.append("")
        out.append("")
    return "\n".join(out), tables


CANDIDATE_TEXT, CANDIDATE_TABLES = candidate_report()
print(CANDIDATE_TEXT)
print("Candidate words listed: %d across %d rules. The list is a report; no rule changes."
      % (sum(len(t) for t in CANDIDATE_TABLES.values()), len(RULES)))

Index built: 2619 distinct words across 2220 responses.



STUDY 1 TEXT MEASURES: CANDIDATE WORDS FOR EACH RULE

WHAT THIS FILE SHOWS

Each rule below is followed by up to 30 words. They are words over-represented in
the responses that rule flags, compared with the rest of the corpus, and not named
by the rule itself: the words that behave like the rule's own words without being
on its list.

The list is reported so that a reader can judge the rules' coverage. A word here
that plainly names a rule's topic shows a hole in that rule's vocabulary; a word that
merely sits beside the rule's subject shows nothing of the kind, and most words are
expected to be of that second kind. No word is ever added to a rule after the data
have been seen: the rules stand as written in text_rules.txt, whatever this list
holds, and nothing in the notebook reads this file back.

The example under each word is a real response containing it, taken from the
responses the rule does NOT flag, so it shows the response the rule would
have caught had the word been on its li


Candidate words listed: 240 across 8 rules. The list is a report; no rule changes.


---

# 11. A general description of the text

Sections 4 to 10 answer the notebook's three questions with rules written for them. This section
describes the text in ways that depend on no rule written for this study, so that the same
description can be computed, unchanged, in every study of the project. It has three parts: affect
and concreteness scores from published word norms, topics found by a topic model, and a check on
those topics by a second method. Every measure is computed from the blind slice of section 2,
without the condition or the rating. The integration step joins those back on and reports the
measures as exploratory description, beside the registered analyses and never in place of them.

**Affect and concreteness.** Warriner, Kuperman and Brysbaert (2013) asked raters to score 13,915
English words, on a scale of 1 to 9, for valence (how pleasant a word is), arousal (how calm or
excited it makes a reader feel) and dominance (how much in control it makes a reader feel).
Brysbaert, Warriner and Kuperman (2014) asked raters to score about 40,000 English words for
concreteness, from 1 for a word known only through other words, such as *justice*, to 5 for a word
known through the senses, such as *dog*. A response's score on each is the mean over its content
words that appear in the norms. The task vocabulary listed in section 10 (the vignette's name and
the words of the rating questions and of their scale, such as *creative*, *effort*, *significant*
and *great*) is removed first. Every participant was handed those words, and *great* and
*significant* would otherwise raise valence wherever a writer repeated the scale. A word is looked
up as written. If it is not in the norms, one final *-ing*, *-ed*, *-es* or *-s* is stripped, the
endings tried in that order, and the first shortened form in the norms is used. Nothing further is
tried. A response with no word in a norm gets no score, an empty value rather than a zero. The
scores are word by word: "not beautiful" scores as *beautiful* does, because a norm describes a
word, not a sentence.

**Topics.** A topic model finds groups of words that tend to occur in the same responses. The one
used here is non-negative matrix factorisation (NMF), fitted separately to the creativity and to the
effort explanations, because they answer different questions. Each explanation is first turned into
a row of tf-idf weights, one per content word. Tf-idf (term frequency by inverse document frequency)
is a word's count in the explanation multiplied by a factor that grows the fewer explanations the
word appears in, so that words nearly everyone uses count for less. Function words and the task
vocabulary are removed, and only words that appear in at least five explanations of that question
are kept. NMF then writes the explanation-by-word table as the product of two smaller tables of
non-negative numbers. One gives every topic a weight on every word, and the other gives every
explanation a weight on every topic. The words with the largest weights say what a topic is about.
An explanation's topic weights, scaled to sum to one, say how much of its wording each topic
accounts for. An explanation with no kept word gets no weights.

**How many topics.** The number of topics, k, is chosen for each question by a rule written in the
code. For each k from 3 to 10, the model is refitted ten times, each time on a different random 80
per cent of the participants. The topics of two refits are paired one to one so that the pairs are
as alike as possible, using the Hungarian algorithm (a standard method for finding the best
one-to-one pairing). The likeness of a pair is the cosine similarity of their word weights, which is
1 when two topics weight the same words in the same proportions and 0 when they share no word.
Stability at k is that likeness averaged over the paired topics and over all 45 pairs of refits. The
k with the highest stability is chosen, and when two values tie to three decimal places the smaller
k is chosen. The final model is fitted once, at the chosen k, to every explanation of that question.

Leaving participants out is what lets the refits differ. NMF here starts from values computed from
the data rather than drawn at random. The setting is called nndsvda, and it takes its starting point
from a singular value decomposition, a standard way of breaking a table into its main directions.
Refits on identical data therefore come back nearly identical at every k, and could not choose
between them. A stable k is one at which the topics come back much the same when a fifth of the
participants is left out. It is not a claim that the explanations fall into k separate groups.
Section 12 records that clustering the explanations into separate groups found no natural grouping.
A topic model does not need one, because it describes each explanation as a mixture of topics.

**A check by a second method.** The topics are found from word counts alone. The check asks whether
a method that takes in the whole sentence groups the explanations the same way. Each explanation is
turned into a list of 768 numbers by a sentence-embedding model (all-mpnet-base-v2, a pretrained
network that places sentences of similar meaning near one another; it gives no explanation a label
and writes no text). The explanations are then split into k groups by k-means clustering on those
numbers, which puts each explanation in the group whose centre it is nearest. The split is compared
with each explanation's dominant topic, the topic with its largest weight. The comparison is the
adjusted Rand index, the agreement between two ways of grouping the same items beyond what chance
would give: 0 is the agreement chance produces and 1 is identical groupings. The model is loaded
only from a copy already on the machine. If no copy is available, the check is recorded as not run
and nothing else changes.

The section writes `nlp/study1_text_affect.csv` (one row per response), `nlp/study1_text_topics.csv`
(one row per response and topic), and three tables: the ten top words of every topic
(`tables/text_topics_top_words.csv`), the stability of every k (`tables/text_topics_stability.csv`),
and the agreement check (`tables/text_topics_embedding_agreement.csv`).


In [18]:
# ---------------------------------------------------------------------------
# Affect and concreteness, from two sets of published word norms.
#
# Warriner, Kuperman and Brysbaert (2013) give each of 13,915 English words a
# mean rating, on a scale of 1 to 9, of valence (how pleasant the word is),
# arousal (how calm or excited it makes a reader feel) and dominance (how much
# in control it makes a reader feel), in the columns V.Mean.Sum, A.Mean.Sum
# and D.Mean.Sum. Brysbaert, Warriner and Kuperman (2014) give about 40,000
# English words a mean rating of concreteness, from 1 (abstract) to 5
# (concrete), in the column Conc.M. Both files sit in nlp/norms/, whose README
# records where each came from and its checksum.
#
# Both are read with keep_default_na=False, because each contains the word
# "null", which pandas would otherwise read as a missing value. Ten words in
# the first file appear twice once lower-cased ("aids" and "AIDS", "tv" and
# "TV"); the entry already spelled in lower case is the one kept.
#
# A response's content words are its tokens from tokenize() (section 8) with
# the task vocabulary of section 10 removed, every occurrence counted. A word
# is looked up as written; if it is not in the norms, one final "ing", "ed",
# "es" or "s" is stripped, trying the endings in that order, and the first
# shortened form found in the norms is used, provided it keeps at least two
# letters. Nothing further is tried. A score is the mean over the matched
# occurrences, and a response with no matched word gets no score (an empty
# value), not a zero. The scores are word by word: "not beautiful" scores as
# "beautiful" does, because a norm describes a word, not a sentence.
# ---------------------------------------------------------------------------
NORMS_DIR = os.path.join(NLP_DIR, "norms")
VAD_PATH = os.path.join(NORMS_DIR, "Warriner_et_al_2013_VAD.csv")
CONCRETENESS_PATH = os.path.join(NORMS_DIR, "Concreteness_ratings_Brysbaert_et_al_BRM.txt")
for path in [VAD_PATH, CONCRETENESS_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            "A word-norm file is missing: " + relative_path(path) + "\n"
            "nlp/norms/README.md says where each file comes from and what it must be called.")


def read_norms(path, sep, columns):
    """One norm file as a dictionary from lower-case word to its scores."""
    table = pd.read_csv(path, sep=sep, usecols=["Word"] + columns, keep_default_na=False)
    table["spelled"] = table["Word"].astype(str).str.strip()
    table["lower"] = table["spelled"].str.lower()
    # where two entries share a lower-case form, keep the one spelled in lower case
    table["spelled_lower"] = table["spelled"] == table["lower"]
    table = (table.sort_values("spelled_lower", ascending=False, kind="stable")
                  .drop_duplicates("lower", keep="first"))
    scores = table[columns].apply(pd.to_numeric, errors="raise")
    return dict(zip(table["lower"], scores.itertuples(index=False, name=None)))


VAD_COLUMNS = ["V.Mean.Sum", "A.Mean.Sum", "D.Mean.Sum"]
VAD_NORMS = read_norms(VAD_PATH, ",", VAD_COLUMNS)
CONCRETENESS_NORMS = read_norms(CONCRETENESS_PATH, "\t", ["Conc.M"])
NORM_ENDINGS = ["ing", "ed", "es", "s"]


def norm_entry(word, norms):
    """The norm entry for `word`, or for `word` with one ending stripped, or None."""
    if word in norms:
        return norms[word]
    for ending in NORM_ENDINGS:
        if word.endswith(ending) and len(word) - len(ending) >= 2:
            shorter = word[:-len(ending)]
            if shorter in norms:
                return norms[shorter]
    return None


def mean_or_empty(values):
    return round(float(np.mean(values)), 4) if values else np.nan


AFFECT_COLUMNS = ["id", "stimulus_id", "dimension", "n_content_words", "n_vad_matched",
                  "valence", "arousal", "dominance", "n_conc_matched", "concreteness"]
affect_rows = []
unmatched_vad = Counter()
for (rid, sid, dim), tokens in zip(blind[["id", "stimulus_id", "dimension"]].itertuples(index=False),
                                   response_tokens):
    words = [w for w in tokens if not is_task_vocabulary(w)]
    vad = [norm_entry(w, VAD_NORMS) for w in words]
    unmatched_vad.update(w for w, entry in zip(words, vad) if entry is None)
    vad = [entry for entry in vad if entry is not None]
    concreteness = [entry[0] for entry in (norm_entry(w, CONCRETENESS_NORMS) for w in words)
                    if entry is not None]
    affect_rows.append(dict(
        id=rid, stimulus_id=sid, dimension=dim, n_content_words=len(words),
        n_vad_matched=len(vad),
        valence=mean_or_empty([v[0] for v in vad]),
        arousal=mean_or_empty([v[1] for v in vad]),
        dominance=mean_or_empty([v[2] for v in vad]),
        n_conc_matched=len(concreteness),
        concreteness=mean_or_empty(concreteness),
    ))
affect = pd.DataFrame(affect_rows, columns=AFFECT_COLUMNS)

# the checks, before anything is written
for forbidden in ["Type", "rating", "response_text", "condition", "arm", "Age", "Gender"]:
    assert forbidden not in affect.columns, \
        "study1_text_affect.csv: %s must not appear in this file." % forbidden
assert list(affect.columns) == AFFECT_COLUMNS, \
    "study1_text_affect.csv: the columns are wrong: %s" % list(affect.columns)
assert len(affect) == len(blind), "Expected one row per response, found %d rows." % len(affect)
assert affect.duplicated(["id", "stimulus_id", "dimension"]).sum() == 0, \
    "The affect table has more than one row for the same response."
assert (affect["n_vad_matched"] <= affect["n_content_words"]).all()
assert (affect["n_conc_matched"] <= affect["n_content_words"]).all()
assert (affect["valence"].isna() == (affect["n_vad_matched"] == 0)).all(), \
    "A valence score is empty where words matched, or present where none did."
assert (affect["concreteness"].isna() == (affect["n_conc_matched"] == 0)).all(), \
    "A concreteness score is empty where words matched, or present where none did."
for column, low, high in [("valence", 1, 9), ("arousal", 1, 9), ("dominance", 1, 9),
                          ("concreteness", 1, 5)]:
    assert affect[column].dropna().between(low, high).all(), "%s fell outside %d to %d." % (column, low, high)

AFFECT_PATH = os.path.join(NLP_DIR, "study1_text_affect.csv")
atomic_write_csv(AFFECT_PATH, affect)

n_words_total = int(affect["n_content_words"].sum())
print("Norms loaded: %d words with valence, arousal and dominance; %d words with concreteness."
      % (len(VAD_NORMS), len(CONCRETENESS_NORMS)))
print("Written:", relative_path(AFFECT_PATH))
print("  rows (one per response):", len(affect))
print("  content words, task vocabulary removed: %d, of which %d (%.1f%%) are in the valence norms"
      % (n_words_total, int(affect["n_vad_matched"].sum()),
         100 * affect["n_vad_matched"].sum() / max(n_words_total, 1)))
print("  and %d (%.1f%%) in the concreteness norms"
      % (int(affect["n_conc_matched"].sum()), 100 * affect["n_conc_matched"].sum() / max(n_words_total, 1)))
print("  responses with a valence score: %d of %d (%.1f%%); with a concreteness score: %d (%.1f%%)"
      % (int(affect["valence"].notna().sum()), len(affect), 100 * affect["valence"].notna().mean(),
         int(affect["concreteness"].notna().sum()), 100 * affect["concreteness"].notna().mean()))
print("  condition and rating are absent. Checked.")
print()
print("MEAN SCORE BY QUESTION, BOTH ARMS TOGETHER (the notebook cannot tell them apart)")
print(affect.groupby("dimension")[["valence", "arousal", "dominance", "concreteness"]]
            .agg(["mean", "std"]).round(2).to_string())
print()
print("The commonest content words found in neither form in the valence norms:")
print("  " + ", ".join("%s (%d)" % pair for pair in unmatched_vad.most_common(15)))


Norms loaded: 13905 words with valence, arousal and dominance; 39954 words with concreteness.
Written: nlp/study1_text_affect.csv
  rows (one per response): 2220
  content words, task vocabulary removed: 15543, of which 11964 (77.0%) are in the valence norms
  and 14185 (91.3%) in the concreteness norms
  responses with a valence score: 2144 of 2220 (96.6%); with a concreteness score: 2180 (98.2%)
  condition and rating are absent. Checked.

MEAN SCORE BY QUESTION, BOTH ARMS TOGETHER (the notebook cannot tell them apart)


           valence       arousal       dominance       concreteness      
              mean   std    mean   std      mean   std         mean   std
dimension                                                                
creativity    6.14  0.55    3.97  0.46      5.85  0.42         2.88  0.54
effort        5.91  0.49    3.89  0.40      5.87  0.45         2.89  0.50

The commonest content words found in neither form in the valence norms:
  ai (438), seen (159), generated (102), many (94), took (93), created (74), used (71), made (69), probably (57), required (54), taken (54), creating (50), futuristic (43), behind (41), shading (38)


In [19]:
# ---------------------------------------------------------------------------
# Topics, by non-negative matrix factorisation, fitted separately to the
# creativity and to the effort explanations.
#
# Each explanation becomes a row of tf-idf weights over the content words that
# appear in at least TOPIC_MIN_DF (five) explanations of that question: the
# words from tokenize() in section 8, with the task vocabulary of section 10
# removed. scikit-learn's TfidfVectorizer computes the weights with its
# defaults, which scale each row to length one so that a long explanation does
# not outweigh a short one. NMF (scikit-learn's NMF, starting values
# "nndsvda", at most 1,000 iterations) writes the explanation-by-word table as
# the product of an explanation-by-topic table W and a topic-by-word table H,
# both non-negative.
#
# The number of topics. For each k from 3 to 10 the model is refitted
# TOPIC_SEEDS (ten) times, with the seeds 0 to 9. Each seed draws a random
# 80 per cent of the participants, and its refit uses only their explanations,
# over the whole question's vocabulary, so that the topics of every refit are
# weights on the same words. The starting values "nndsvda" are computed from
# the data by a singular value decomposition, so refits on identical data with
# different seeds give nearly identical topics; leaving a fifth of the
# participants out is what lets the refits differ. For two refits, the topics
# are paired one to one by the Hungarian algorithm (scipy's
# linear_sum_assignment), which finds the pairing with the largest total
# cosine similarity between the paired topics' word weights, and the
# agreement of the two refits is the mean cosine similarity over the pairs.
# Stability at k is the mean agreement over all 45 pairs of refits, with its
# standard deviation beside it. The k with the highest stability, rounded to
# three decimals, is chosen, and a tie goes to the smaller k.
#
# The final model is fitted once, at the chosen k, to every explanation of the
# question, with the seed 20260421. Topics are numbered from 1 in order of
# their mean weight across explanations, largest first. An explanation's
# weights are its row of W divided by the row's sum, so that they sum to one;
# an explanation with no kept word, or one the model gives no weight on any
# topic, gets empty weights.
# ---------------------------------------------------------------------------
import warnings
from itertools import combinations

from scipy.optimize import linear_sum_assignment
from sklearn.decomposition import NMF
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer

TOPIC_MIN_DF = 5            # a kept word appears in at least this many explanations of the question
TOPIC_K_RANGE = range(3, 11)
TOPIC_SEEDS = 10            # refits per k, with the seeds 0 to 9
TOPIC_SHARE_KEPT = 0.8      # the share of participants each refit keeps
TOPIC_MAX_ITER = 1000
TOPIC_FINAL_SEED = 20260421
TOPIC_TOP_WORDS = 10


def topic_words(text):
    """Content words for the topic model: tokenize() without the task vocabulary."""
    return [w for w in tokenize(text) if not is_task_vocabulary(w)]


def unit_rows(matrix):
    """Each row divided by its length, so that the product of two rows is their cosine."""
    lengths = np.linalg.norm(matrix, axis=1, keepdims=True)
    lengths[lengths == 0] = 1.0
    return matrix / lengths


def topic_agreement(topics_a, topics_b):
    """Mean cosine similarity of two sets of topics, paired one to one by the
    Hungarian algorithm so that the total similarity of the pairs is as large
    as it can be."""
    similarity = unit_rows(topics_a) @ unit_rows(topics_b).T
    rows, cols = linear_sum_assignment(-similarity)
    return float(similarity[rows, cols].mean())


def fit_nmf(matrix, k, seed):
    """One NMF fit, and whether it finished inside the iteration limit."""
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        model = NMF(n_components=k, init="nndsvda", max_iter=TOPIC_MAX_ITER, random_state=seed)
        weights = model.fit_transform(matrix)
    converged = not any(issubclass(w.category, ConvergenceWarning) for w in caught)
    return model, weights, converged


texts_all = blind["response_text"].fillna("").astype(str).tolist()
stability_rows, top_word_rows, topic_rows = [], [], []
TOPIC_MODELS = {}

for dim in ["creativity", "effort"]:
    rows_of_dim = np.where((blind["dimension"] == dim).values)[0]
    texts = [texts_all[i] for i in rows_of_dim]
    vectorizer = TfidfVectorizer(analyzer=topic_words, min_df=TOPIC_MIN_DF)
    matrix = vectorizer.fit_transform(texts)
    vocab = vectorizer.get_feature_names_out()
    person = blind["id"].values[rows_of_dim]
    people = np.array(sorted(set(person)))
    n_kept = int(round(TOPIC_SHARE_KEPT * len(people)))

    # the stability of every k, from ten refits on 80 per cent of the participants
    results, n_short = [], 0
    for k in TOPIC_K_RANGE:
        topic_sets = []
        for seed in range(TOPIC_SEEDS):
            kept = np.random.default_rng(seed).choice(people, size=n_kept, replace=False)
            model, _, converged = fit_nmf(matrix[np.isin(person, kept)], k, seed)
            topic_sets.append(model.components_)
            n_short += int(not converged)
        agreements = [topic_agreement(a, b) for a, b in combinations(topic_sets, 2)]
        results.append((k, float(np.mean(agreements)), float(np.std(agreements, ddof=1))))
    chosen_k = max(results, key=lambda r: (round(r[1], 3), -r[0]))[0]
    for k, mean, sd in results:
        stability_rows.append(dict(dimension=dim, k=k, stability_mean=round(mean, 4),
                                   stability_sd=round(sd, 4), chosen=bool(k == chosen_k)))

    # the final model, at the chosen k, on every explanation of the question
    model, weights, final_converged = fit_nmf(matrix, chosen_k, TOPIC_FINAL_SEED)
    sums = weights.sum(axis=1)
    has_weight = sums > 0
    shares = np.full(weights.shape, np.nan)
    shares[has_weight] = weights[has_weight] / sums[has_weight, None]
    order = np.argsort(-np.nanmean(shares, axis=0), kind="stable")
    shares, components = shares[:, order], model.components_[order]

    for t in range(chosen_k):
        top = np.argsort(-components[t], kind="stable")[:TOPIC_TOP_WORDS]
        for rank, j in enumerate(top, start=1):
            top_word_rows.append(dict(dimension=dim, topic=t + 1, rank=rank, word=vocab[j],
                                      weight=round(float(components[t, j]), 4)))
    for position, i in enumerate(rows_of_dim):
        for t in range(chosen_k):
            topic_rows.append(dict(
                id=blind["id"].iloc[i], stimulus_id=blind["stimulus_id"].iloc[i], dimension=dim,
                topic=t + 1,
                weight=round(float(shares[position, t]), 4) if has_weight[position] else np.nan))

    TOPIC_MODELS[dim] = dict(k=chosen_k, texts=texts, has_weight=has_weight, shares=shares,
                             n_vocab=len(vocab), n_short=n_short, final_converged=final_converged)

TOPICS_COLUMNS = ["id", "stimulus_id", "dimension", "topic", "weight"]
topics_long = pd.DataFrame(topic_rows, columns=TOPICS_COLUMNS)
topic_top_words = pd.DataFrame(top_word_rows, columns=["dimension", "topic", "rank", "word", "weight"])
topic_stability = pd.DataFrame(stability_rows,
                               columns=["dimension", "k", "stability_mean", "stability_sd", "chosen"])

# the checks, before anything is written
for forbidden in ["Type", "rating", "response_text", "condition", "arm", "Age", "Gender"]:
    assert forbidden not in topics_long.columns, \
        "study1_text_topics.csv: %s must not appear in this file." % forbidden
assert list(topics_long.columns) == TOPICS_COLUMNS, \
    "study1_text_topics.csv: the columns are wrong: %s" % list(topics_long.columns)
rows_per_response = topics_long.groupby(["id", "stimulus_id", "dimension"]).size()
assert len(rows_per_response) == len(blind), "Not every response has topic rows."
for dim, fitted in TOPIC_MODELS.items():
    assert (rows_per_response.xs(dim, level="dimension") == fitted["k"]).all(), \
        "A %s response does not have exactly one row per topic." % dim
assert topics_long["weight"].dropna().between(0, 1).all(), "A topic weight fell outside 0 to 1."
weight_sums = topics_long.groupby(["id", "stimulus_id", "dimension"])["weight"].sum(min_count=1).dropna()
assert ((weight_sums - 1).abs() < 1e-3).all(), "A response's topic weights do not sum to one."
assert (topic_stability.groupby("dimension")["chosen"].sum() == 1).all(), "Each question needs one chosen k."

TOPICS_PATH = os.path.join(NLP_DIR, "study1_text_topics.csv")
atomic_write_csv(TOPICS_PATH, topics_long)
TOPIC_TABLE_PATHS = [
    atomic_write_csv(os.path.join(TABLES_DIR, "text_topics_top_words.csv"), topic_top_words),
    atomic_write_csv(os.path.join(TABLES_DIR, "text_topics_stability.csv"), topic_stability),
]

print("Written:", relative_path(TOPICS_PATH))
print("  rows (one per response and topic):", len(topics_long))
print("  condition and rating are absent. Checked.")
for path in TOPIC_TABLE_PATHS:
    print("Written:", relative_path(path))
for dim, fitted in TOPIC_MODELS.items():
    print()
    print("=" * 104)
    print("TOPICS IN THE %s EXPLANATIONS, BOTH ARMS TOGETHER" % dim.upper())
    print("=" * 104)
    print("  words kept (in at least %d explanations): %d;   explanations with no weights: %d of %d"
          % (TOPIC_MIN_DF, fitted["n_vocab"], int((~fitted["has_weight"]).sum()), len(fitted["texts"])))
    print("  refits that reached the iteration limit before settling: %d of %d; final fit settled: %s"
          % (fitted["n_short"], TOPIC_SEEDS * len(TOPIC_K_RANGE), fitted["final_converged"]))
    print()
    print(topic_stability[topic_stability["dimension"] == dim].to_string(index=False))
    print()
    print("  chosen: k = %d. Topics, with their mean weight and their ten top words:" % fitted["k"])
    mean_weight = np.nanmean(fitted["shares"], axis=0)
    for t in range(fitted["k"]):
        words = topic_top_words[(topic_top_words["dimension"] == dim) & (topic_top_words["topic"] == t + 1)]
        print("  %2d. (%.2f) %s" % (t + 1, mean_weight[t], ", ".join(words["word"])))


Written: nlp/study1_text_topics.csv
  rows (one per response and topic): 13320
  condition and rating are absent. Checked.
Written: tables/text_topics_top_words.csv
Written: tables/text_topics_stability.csv

TOPICS IN THE CREATIVITY EXPLANATIONS, BOTH ARMS TOGETHER
  words kept (in at least 5 explanations): 391;   explanations with no weights: 18 of 1110
  refits that reached the iteration limit before settling: 0 of 80; final fit settled: True

 dimension  k  stability_mean  stability_sd  chosen
creativity  3          0.8295        0.1269   False
creativity  4          0.8909        0.0797   False
creativity  5          0.8765        0.0695   False
creativity  6          0.9202        0.0531    True
creativity  7          0.8443        0.0681   False
creativity  8          0.8220        0.0761   False
creativity  9          0.8412        0.0604   False
creativity 10          0.8650        0.0655   False

  chosen: k = 6. Topics, with their mean weight and their ten top words:
   1. (0

In [20]:
# ---------------------------------------------------------------------------
# A check on the topics by a second method: do sentence embeddings group the
# explanations the same way?
#
# Each explanation with topic weights is turned into a list of 768 numbers by
# the sentence-embedding model all-mpnet-base-v2, a pretrained network that
# places sentences of similar meaning near one another. It assigns no label
# and writes no text, and nothing it produces is kept beyond the agreement
# figure below. The numbers are scaled to length one; the explanations of
# each question are split into as many groups as that question has topics by
# k-means (scikit-learn's KMeans, seed 20260421, ten starting points, the best
# of them kept, run on one processor thread), which puts each explanation in
# the group whose centre it is nearest; and the split is compared with each
# explanation's dominant topic, the topic with its largest weight. The
# comparison is the adjusted Rand index: the agreement between two ways of
# grouping the same items beyond what chance would give, 0 for chance
# agreement and 1 for identical groupings.
#
# The model is loaded only from a copy already on this machine
# (local_files_only=True, with the Hugging Face offline switches set), so the
# notebook never downloads anything. If the package or the copy is missing,
# the table is written with an empty agreement value, the reason is printed
# and recorded in tables/study1_text_environment.csv, and the run carries on.
# The model runs on the processor rather than on a graphics card, so that the
# numbers do not depend on which device a machine happens to have.
# ---------------------------------------------------------------------------
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from threadpoolctl import threadpool_limits

EMBEDDING_MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
EMBEDDING_SEED = 20260421
# Multithreaded k-means adds partial sums in a varying order, so only one thread repeats exactly.
KMEANS_OPENMP_THREADS = 1

try:
    from sentence_transformers import SentenceTransformer
    embedder = SentenceTransformer(EMBEDDING_MODEL_NAME, device="cpu", local_files_only=True)
    EMBEDDING_STATUS = "loaded from the local copy"
except Exception as error:
    embedder = None
    reason = str(error).strip().splitlines()[0][:200] if str(error).strip() else ""
    EMBEDDING_STATUS = "not run (%s: %s)" % (type(error).__name__, reason)
print("Embedding model %s: %s" % (EMBEDDING_MODEL_NAME, EMBEDDING_STATUS))

agreement_rows = []
for dim, fitted in TOPIC_MODELS.items():
    ari = np.nan
    if embedder is not None:
        has_weight = fitted["has_weight"]
        texts = [t for t, keep in zip(fitted["texts"], has_weight) if keep]
        vectors = embedder.encode(texts, batch_size=64, normalize_embeddings=True,
                                  show_progress_bar=False)
        with threadpool_limits(limits=KMEANS_OPENMP_THREADS, user_api="openmp"):
            groups = KMeans(n_clusters=fitted["k"], random_state=EMBEDDING_SEED,
                            n_init=10).fit_predict(vectors)
        dominant = np.argmax(fitted["shares"][has_weight], axis=1)
        ari = round(float(adjusted_rand_score(dominant, groups)), 4)
        print("  %-10s %d explanations, %d numbers each, k = %d: adjusted Rand index %.3f"
              % (dim, len(texts), vectors.shape[1], fitted["k"], ari))
    agreement_rows.append(dict(dimension=dim, k=fitted["k"], ari=ari))

topic_agreement_table = pd.DataFrame(agreement_rows, columns=["dimension", "k", "ari"])
TOPIC_TABLE_PATHS.append(
    atomic_write_csv(os.path.join(TABLES_DIR, "text_topics_embedding_agreement.csv"),
                     topic_agreement_table))
print("Written:", relative_path(TOPIC_TABLE_PATHS[-1]))
if embedder is None:
    print("The agreement check did not run, and its table holds empty values. Nothing else changes.")


Embedding model sentence-transformers/all-mpnet-base-v2: loaded from the local copy


  creativity 1092 explanations, 768 numbers each, k = 6: adjusted Rand index 0.092


  effort     1092 explanations, 768 numbers each, k = 6: adjusted Rand index 0.136
Written: tables/text_topics_embedding_agreement.csv


---

# 12. Approaches tested on this corpus and set aside

This section records methods that were run on these 2,220 justifications and do not appear anywhere
above, with what each one showed. It is kept so that anyone who comes back to this corpus does not
spend a week finding the same results again. Nothing in this section produces a number used in the
paper.

**Sentence-embedding measures.** Each justification was turned into a list of a few hundred numbers
that encode its meaning, and those numbers were used to predict the ratings. This worked, but it
worked equally well for both ratings. The numbers predicted creativity ratings about as well as
effort ratings. The correlation between predicted and actual ratings (written r, running from 0 for
no relation to 1 for perfect prediction) was .727 for creativity against .740 for effort. The
difference, +0.013, has a 95 per cent interval (the range of plausible values for the difference)
running from -0.043 to +0.060. An interval that straddles zero that comfortably means the measure
did not distinguish the two constructs. What it picks up is how positive the writing is, which both
ratings share, rather than what the writing is about, which is the question here. A measure that
cannot tell effort talk from creativity talk cannot answer question 1.

**Clustering and topic discovery.** Letting an algorithm find groups in the justifications, rather
than specifying categories in advance, was tried in 238 configurations across two clustering
algorithms and fourteen settings. Every interpretable group that emerged corresponded to a category
already on the list in section 1 (detail, time, skill, prompting, familiarity), and no configuration
produced a category that was both new and legible. The best separation achieved across all 238 runs
was 0.12, on a scale where values near zero mean the groups overlap almost completely. There is no
natural grouping in this corpus to find. The responses are short, they are about ten shared images,
and they sit on a continuum rather than in clusters. Specified categories are the right tool here.

**A grammatical rule for who made the work.** A rule that parses each sentence (works out its
grammatical structure) and identifies who it says performed the making (the artist, the machine, or
nobody named) agreed with human judgment at Fleiss kappa 0.90. Kappa is an agreement score that is 1
when two coders always agree and 0 when they agree only as often as chance would produce, so 0.90 is
excellent agreement. The rule also fired on only 27 per cent of responses, because the other 73 per
cent have no such sentence: they describe the picture, not the making. A measure defined on a
quarter of the corpus, where that quarter self-selects on the grammatical construction people
happened to use, cannot support a claim about the corpus.

**Matching text to images.** Scoring how well each justification matches the artwork it describes
was also tested and not kept. The measure is dominated by which of the ten images is being described
rather than by anything about the writer. With ten images shared by all 111 participants, there is
no variation left for it to explain.

**Anything framed as text explaining a rating.** Mediation models (models in which a measure of the
text is treated as the route by which the condition affects the rating) are not used, and neither is
any analysis in which a text measure is treated as a cause of the rating. The design cannot support
them. Every justification was written after the rating it justifies, by the same person, about the
same trial. The text and the rating are two expressions of one judgment made at one moment. They can
be described together, which sections 4 and 5 do. One cannot be shown to have produced the other,
and the notebook does not try.

---

# 13. Export

The notebook writes six data files, a set of tables, a checksum of the rules, a record of the
software it ran on, and a run manifest.

The main data file, `nlp/study1_text_measures.csv`, has one row per response and exactly the
nineteen columns agreed on: the identifiers, the word count, the eight measures, the reading of
machine-made talk beside its flag, and the four polarity columns that travel with detail, time,
skill and concept talk. It contains **no experimental condition and no rating**, and the cell below
asserts that before writing, together with the rule that a polarity or a reading is "neither"
exactly when its flag is false. The integration step joins condition and rating back on from
`study1_long_with_text.csv` using participant, artwork and question. Keeping the condition out of
this file is what makes the measurement independently checkable: anyone can re-run this notebook and
get the same flags without ever knowing which arm anything came from.

The other five data files were written in sections 9 and 11: the word-count table and its
vocabulary, the content-word lists, the affect and concreteness scores, and the topic weights. Like
the flags file they carry no condition and no rating. The cell below reads each of the six back from
disk and checks that it carries exactly its agreed columns and none of the forbidden ones.

Every file is written under a temporary name and renamed once complete. The last one written,
`tables/study1_text_run_manifest.csv`, lists every output with its size and its SHA-256 fingerprint,
beside the run identifier, the fingerprint of the input file and the checksum of the rules. A file
whose fingerprint differs from its row in the manifest was not written by the run the manifest
describes. Section 0 deletes the previous run's manifest before anything is written, so a manifest
on disk means that the run which wrote it reached this cell. `tables/study1_text_environment.csv`
records the Python version and the versions of the packages the measures depend on.


In [21]:
# ---------------------------------------------------------------------------
# Assemble the export file: exactly the agreed columns, in the agreed order.
#
# Nineteen columns: five identifiers, the word count, the eight measures, the
# reading of machine-made talk beside its flag, and the four polarity columns.
# ---------------------------------------------------------------------------
EXPORT_COLUMNS = [
    "id", "stimulus_id", "stimulus_num", "stimulus_type", "dimension", "n_words",
    "says_detail", "says_time", "says_skill", "says_concept", "says_prompt", "says_machine_made",
    "machine_made_reading",
    "prior_exposure", "anchor_echo",
    "detail_polarity", "time_polarity", "skill_polarity", "concept_polarity",
]
TRUE_FALSE_FLAGS = ["says_detail", "says_time", "says_skill", "says_concept", "says_prompt",
                    "says_machine_made", "anchor_echo"]
THREE_VALUES = {"asserted", "negated", "neither"}
READING_VALUES = {"asserted", "not_asserted", "neither"}
assert len(EXPORT_COLUMNS) == 19, "The export contract has nineteen columns."

# Columns that no data file this notebook writes may carry: the condition, the
# rating, the response text and the demographics.
FORBIDDEN_IN_OUTPUTS = ["Type", "rating", "response_text", "condition", "arm",
                        "Age", "Gender", "Race", "Language", "Experience"]


def check_export(frame, label):
    """Every check a flags file has to pass before it is written."""
    assert list(frame.columns) == EXPORT_COLUMNS, \
        "%s: export columns are wrong: %s" % (label, list(frame.columns))
    assert len(frame) == 2220, "%s: expected 2,220 rows, found %d." % (label, len(frame))
    for forbidden in FORBIDDEN_IN_OUTPUTS:
        assert forbidden not in frame.columns, \
            "%s: %s must not appear in the exported measures file." % (label, forbidden)
    assert set(frame["prior_exposure"]) <= THREE_VALUES, \
        "%s: prior_exposure has an unexpected value." % label
    for flag in TRUE_FALSE_FLAGS:
        assert frame[flag].dtype == bool, "%s: %s should be true/false." % (label, flag)
    # a polarity is "neither" exactly when its flag is false, and is asserted or
    # negated, nothing else, when the flag is true
    for flag, column in POLARITY_COLUMNS.items():
        assert set(frame[column]) <= THREE_VALUES, \
            "%s: %s has an unexpected value." % (label, column)
        assert ((frame[column] == "neither") == ~frame[flag]).all(), \
            "%s: %s must be 'neither' exactly when %s is false." % (label, column, flag)
    # the machine-made reading follows the same rule against its own flag
    assert set(frame["machine_made_reading"]) <= READING_VALUES, \
        "%s: machine_made_reading has an unexpected value." % label
    assert ((frame["machine_made_reading"] == "neither") == ~frame["says_machine_made"]).all(), \
        "%s: machine_made_reading must be 'neither' exactly when says_machine_made is false." % label
    assert frame.duplicated(["id", "stimulus_id", "dimension"]).sum() == 0, \
        "%s: the export has duplicated participant/artwork/question rows." % label


MEASURES_PATH = os.path.join(NLP_DIR, "study1_text_measures.csv")

export = measures[EXPORT_COLUMNS].copy()
check_export(export, "study1_text_measures.csv")
atomic_write_csv(MEASURES_PATH, export)
print("Written:", relative_path(MEASURES_PATH))
print("  rows:", len(export))
print("  columns:", list(export.columns))
print("  condition and rating are absent. Checked.")
print("  each polarity, and the machine-made reading, is 'neither' exactly where its flag is false.")
print("  Checked.")
print()

# ---------------------------------------------------------------------------
# Every data file the notebook writes, read back from disk and held to the
# columns its contract fixes, so that the check is of what the integration
# step will read rather than of what the notebook meant to write. None may
# carry a forbidden column. Every file keyed by response, apart from the
# word-count table (which has no row for a response with no counted word),
# must cover exactly the 2,220 responses.
# ---------------------------------------------------------------------------
DATA_CONTRACTS = [
    (MEASURES_PATH, EXPORT_COLUMNS),
    (WORDCOUNTS_PATH, ["id", "stimulus_id", "dimension", "word", "count"]),
    (VOCABULARY_PATH, ["word", "n_responses"]),
    (TOKENS_PATH, TOKENS_COLUMNS),
    (AFFECT_PATH, AFFECT_COLUMNS),
    (TOPICS_PATH, TOPICS_COLUMNS),
]
response_keys = set(blind[["id", "stimulus_id", "dimension"]].astype(str)
                    .itertuples(index=False, name=None))
for path, columns in DATA_CONTRACTS:
    name = os.path.basename(path)
    on_disk = pd.read_csv(path, dtype=str, keep_default_na=False)
    assert list(on_disk.columns) == columns, \
        "%s: the columns on disk are %s, not %s." % (name, list(on_disk.columns), columns)
    present = [c for c in FORBIDDEN_IN_OUTPUTS if c in on_disk.columns]
    assert not present, "%s carries a forbidden column: %s" % (name, present)
    if "dimension" in columns and path != WORDCOUNTS_PATH:
        keys = set(on_disk[["id", "stimulus_id", "dimension"]].itertuples(index=False, name=None))
        assert keys == response_keys, "%s does not cover exactly the 2,220 responses." % name
    print("  %-28s %6d rows, %2d columns as agreed, no forbidden column"
          % (name, len(on_disk), len(columns)))
print("All six data files carry exactly their agreed columns. Checked against the files on disk.")
print()

print(export.head(6).to_string(index=False))


Written: nlp/study1_text_measures.csv
  rows: 2220
  columns: ['id', 'stimulus_id', 'stimulus_num', 'stimulus_type', 'dimension', 'n_words', 'says_detail', 'says_time', 'says_skill', 'says_concept', 'says_prompt', 'says_machine_made', 'machine_made_reading', 'prior_exposure', 'anchor_echo', 'detail_polarity', 'time_polarity', 'skill_polarity', 'concept_polarity']
  condition and rating are absent. Checked.
  each polarity, and the machine-made reading, is 'neither' exactly where its flag is false.
  Checked.

  study1_text_measures.csv       2220 rows, 19 columns as agreed, no forbidden column
  study1_text_wordcounts.csv    17231 rows,  5 columns as agreed, no forbidden column
  study1_text_vocabulary.csv     1300 rows,  2 columns as agreed, no forbidden column
  study1_text_tokens.csv         2220 rows,  4 columns as agreed, no forbidden column
  study1_text_affect.csv         2220 rows, 10 columns as agreed, no forbidden column
  study1_text_topics.csv        13320 rows,  5 columns 

In [22]:
# ---------------------------------------------------------------------------
# The tables, and the fingerprint that belongs beside the first of them.
# ---------------------------------------------------------------------------
import hashlib

written = []

def write_text(filename, content):
    """Write a table file through a temporary name (section 0) and list it."""
    path = atomic_write_text(os.path.join(TABLES_DIR, filename), content)
    written.append(path)
    return path

def write_csv(filename, frame):
    """Write a CSV table through a temporary name (section 0) and list it."""
    path = atomic_write_csv(os.path.join(TABLES_DIR, filename), frame)
    written.append(path)
    return path

# 1. the rules, in full
RULES_FILE_TEXT = RULES_TEXT + "\n"
write_text("text_rules.txt", RULES_FILE_TEXT)

# 1a. a checksum of the rules. A checksum is a short string computed from the
# exact contents of a file by a fixed recipe (here SHA-256), such that
# changing the contents by even one character changes the string, so a reader
# can confirm that the word lists in their hand are the word lists that were
# run. The preregistration commits to depositing the rules together with one.
# The line is written in the format the shasum command reads, so a reader can
# verify it from the tables folder with
#     shasum -a 256 -c text_rules.sha256
RULES_SHA256 = hashlib.sha256(RULES_FILE_TEXT.encode("utf-8")).hexdigest()
write_text("text_rules.sha256", "%s  text_rules.txt\n" % RULES_SHA256)
print("SHA-256 of text_rules.txt:", RULES_SHA256)
print()

# 1b. the grounding notes: for each rule, the construct it measures, where
#     that reading comes from in the literature, and what the rule does and
#     does not record. They go in a file of their own so that text_rules.txt
#     holds the rules and nothing else, and the checksum above covers only
#     the rules that ran.
def format_grounding(rules):
    out = []
    out.append("=" * 100)
    out.append("STUDY 1 TEXT MEASURES: WHAT EACH RULE IS GROUNDED IN")
    out.append("=" * 100)
    out.append("")
    out.append("One note per rule, naming the construct the rule measures, the reading of effort or")
    out.append("creativity it stands for, the literature that reading comes from, and what the rule")
    out.append("does and does not record. The notes are not part of the patterns: the patterns are in")
    out.append("text_rules.txt, and its checksum covers that file alone.")
    out.append("")
    for i, rule in enumerate(rules, start=1):
        out.append("-" * 100)
        out.append("RULE %d OF %d: %s" % (i, len(rules), rule["name"]))
        out.append("-" * 100)
        out.append("In plain words: " + rule["plain"])
        out.append("")
        out.append(textwrap.fill(rule["grounding"], width=96))
        out.append("")
    return "\n".join(out)

GROUNDING_TEXT = format_grounding(RULES) + "\n"
write_text("text_rules_grounding.txt", GROUNDING_TEXT)
print("Written: text_rules_grounding.txt, one note per rule. The checksum above covers")
print("text_rules.txt alone.")
print()

# 2. matched and unmatched responses, rule by rule
write_text("text_audit_matches.txt", AUDIT_TEXT + "\n")

# 3 and 4. the distinctive-word tables
write_csv("text_keyness_effort.csv", keyness_effort[["word", "z", "n_high", "n_low"]])
write_csv("text_keyness_creativity.csv", keyness_creativity[["word", "z", "n_high", "n_low"]])

# 5. the candidate words for each rule, a report on coverage that nothing reads back
write_text("text_candidate_words.txt", CANDIDATE_TEXT + "\n")

# 6. a summary of the corpus and of how often each rule fired. `flags` is the
#    blind frame from section 2; `flags_joined` is the same frame with condition
#    and rating joined on, as in section 4. The true/false flags are listed in
#    TRUE_FALSE_FLAGS, in the order of the export contract.
def corpus_summary(flags, flags_joined):
    summary_rows = []
    summary_rows.append(dict(group="corpus", statistic="responses", value=len(flags)))
    summary_rows.append(dict(group="corpus", statistic="participants", value=flags["id"].nunique()))
    summary_rows.append(dict(group="corpus", statistic="artworks", value=flags["stimulus_id"].nunique()))
    for label, value in [("words_mean", flags["n_words"].mean()),
                         ("words_sd", flags["n_words"].std()),
                         ("words_min", flags["n_words"].min()),
                         ("words_q25", flags["n_words"].quantile(.25)),
                         ("words_median", flags["n_words"].median()),
                         ("words_q75", flags["n_words"].quantile(.75)),
                         ("words_max", flags["n_words"].max())]:
        summary_rows.append(dict(group="word_count", statistic=label, value=round(float(value), 2)))

    for flag in TRUE_FALSE_FLAGS:
        summary_rows.append(dict(group="prevalence_all", statistic=flag,
                                 value=round(100 * float(flags[flag].mean()), 2)))
    for category in ["asserted", "negated", "neither"]:
        summary_rows.append(dict(group="prevalence_all", statistic="prior_exposure_" + category,
                                 value=round(100 * float((flags["prior_exposure"] == category).mean()), 2)))
    for category in ["asserted", "not_asserted"]:
        summary_rows.append(dict(group="prevalence_all", statistic="machine_made_reading_" + category,
                                 value=round(100 * float((flags["machine_made_reading"] == category).mean()), 2)))

    for arm in ["Human", "AI"]:
        for dim in ["effort", "creativity"]:
            sub = flags_joined[(flags_joined["Type"] == arm) & (flags_joined["dimension"] == dim)]
            for flag in TRUE_FALSE_FLAGS:
                summary_rows.append(dict(group="prevalence_%s_%s" % (arm.lower(), dim),
                                         statistic=flag, value=round(100 * float(sub[flag].mean()), 2)))
            for category in ["asserted", "negated", "neither"]:
                summary_rows.append(dict(group="prevalence_%s_%s" % (arm.lower(), dim),
                                         statistic="prior_exposure_" + category,
                                         value=round(100 * float((sub["prior_exposure"] == category).mean()), 2)))
            for category in ["asserted", "not_asserted"]:
                summary_rows.append(dict(group="prevalence_%s_%s" % (arm.lower(), dim),
                                         statistic="machine_made_reading_" + category,
                                         value=round(100 * float((sub["machine_made_reading"] == category).mean()), 2)))

    # The share of flagged responses read as negated, for each of the four
    # topic rules: over the whole corpus, within each question, and within each
    # kind of artwork. Computed from the blind frame, so no condition enters.
    slices = [("all", flags)]
    for dim in ["effort", "creativity"]:
        slices.append((dim, flags[flags["dimension"] == dim]))
    for kind in sorted(flags["stimulus_type"].unique()):
        slices.append((str(kind).lower(), flags[flags["stimulus_type"] == kind]))
    for flag, column in POLARITY_COLUMNS.items():
        for label, sub in slices:
            hit = sub.loc[sub[flag], column]
            share = 100 * float((hit == "negated").mean()) if len(hit) else float("nan")
            summary_rows.append(dict(group="negated_among_flagged_" + label, statistic=flag,
                                     value=round(share, 2)))

    return pd.DataFrame(summary_rows)[["group", "statistic", "value"]]


write_csv("text_corpus_summary.csv", corpus_summary(measures, joined))

# 7. the software this run used: the Python version and the versions of the
#    packages the measures depend on (pandas and NumPy throughout, scikit-learn
#    and SciPy for the topic model, sentence-transformers and torch for the
#    agreement check), with whether the embedding model loaded and the number
#    of processor threads the k-means step of the agreement check ran on.
import platform


def package_version(module_name):
    """The installed version of a package, or a note saying it is missing."""
    try:
        module = __import__(module_name)
    except Exception:
        return "not installed"
    return str(getattr(module, "__version__", "installed, version not recorded"))


environment = pd.DataFrame([
    ("run_id", RUN_ID),
    ("python", platform.python_version()),
    ("platform", platform.platform()),
    ("pandas", pd.__version__),
    ("numpy", np.__version__),
    ("scikit-learn", package_version("sklearn")),
    ("scipy", package_version("scipy")),
    ("sentence-transformers", package_version("sentence_transformers")),
    ("torch", package_version("torch")),
    ("embedding_model", "%s, %s" % (EMBEDDING_MODEL_NAME, EMBEDDING_STATUS)),
    ("kmeans_openmp_threads", str(KMEANS_OPENMP_THREADS)),
], columns=["item", "value"])
write_csv("study1_text_environment.csv", environment)
print(environment.to_string(index=False))
print()

DESCRIPTIONS = {
    "study1_text_measures.csv":
        "one row per response, nineteen columns, no condition and no rating.\n"
        "        Read next by the integration step, which joins condition and rating back on\n"
        "        from study1_long_with_text.csv using (id, stimulus_id, dimension).",
    "study1_text_wordcounts.csv":
        "one row per response and content word, with the count: id, stimulus_id, dimension,\n"
        "        word, count. Words appearing in only one response are left out; the integration\n"
        "        step applies the registered five-response floor inside each round of its\n"
        "        cross-validation. No condition and no rating. Read by the integration step for\n"
        "        the first registered check, how much of the rating and of the arm the words\n"
        "        alone recover.",
    "study1_text_vocabulary.csv":
        "the words in the word-count table and the number of responses each appears in.",
    "study1_text_tokens.csv":
        "one row per response: id, stimulus_id, dimension and content_words, the response's\n"
        "        distinct content words, sorted and separated by spaces (empty when there are\n"
        "        none). No condition and no rating. Read by the integration step, which computes\n"
        "        the within-person and stranger word overlaps of the second registered check.",
    "study1_text_affect.csv":
        "one row per response: the number of content words once the task vocabulary is\n"
        "        removed, how many of them the published norms contain, and their mean valence,\n"
        "        arousal, dominance and concreteness. No condition and no rating. Exploratory\n"
        "        description, read by the integration step.",
    "study1_text_topics.csv":
        "one row per response and topic: the response's weight on that topic, a response's\n"
        "        weights summing to one. Topics fitted separately to the two questions. No\n"
        "        condition and no rating. Exploratory description, read by the integration step.",
    "text_topics_top_words.csv":
        "the ten words with the largest weight in every topic, for each question.",
    "text_topics_stability.csv":
        "the stability of every number of topics from 3 to 10, for each question, and the one\n"
        "        chosen.",
    "text_topics_embedding_agreement.csv":
        "for each question, the adjusted Rand index between each response's dominant topic and\n"
        "        k-means groups of its sentence embedding; empty if the embedding model was not\n"
        "        available.",
    "text_rules.txt":
        "every search pattern printed in full, human-readable. The definition of\n"
        "        every number in this notebook. Belongs in the supplement.",
    "text_rules.sha256":
        "the SHA-256 checksum of text_rules.txt, in the format shasum reads, so\n"
        "        that a reader can confirm the rules they are looking at are the rules that ran.",
    "text_rules_grounding.txt":
        "one note per rule naming the construct it measures, the reading of effort\n"
        "        or creativity it stands for and the literature that reading comes from. Not part\n"
        "        of the patterns, and not covered by the checksum on text_rules.txt.",
    "text_audit_matches.txt":
        "40 matched and 20 unmatched responses per rule, with the triggering\n"
        "        phrase marked and, for the four topic rules and for machine-made talk, the value\n"
        "        of the polarity or reading column beside each listed response.",
    "text_candidate_words.txt":
        "up to 30 words per rule that behave like the rule's own words without being\n"
        "        on its list, each with counts and one example sentence. Reported so that a reader\n"
        "        can judge the rules' coverage; no word is added to any rule after the data have\n"
        "        been seen, and nothing reads this file back. It is not a recall figure and none\n"
        "        should be reported from it.",
    "text_keyness_effort.csv":
        "word, z, count in high-rated writing, count in low-rated writing, for\n"
        "        effort justifications in the human-attribution arm.",
    "text_keyness_creativity.csv":
        "the same for creativity justifications.",
    "text_corpus_summary.csv":
        "response and participant counts, the word-length distribution, how often\n"
        "        each rule fired overall and within each arm and question, the share of responses\n"
        "        whose machine-made reading is asserted or not asserted, and, for detail, time,\n"
        "        skill and concept talk, the share of flagged responses read as negated, overall,\n"
        "        by question and by kind of artwork.",
    "study1_text_environment.csv":
        "the run identifier, the Python version, the versions of the packages the measures\n"
        "        depend on, whether the embedding model loaded, and the number of processor\n"
        "        threads the k-means step of the agreement check ran on.",
    "study1_text_run_manifest.csv":
        "every file above with its size and SHA-256 fingerprint, beside the run identifier,\n"
        "        the input file's SHA-256 and the rules checksum. Written last.",
}

# 8. the run manifest, written last. One row per output with its size and its
#    SHA-256 fingerprint, beside the run identifier, the fingerprint of the
#    input file and the checksum of the rules. A file on disk whose fingerprint
#    differs from its row here was not written by this run. The previous
#    run's manifest was deleted in section 0, before this run wrote anything.
def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


all_paths = ([MEASURES_PATH, WORDCOUNTS_PATH, VOCABULARY_PATH, TOKENS_PATH, AFFECT_PATH, TOPICS_PATH]
             + TOPIC_TABLE_PATHS + written)
completed_utc = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
manifest = pd.DataFrame([dict(
    run_id=RUN_ID, completed_utc=completed_utc,
    input_file=INPUT_NAME, input_sha256=INPUT_SHA256, rules_sha256=RULES_SHA256,
    file=os.path.relpath(path, BASE), bytes=os.path.getsize(path), sha256=file_sha256(path),
) for path in all_paths])
atomic_write_csv(MANIFEST_PATH, manifest)

print("=" * 104)
print("MANIFEST: EVERY FILE THIS NOTEBOOK WROTE")
print("=" * 104)
print("run %s, input %s, input SHA-256 %s" % (RUN_ID, INPUT_NAME, INPUT_SHA256))
for path in all_paths + [MANIFEST_PATH]:
    name = os.path.basename(path)
    size_kb = os.path.getsize(path) / 1024.0
    print()
    print("  %s   (%.1f KB)" % (relative_path(path), size_kb))
    print("        " + DESCRIPTIONS.get(name, ""))
print()
print("=" * 104)
print("Done. %d files written, the run manifest last." % (len(all_paths) + 1))
print("=" * 104)


SHA-256 of text_rules.txt: 6aa77dbc26c9086c5349a863c1cab808d7bd3758265fc365de2fb36c46a30239

Written: text_rules_grounding.txt, one note per rule. The checksum above covers
text_rules.txt alone.

                 item                                                               value
               run_id                                           20261015T205712Z-67bfab03
               python                                                              3.12.3
             platform                                        macOS-26.6.2-arm64-arm-64bit
               pandas                                                               2.2.2
                numpy                                                              1.26.4
         scikit-learn                                                               1.5.2
                scipy                                                              1.13.0
sentence-transformers                                                               

---

## What this notebook established

The summary below is written out so that a reader need not reconstruct it from the output.

**Question 1, what people meant by effort.** Section 4 gives the answer in one table. In the arm
told a human artist made the work, effort justifications are overwhelmingly about visible detail,
and only rarely about time or about skill. The effort rating in this study is a judgment about how
much is in the picture and how finely it is rendered. Section 8 finds the same thing without being
told to look for it.

**Question 2, what people meant by creativity.** Section 5 puts three candidate answers in one
table, in the same shape as section 4's, with the effort column beside them for contrast: the idea
behind the work, whether the work is new or already familiar, and how finely it is executed. Two of
the three are measured by rules the notebook uses elsewhere, so the execution row of that table and
the detail row of section 4's are the same measurement read in two places. Within that question sits
the doctrinal one. Novelty is the patent test, not the copyright one, so familiarity talk in
creativity justifications is lay judgment importing a criterion copyright rejects. Section 5 reports
how often raters volunteered that a work looked familiar, generic or already seen, and how the
creativity ratings attached to those responses compare with the rest. The direction of that
association is descriptive only, for the reason stated there.

**Question 3, machines in the human arm's writing.** Section 6 counts, in the arm told a human
artist made the work, the responses that name a machine at all, the mentions of a machine making the
work, and the statements that a machine made the work. Each is counted in responses and in distinct
participants, and every flagged response is printed with its reading. These counts describe what
participants wrote. They do not measure how many participants disbelieved the vignette.

**Two registered quantities, measured but not read here.** Section 9 writes the word-count table and
the content-word lists from which the two registered text checks are computed. Their readings belong
to the integration step, where the arm and the rating are joined to the finished measures for
inference: how much of the rating and of the arm the words recover, and whether the machine label
brings more self-repetition than the human label. Nothing about them can be read from this notebook.

**A general description, read elsewhere.** Section 11 writes affect and concreteness scores and
topic weights for every response, with the topic tables and the agreement check beside them, all
from the blind slice. The integration step reports them by arm as exploratory description, in this
study and in every other. Nothing about the arms can be read from them here.

**What has not been established.** Recall (the share of its subject each rule catches) is not
measured anywhere in this notebook, and section 10 does not measure it either. What section 10 does
is narrower: it asks the corpus which words behave like each rule's words without being on its list,
and reports them so that a reader can judge the rules' coverage. A rule whose list holds no word
that plainly names its topic has a coverage argument behind it, and no word is added to any rule
after the data have been seen. It still has no recall figure, because that would require reading
responses the rule stayed silent on and judging each one, and a response can be missed without
containing any unusual word at all. Nothing here is causal. Every count depends on the rules in
section 1, which is why they are printed in full rather than summarised.